In [2]:
# ============================================================
# AMLC 2026 — CELL 01
# Runtime diagnostics
# ============================================================

import os
import sys
import platform
import shutil
import subprocess

print("=" * 70)
print("AMLC 2026 ENTITY RESOLUTION — RUNTIME CHECK")
print("=" * 70)

print("\nPython:")
print(sys.version)

print("\nPlatform:")
print(platform.platform())

print("\nCPU:")
try:
    print(subprocess.check_output(["bash", "-lc", "nproc"]).decode().strip())
except Exception:
    print("Could not determine CPU count")

print("\nRAM:")
try:
    print(subprocess.check_output(
        ["bash", "-lc", "free -h"]
    ).decode())
except Exception:
    print("Could not determine RAM")

print("GPU:")
try:
    print(subprocess.check_output(
        ["bash", "-lc", "nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader"]
    ).decode())
except Exception as e:
    print("No GPU detected:", e)

print("\nDisk:")
try:
    print(subprocess.check_output(
        ["bash", "-lc", "df -h /content"]
    ).decode())
except Exception:
    print("Could not determine disk")

print("\nWorking directory:")
print(os.getcwd())

print("=" * 70)

AMLC 2026 ENTITY RESOLUTION — RUNTIME CHECK

Python:
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

Platform:
Linux-6.6.122+-x86_64-with-glibc2.39

CPU:
2

RAM:
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.4Gi        10Gi       4.0Mi       990Mi        11Gi
Swap:             0B          0B          0B

GPU:
Tesla T4, 15360 MiB, 580.82.07


Disk:
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   49G   64G  44% /


Working directory:
/content


In [51]:
# ============================================================
# AMLC 2026 — CELL 02
# Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# ============================================================
# AMLC 2026 — CELL 03
# Verify project structure
# ============================================================

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/AMLC2026")

DATA_ROOT = DRIVE_ROOT / "dataset"
TRAIN_ROOT = DATA_ROOT / "train"
TEST_ROOT = DATA_ROOT / "test"

print("DRIVE_ROOT:", DRIVE_ROOT)
print("DATA_ROOT :", DATA_ROOT)
print("TRAIN_ROOT:", TRAIN_ROOT)
print("TEST_ROOT :", TEST_ROOT)

print("\nTrain files:")
for p in sorted(TRAIN_ROOT.glob("*")):
    print(f"{p.name:30s} {p.stat().st_size / (1024**3):.2f} GB")

print("\nTest files:")
for p in sorted(TEST_ROOT.glob("*")):
    print(f"{p.name:30s} {p.stat().st_size / (1024**3):.2f} GB")

DRIVE_ROOT: /content/drive/MyDrive/AMLC2026
DATA_ROOT : /content/drive/MyDrive/AMLC2026/dataset
TRAIN_ROOT: /content/drive/MyDrive/AMLC2026/dataset/train
TEST_ROOT : /content/drive/MyDrive/AMLC2026/dataset/test

Train files:
train_ground_truth.tsv         0.12 GB
train_source1.tsv              0.20 GB
train_source2.tsv              0.46 GB
train_source3.tsv              0.47 GB

Test files:
test_source1.tsv               0.16 GB
test_source2.tsv               0.47 GB
test_source3.tsv               0.47 GB


In [5]:
# ============================================================
# AMLC 2026 — CELL 04
# Local working directories
# ============================================================

from pathlib import Path

LOCAL_ROOT = Path("/content/AMLC2026")

LOCAL_DATA = LOCAL_ROOT / "dataset"
LOCAL_CACHE = LOCAL_ROOT / "cache"
LOCAL_FEATURES = LOCAL_ROOT / "features"
LOCAL_EMBEDDINGS = LOCAL_ROOT / "embeddings"
LOCAL_MODELS = LOCAL_ROOT / "models"
LOCAL_OOF = LOCAL_ROOT / "oof"
LOCAL_SUBMISSIONS = LOCAL_ROOT / "submissions"
LOCAL_LOGS = LOCAL_ROOT / "logs"

for p in [
    LOCAL_DATA,
    LOCAL_CACHE,
    LOCAL_FEATURES,
    LOCAL_EMBEDDINGS,
    LOCAL_MODELS,
    LOCAL_OOF,
    LOCAL_SUBMISSIONS,
    LOCAL_LOGS,
]:
    p.mkdir(parents=True, exist_ok=True)

print("Created local workspace:")
print(LOCAL_ROOT)

!df -h /content

Created local workspace:
/content/AMLC2026
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   49G   64G  44% /


In [52]:
# ============================================================
# AMLC 2026 — CELL 05
# Core data / ER stack
# ============================================================

!pip -q install \
    polars \
    pyarrow \
    duckdb \
    rapidfuzz \
    unidecode \
    faiss-cpu \
    scikit-learn \
    lightgbm \
    xgboost

In [7]:
# ============================================================
# AMLC 2026 — CELL 06
# Verify packages
# ============================================================

import polars as pl
import pandas as pd
import numpy as np
import duckdb
import rapidfuzz
import sklearn
import lightgbm
import xgboost

print("Polars     :", pl.__version__)
print("Pandas     :", pd.__version__)
print("NumPy      :", np.__version__)
print("DuckDB     :", duckdb.__version__)
print("RapidFuzz  :", rapidfuzz.__version__)
print("Scikit     :", sklearn.__version__)
print("LightGBM   :", lightgbm.__version__)
print("XGBoost    :", xgboost.__version__)

try:
    import faiss
    print("FAISS      :", faiss.__version__)
except Exception as e:
    print("FAISS import error:", e)

Polars     : 1.35.2
Pandas     : 2.2.3
NumPy      : 2.1.3
DuckDB     : 1.3.2
RapidFuzz  : 3.14.6
Scikit     : 1.6.1
LightGBM   : 4.6.0
XGBoost    : 3.4.1
FAISS      : 1.15.1


In [8]:
# ============================================================
# AMLC 2026 — CELL 07
# Copy raw dataset to local Colab storage
#
# Drive = persistent master
# /content = high-speed working copy
# ============================================================

import subprocess
from pathlib import Path

LOCAL_DATA = Path("/content/AMLC2026/dataset")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

print("Copying dataset from Google Drive to local storage...")
print("This should only need to happen once per Colab runtime.\n")

cmd = [
    "rsync",
    "-ah",
    "--info=progress2",
    "--exclude=.DS_Store",
    "/content/drive/MyDrive/AMLC2026/dataset/",
    "/content/AMLC2026/dataset/"
]

result = subprocess.run(cmd)

if result.returncode != 0:
    raise RuntimeError("Dataset copy failed.")

print("\n✅ Dataset copy complete.")

Copying dataset from Google Drive to local storage...
This should only need to happen once per Colab runtime.


✅ Dataset copy complete.


In [9]:
# ============================================================
# AMLC 2026 — CELL 08
# Verify local dataset
# ============================================================

from pathlib import Path

LOCAL_TRAIN = Path("/content/AMLC2026/dataset/train")
LOCAL_TEST  = Path("/content/AMLC2026/dataset/test")

print("TRAIN")
print("-" * 70)

for p in sorted(LOCAL_TRAIN.glob("*")):
    size_gb = p.stat().st_size / (1024 ** 3)
    print(f"{p.name:30s} {size_gb:8.3f} GB")

print("\nTEST")
print("-" * 70)

for p in sorted(LOCAL_TEST.glob("*")):
    size_gb = p.stat().st_size / (1024 ** 3)
    print(f"{p.name:30s} {size_gb:8.3f} GB")

print("\nDisk after copy:")
!df -h /content

TRAIN
----------------------------------------------------------------------
train_ground_truth.tsv            0.118 GB
train_source1.tsv                 0.196 GB
train_source2.tsv                 0.456 GB
train_source3.tsv                 0.469 GB

TEST
----------------------------------------------------------------------
test_source1.tsv                  0.163 GB
test_source2.tsv                  0.474 GB
test_source3.tsv                  0.471 GB

Disk after copy:
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   53G   61G  47% /


In [10]:
# ============================================================
# AMLC 2026 — CELL 09
# Robust TSV loader + schema inspection
# ============================================================

import polars as pl
from pathlib import Path

TRAIN_DIR = Path("/content/AMLC2026/dataset/train")
TEST_DIR  = Path("/content/AMLC2026/dataset/test")

files = {
    "train_source1": TRAIN_DIR / "train_source1.tsv",
    "train_source2": TRAIN_DIR / "train_source2.tsv",
    "train_source3": TRAIN_DIR / "train_source3.tsv",
    "train_ground_truth": TRAIN_DIR / "train_ground_truth.tsv",
    "test_source1": TEST_DIR / "test_source1.tsv",
    "test_source2": TEST_DIR / "test_source2.tsv",
    "test_source3": TEST_DIR / "test_source3.tsv",
}


def clean_column_name(name: str) -> str:
    """
    Remove BOM / leading / trailing whitespace from column names.
    Raw data is never modified.
    """
    return (
        str(name)
        .replace("\ufeff", "")
        .strip()
    )


def scan_tsv(path: Path) -> pl.LazyFrame:
    """
    Robust lazy TSV scanner for AMLC 2026.
    Normalizes only the column names.
    """
    lf = pl.scan_csv(
        path,
        separator="\t",
        infer_schema_length=1000,
        null_values=["", "null", "NULL", "None"],
    )

    schema = lf.collect_schema()

    rename_map = {
        col: clean_column_name(col)
        for col in schema.names()
        if clean_column_name(col) != col
    }

    if rename_map:
        print(f"⚠️ Header normalization: {path.name}")
        print("   ", rename_map)
        lf = lf.rename(rename_map)

    return lf


print("=" * 90)
print("SCHEMA AUDIT")
print("=" * 90)

for name, path in files.items():

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    lf = scan_tsv(path)

    print(lf.collect_schema())

SCHEMA AUDIT

train_source1
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})

train_source2
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})

train_source3
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})

train_ground_truth
Schema({'source1_entity_id': String, 'matched_entity_ids': String})

test_source1
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})

test_source2
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})

test_source3
⚠️ Header normalization: test_source3.tsv
    {' entity_id': 'entity_id'}
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})


In [11]:
# ============================================================
# AMLC 2026 — CELL 10
# Exact row counts
# ============================================================

row_counts = {}

for name, path in files.items():

    count = (
        scan_tsv(path)
        .select(pl.len())
        .collect()
        .item()
    )

    row_counts[name] = count

print("=" * 80)
print("ROW COUNTS")
print("=" * 80)

for name, count in row_counts.items():
    print(f"{name:22s}: {count:,}")

print("\n" + "-" * 80)

train_total = sum(
    row_counts[k]
    for k in [
        "train_source1",
        "train_source2",
        "train_source3",
    ]
)

test_total = sum(
    row_counts[k]
    for k in [
        "test_source1",
        "test_source2",
        "test_source3",
    ]
)

print(f"Total train source rows : {train_total:,}")
print(f"Total test source rows  : {test_total:,}")

⚠️ Header normalization: test_source3.tsv
    {' entity_id': 'entity_id'}
ROW COUNTS
train_source1         : 2,206,821
train_source2         : 5,034,616
train_source3         : 5,285,603
train_ground_truth    : 2,206,821
test_source1          : 1,732,544
test_source2          : 4,887,273
test_source3          : 5,082,316

--------------------------------------------------------------------------------
Total train source rows : 12,527,040
Total test source rows  : 11,702,133


In [12]:
# ============================================================
# AMLC 2026 — CELL 11
# Sample records
# ============================================================

for name, path in files.items():

    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    df = (
        pl.scan_csv(
            path,
            separator="\t",
            infer_schema_length=1000,
            null_values=["", "null", "NULL", "None"]
        )
        .head(10)
        .collect()
    )

    print(df)


train_source1
shape: (10, 4)
┌──────────────┬─────────────────────────────────┬─────────────────────────────────┬─────────┐
│ entity_id    ┆ business_name                   ┆ business_address                ┆ country │
│ ---          ┆ ---                             ┆ ---                             ┆ ---     │
│ str          ┆ str                             ┆ str                             ┆ str     │
╞══════════════╪═════════════════════════════════╪═════════════════════════════════╪═════════╡
│ S1-925783039 ┆ Orelee's Barbershop             ┆ 1795 Westchester Drive, High P… ┆ US      │
│ S1-773889195 ┆ Prime Money                     ┆ 17560 Ellis Road, Tahlequah, O… ┆ US      │
│ S1-377745466 ┆ B+ Retail Inc                   ┆ 1712 Montebello Avenue, Phoeni… ┆ US      │
│ S1-133037285 ┆ Christ Chapel                   ┆ 2100 Cameron Drive, Unit APART… ┆ US      │
│ S1-755362802 ┆ Prabhav Business Center         ┆ 797, Lake Town Block A, Kolkat… ┆ India   │
│ S1-851869949 ┆ Cus

In [13]:
# ============================================================
# AMLC 2026 — CELL 12
# Source-level structural statistics
# ============================================================

source_files = [
    "train_source1",
    "train_source2",
    "train_source3",
    "test_source1",
    "test_source2",
    "test_source3",
]

for name in source_files:

    path = files[name]

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    lf = scan_tsv(path)

    stats = (
        lf.select([
            pl.len().alias("rows"),

            pl.col("entity_id")
              .n_unique()
              .alias("unique_entity_ids"),

            pl.col("business_name")
              .is_null()
              .sum()
              .alias("null_business_name"),

            pl.col("business_address")
              .is_null()
              .sum()
              .alias("null_business_address"),

            pl.col("country")
              .is_null()
              .sum()
              .alias("null_country"),

            pl.col("business_name")
              .str.len_chars()
              .mean()
              .alias("mean_name_chars"),

            pl.col("business_name")
              .str.len_chars()
              .median()
              .alias("median_name_chars"),

            pl.col("business_address")
              .str.len_chars()
              .mean()
              .alias("mean_address_chars"),

            pl.col("business_address")
              .str.len_chars()
              .median()
              .alias("median_address_chars"),

            pl.col("country")
              .n_unique()
              .alias("unique_countries"),
        ])
        .collect()
    )

    print(stats)


train_source1
shape: (1, 10)
┌─────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ rows    ┆ unique_ent ┆ null_busi ┆ null_busi ┆ … ┆ median_na ┆ mean_addr ┆ median_ad ┆ unique_co │
│ ---     ┆ ity_ids    ┆ ness_name ┆ ness_addr ┆   ┆ me_chars  ┆ ess_chars ┆ dress_cha ┆ untries   │
│ u32     ┆ ---        ┆ ---       ┆ ess       ┆   ┆ ---       ┆ ---       ┆ rs        ┆ ---       │
│         ┆ u32        ┆ u32       ┆ ---       ┆   ┆ f64       ┆ f64       ┆ ---       ┆ u32       │
│         ┆            ┆           ┆ u32       ┆   ┆           ┆           ┆ f64       ┆           │
╞═════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2206821 ┆ 2206821    ┆ 0         ┆ 0         ┆ … ┆ 24.0      ┆ 52.066213 ┆ 41.0      ┆ 2         │
└─────────┴────────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴───────────┘

train_source2
shape: (1, 10)
┌─────────┬────────────┬───────

In [14]:
# ============================================================
# AMLC 2026 — CELL 13
# Country distribution
# ============================================================

for name in source_files:

    path = files[name]

    lf = scan_tsv(path)

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    result = (
        lf.group_by("country")
          .agg(pl.len().alias("count"))
          .sort("count", descending=True)
          .collect()
    )

    print(result)


train_source1
shape: (2, 2)
┌─────────┬─────────┐
│ country ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ US      ┆ 1323633 │
│ India   ┆ 883188  │
└─────────┴─────────┘

train_source2
shape: (2, 2)
┌─────────┬─────────┐
│ country ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ US      ┆ 3016817 │
│ India   ┆ 2017799 │
└─────────┴─────────┘

train_source3
shape: (2, 2)
┌─────────┬─────────┐
│ country ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ US      ┆ 3170056 │
│ India   ┆ 2115547 │
└─────────┴─────────┘

test_source1
shape: (3, 2)
┌─────────┬────────┐
│ country ┆ count  │
│ ---     ┆ ---    │
│ str     ┆ u32    │
╞═════════╪════════╡
│ India   ┆ 809986 │
│ US      ┆ 663106 │
│ France  ┆ 259452 │
└─────────┴────────┘

test_source2
shape: (3, 2)
┌─────────┬─────────┐
│ country ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ India   ┆ 2312565 │
│ US    

In [15]:
# ============================================================
# AMLC 2026 — CELL 14
# Ground-truth match cardinality
# ============================================================

GT_PATH = TRAIN_DIR / "train_ground_truth.tsv"

gt = pl.read_csv(
    GT_PATH,
    separator="\t",
    infer_schema_length=1000,
    null_values=["", "null", "NULL", "None"]
)

print(gt.head())

print("\nSchema:")
print(gt.schema)

gt_cardinality = (
    gt.with_columns(
        pl.when(
            pl.col("matched_entity_ids").is_null()
            | (pl.col("matched_entity_ids").str.strip_chars() == "")
        )
        .then(0)
        .otherwise(
            pl.col("matched_entity_ids")
            .str.split(",")
            .list.len()
        )
        .alias("match_count")
    )
)

print("\nMatch-count distribution:")
print(
    gt_cardinality
    .group_by("match_count")
    .agg(pl.len().alias("s1_count"))
    .sort("match_count")
)

print("\nSingleton rate:")
singleton_rate = (
    gt_cardinality
    .select(
        (pl.col("match_count") == 0)
        .mean()
        .alias("singleton_fraction")
    )
    .item()
)

print(f"{singleton_rate:.4%}")

shape: (5, 2)
┌───────────────────┬─────────────────────────────────┐
│ source1_entity_id ┆ matched_entity_ids              │
│ ---               ┆ ---                             │
│ str               ┆ str                             │
╞═══════════════════╪═════════════════════════════════╡
│ S1-965667         ┆ S2-681193310,S2-743505751,S3-7… │
│ S1-55344266       ┆ S2-249013014,S2-197070651,S3-4… │
│ S1-343815751      ┆ S2-790675320,S2-479876582,S3-8… │
│ S1-656753428      ┆ S2-153058913,S2-24659151,S3-67… │
│ S1-102811957      ┆ S2-478959098,S2-553508714,S2-6… │
└───────────────────┴─────────────────────────────────┘

Schema:
Schema({'source1_entity_id': String, 'matched_entity_ids': String})

Match-count distribution:
shape: (12, 2)
┌─────────────┬──────────┐
│ match_count ┆ s1_count │
│ ---         ┆ ---      │
│ u32         ┆ u32      │
╞═════════════╪══════════╡
│ 0           ┆ 123247   │
│ 1           ┆ 119157   │
│ 2           ┆ 375212   │
│ 3           ┆ 530841   │
│ 4     

In [16]:
# ============================================================
# AMLC 2026 — CELL 15
# Match composition: S2 vs S3
# ============================================================

def classify_match_set(s):

    if s is None or str(s).strip() == "":
        return "none"

    ids = [x.strip() for x in str(s).split(",") if x.strip()]

    has_s2 = any(x.startswith("S2-") for x in ids)
    has_s3 = any(x.startswith("S3-") for x in ids)

    if has_s2 and has_s3:
        return "both"
    elif has_s2:
        return "S2_only"
    elif has_s3:
        return "S3_only"
    else:
        return "other"


composition = (
    gt.with_columns(
        pl.col("matched_entity_ids")
        .map_elements(classify_match_set, return_dtype=pl.String)
        .alias("match_type")
    )
    .group_by("match_type")
    .agg(pl.len().alias("s1_count"))
    .sort("s1_count", descending=True)
)

print(composition)

shape: (4, 2)
┌────────────┬──────────┐
│ match_type ┆ s1_count │
│ ---        ┆ ---      │
│ str        ┆ u32      │
╞════════════╪══════════╡
│ both       ┆ 1776047  │
│ S3_only    ┆ 164498   │
│ S2_only    ┆ 143029   │
│ null       ┆ 123247   │
└────────────┴──────────┘


In [17]:
# ============================================================
# AMLC 2026 — CELL 16
# Cross-entity exclusivity check
# ============================================================

edges = (
    gt
    .with_columns(
        pl.when(
            pl.col("matched_entity_ids").is_null()
            | (pl.col("matched_entity_ids").str.strip_chars() == "")
        )
        .then(pl.lit(None))
        .otherwise(
            pl.col("matched_entity_ids").str.split(",")
        )
        .alias("matched_list")
    )
    .explode("matched_list")
    .filter(
        pl.col("matched_list").is_not_null()
        & (pl.col("matched_list").str.strip_chars() != "")
    )
    .select([
        "source1_entity_id",
        pl.col("matched_list").alias("matched_entity_id")
    ])
)

print("Total positive edges:", edges.height)

uniqueness = (
    edges
    .group_by("matched_entity_id")
    .agg(
        pl.col("source1_entity_id").n_unique().alias("s1_count")
    )
)

print("\nHow many S2/S3 entities map to:")
print(
    uniqueness
    .group_by("s1_count")
    .agg(pl.len().alias("entity_count"))
    .sort("s1_count")
)

print("\nMaximum number of S1 entities sharing one S2/S3 ID:",
      uniqueness["s1_count"].max())

Total positive edges: 7638365

How many S2/S3 entities map to:
shape: (1, 2)
┌──────────┬──────────────┐
│ s1_count ┆ entity_count │
│ ---      ┆ ---          │
│ u32      ┆ u32          │
╞══════════╪══════════════╡
│ 1        ┆ 7638365      │
└──────────┴──────────────┘

Maximum number of S1 entities sharing one S2/S3 ID: 1


In [18]:
# ============================================================
# AMLC 2026 — CELL 17
# Full match cardinality table
# ============================================================

print(
    gt_cardinality
    .group_by("match_count")
    .agg(pl.len().alias("s1_count"))
    .sort("match_count")
    .to_pandas()
    .to_string(index=False)
)

 match_count  s1_count
           0    123247
           1    119157
           2    375212
           3    530841
           4    484115
           5    321957
           6    164868
           7     63968
           8     18680
           9      4205
          10       534
          11        37


In [19]:
# ============================================================
# AMLC 2026 — CELL 18
# Detailed text-length distributions
# ============================================================

for name in [
    "train_source1",
    "train_source2",
    "train_source3",
]:

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    lf = (
        scan_tsv(files[name])
        .select([
            pl.col("business_name")
              .str.len_chars()
              .alias("name_len"),

            pl.col("business_address")
              .fill_null("")
              .str.len_chars()
              .alias("address_len"),
        ])
    )

    result = (
        lf.select([
            pl.col("name_len").quantile(0.01).alias("name_p01"),
            pl.col("name_len").quantile(0.10).alias("name_p10"),
            pl.col("name_len").quantile(0.25).alias("name_p25"),
            pl.col("name_len").median().alias("name_p50"),
            pl.col("name_len").quantile(0.75).alias("name_p75"),
            pl.col("name_len").quantile(0.90).alias("name_p90"),
            pl.col("name_len").quantile(0.99).alias("name_p99"),

            pl.col("address_len").quantile(0.01).alias("addr_p01"),
            pl.col("address_len").quantile(0.10).alias("addr_p10"),
            pl.col("address_len").quantile(0.25).alias("addr_p25"),
            pl.col("address_len").median().alias("addr_p50"),
            pl.col("address_len").quantile(0.75).alias("addr_p75"),
            pl.col("address_len").quantile(0.90).alias("addr_p90"),
            pl.col("address_len").quantile(0.99).alias("addr_p99"),
        ])
        .collect()
    )

    print(result)


train_source1
shape: (1, 14)
┌──────────┬──────────┬──────────┬──────────┬───┬──────────┬──────────┬──────────┬──────────┐
│ name_p01 ┆ name_p10 ┆ name_p25 ┆ name_p50 ┆ … ┆ addr_p50 ┆ addr_p75 ┆ addr_p90 ┆ addr_p99 │
│ ---      ┆ ---      ┆ ---      ┆ ---      ┆   ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ f64      ┆ f64      ┆ f64      ┆ f64      ┆   ┆ f64      ┆ f64      ┆ f64      ┆ f64      │
╞══════════╪══════════╪══════════╪══════════╪═══╪══════════╪══════════╪══════════╪══════════╡
│ 8.0      ┆ 14.0     ┆ 18.0     ┆ 24.0     ┆ … ┆ 41.0     ┆ 70.0     ┆ 90.0     ┆ 124.0    │
└──────────┴──────────┴──────────┴──────────┴───┴──────────┴──────────┴──────────┴──────────┘

train_source2
shape: (1, 14)
┌──────────┬──────────┬──────────┬──────────┬───┬──────────┬──────────┬──────────┬──────────┐
│ name_p01 ┆ name_p10 ┆ name_p25 ┆ name_p50 ┆ … ┆ addr_p50 ┆ addr_p75 ┆ addr_p90 ┆ addr_p99 │
│ ---      ┆ ---      ┆ ---      ┆ ---      ┆   ┆ ---      ┆ ---      ┆ ---      ┆ ---      │


In [20]:
# ============================================================
# AMLC 2026 — CELL 19
# Unicode / script diagnostics
# ============================================================

import re

def script_flags(text):

    if text is None:
        text = ""

    text = str(text)

    return {
        "latin": int(bool(re.search(r"[A-Za-z]", text))),
        "devanagari": int(bool(re.search(r"[\u0900-\u097F]", text))),
        "tamil": int(bool(re.search(r"[\u0B80-\u0BFF]", text))),
        "kannada": int(bool(re.search(r"[\u0C80-\u0CFF]", text))),
        "telugu": int(bool(re.search(r"[\u0C00-\u0C7F]", text))),
        "malayalam": int(bool(re.search(r"[\u0D00-\u0D7F]", text))),
        "bengali": int(bool(re.search(r"[\u0980-\u09FF]", text))),
        "gujarati": int(bool(re.search(r"[\u0A80-\u0AFF]", text))),
        "gurmukhi": int(bool(re.search(r"[\u0A00-\u0A7F]", text))),
        "arabic": int(bool(re.search(r"[\u0600-\u06FF]", text))),
        "non_ascii": int(any(ord(c) > 127 for c in text)),
    }


# Sample-based first pass.
# We deliberately do NOT materialize all 12M records yet.
for name in [
    "train_source1",
    "train_source2",
    "train_source3",
]:
    sample = (
        scan_tsv(files[name])
        .select("business_name")
        .head(200_000)
        .collect()
        .to_series()
        .to_list()
    )

    counts = {
        key: 0
        for key in script_flags("").keys()
    }

    for value in sample:
        flags = script_flags(value)
        for key, v in flags.items():
            counts[key] += v

    print("\n", name)
    print("-" * 60)

    for key, value in counts.items():
        print(f"{key:15s}: {value / len(sample):8.2%}")


 train_source1
------------------------------------------------------------
latin          :  100.00%
devanagari     :    0.00%
tamil          :    0.00%
kannada        :    0.00%
telugu         :    0.00%
malayalam      :    0.00%
bengali        :    0.00%
gujarati       :    0.00%
gurmukhi       :    0.00%
arabic         :    0.00%
non_ascii      :    0.00%

 train_source2
------------------------------------------------------------
latin          :   91.11%
devanagari     :    5.27%
tamil          :    0.62%
kannada        :    0.73%
telugu         :    0.78%
malayalam      :    0.37%
bengali        :    0.62%
gujarati       :    0.60%
gurmukhi       :    0.13%
arabic         :    0.00%
non_ascii      :   15.07%

 train_source3
------------------------------------------------------------
latin          :   95.38%
devanagari     :    3.02%
tamil          :    0.38%
kannada        :    0.42%
telugu         :    0.46%
malayalam      :    0.19%
bengali        :    0.35%
gujarati       

In [21]:
# ============================================================
# AMLC 2026 — CELL 20
# Name collision analysis
# ============================================================

import re
import unicodedata

LEGAL_SUFFIXES = {
    "limited", "ltd", "ltd.",
    "private", "pvt",
    "inc", "inc.",
    "corporation", "corp", "corp.",
    "company", "co", "co.",
    "llc", "llp",
    "limitedliabilitycompany",
    "private limited",
}

def simple_normalize_name(s):

    if s is None:
        return ""

    s = unicodedata.normalize("NFKC", str(s)).casefold()

    s = s.replace("&", " and ")

    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    tokens = [
        t for t in s.split()
        if t not in LEGAL_SUFFIXES
    ]

    return " ".join(tokens)


# IMPORTANT:
# Start with a 300k sample.
# We are measuring collision structure, not building the final index.
sample = (
    scan_tsv(files["train_source1"])
    .select([
        "entity_id",
        "business_name"
    ])
    .head(300_000)
    .collect()
)

sample = sample.with_columns(
    pl.col("business_name")
    .map_elements(
        simple_normalize_name,
        return_dtype=pl.String
    )
    .alias("name_norm")
)

collisions = (
    sample
    .group_by("name_norm")
    .agg([
        pl.len().alias("count"),
        pl.col("entity_id").n_unique().alias("unique_ids")
    ])
    .sort("count", descending=True)
)

print(collisions.head(50))

shape: (50, 3)
┌──────────────────────────┬───────┬────────────┐
│ name_norm                ┆ count ┆ unique_ids │
│ ---                      ┆ ---   ┆ ---        │
│ str                      ┆ u32   ┆ u32        │
╞══════════════════════════╪═══════╪════════════╡
│ meridian                 ┆ 64    ┆ 64         │
│ redwood                  ┆ 49    ┆ 49         │
│ aurora                   ┆ 47    ┆ 47         │
│ beacon                   ┆ 47    ┆ 47         │
│ summit                   ┆ 46    ┆ 46         │
│ …                        ┆ …     ┆ …          │
│ behavioral health center ┆ 38    ┆ 38         │
│ urgent care health       ┆ 38    ┆ 38         │
│ cypress                  ┆ 38    ┆ 38         │
│ cascade                  ┆ 38    ┆ 38         │
│ atlas                    ┆ 38    ┆ 38         │
└──────────────────────────┴───────┴────────────┘


In [53]:
# ============================================================
# AMLC 2026 — CELL 21
# Unicode transliteration support
# ============================================================

!pip -q install anyascii

In [23]:
from anyascii import anyascii

print(anyascii("श्री साईं इन्फ्राटेक"))
print(anyascii("ஈஸ்டர்ன் கன்சல்டன்சி"))
print(anyascii("શ્રી સાઈ"))

sri saim inphratek
istrn kncltnci
sri sai


In [24]:
# ============================================================
# AMLC 2026 — CELL 22
# Dual-view name normalization
# ============================================================

import re
import unicodedata
from anyascii import anyascii

LEGAL_SUFFIXES = {
    "limited",
    "ltd",
    "private",
    "pvt",
    "inc",
    "corporation",
    "corp",
    "company",
    "co",
    "llc",
    "llp",
    "sarl",
    "sasu",
    "sas",
    "plc",
    "gmbh",
}


def normalize_unicode(text):
    """
    Preserve native script.
    Used for exact Unicode / token comparisons.
    """
    if text is None:
        return ""

    text = unicodedata.normalize("NFKC", str(text)).casefold()
    text = text.replace("&", " and ")

    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_translit(text):
    """
    Transliterate to ASCII, then normalize.
    Used for cross-script matching.
    """
    if text is None:
        return ""

    text = anyascii(str(text))
    text = unicodedata.normalize("NFKC", text).casefold()
    text = text.replace("&", " and ")

    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_name_views(text):
    """
    Returns both representations.
    """
    native = normalize_unicode(text)
    translit = normalize_translit(text)

    return native, translit


examples = [
    "Orelee's Barbershop",
    "SHIVSHAKTI VIDYALAYA",
    "श्री साईं इन्फ्राटेक",
    "ஈஸ்டர்ன் கன்சல்டன்சி",
    "Fractales Amis Groupe S.A.S",
]

for x in examples:
    native, translit = normalize_name_views(x)

    print("\nRAW     :", x)
    print("NATIVE  :", native)
    print("TRANSLIT:", translit)


RAW     : Orelee's Barbershop
NATIVE  : orelee s barbershop
TRANSLIT: orelee s barbershop

RAW     : SHIVSHAKTI VIDYALAYA
NATIVE  : shivshakti vidyalaya
TRANSLIT: shivshakti vidyalaya

RAW     : श्री साईं इन्फ्राटेक
NATIVE  : श र स ई इन फ र ट क
TRANSLIT: sri saim inphratek

RAW     : ஈஸ்டர்ன் கன்சல்டன்சி
NATIVE  : ஈஸ டர ன கன சல டன ச
TRANSLIT: istrn kncltnci

RAW     : Fractales Amis Groupe S.A.S
NATIVE  : fractales amis groupe s a s
TRANSLIT: fractales amis groupe s a s


In [25]:
# ============================================================
# AMLC 2026 — CELL 23
# Ground-truth positive edge table
# ============================================================

gt_edges = (
    gt
    .with_columns(
        pl.when(
            pl.col("matched_entity_ids").is_null()
            | (pl.col("matched_entity_ids").str.strip_chars() == "")
        )
        .then(pl.lit(None))
        .otherwise(
            pl.col("matched_entity_ids").str.split(",")
        )
        .alias("matched_list")
    )
    .explode("matched_list")
    .filter(
        pl.col("matched_list").is_not_null()
        & (pl.col("matched_list").str.strip_chars() != "")
    )
    .select([
        "source1_entity_id",
        pl.col("matched_list").alias("matched_entity_id"),
    ])
    .with_columns(
        pl.when(
            pl.col("matched_entity_id").str.starts_with("S2-")
        )
        .then(pl.lit("S2"))
        .otherwise(pl.lit("S3"))
        .alias("matched_source")
    )
)

print(gt_edges.head(20))
print("\nRows:", gt_edges.height)
print("\nSource counts:")
print(
    gt_edges
    .group_by("matched_source")
    .agg(pl.len().alias("edges"))
)

shape: (20, 3)
┌───────────────────┬───────────────────┬────────────────┐
│ source1_entity_id ┆ matched_entity_id ┆ matched_source │
│ ---               ┆ ---               ┆ ---            │
│ str               ┆ str               ┆ str            │
╞═══════════════════╪═══════════════════╪════════════════╡
│ S1-965667         ┆ S2-681193310      ┆ S2             │
│ S1-965667         ┆ S2-743505751      ┆ S2             │
│ S1-965667         ┆ S3-775321672      ┆ S3             │
│ S1-965667         ┆ S3-11291185       ┆ S3             │
│ S1-965667         ┆ S3-860443364      ┆ S3             │
│ …                 ┆ …                 ┆ …              │
│ S1-102811957      ┆ S2-478959098      ┆ S2             │
│ S1-102811957      ┆ S2-553508714      ┆ S2             │
│ S1-102811957      ┆ S2-625774905      ┆ S2             │
│ S1-102811957      ┆ S3-728090388      ┆ S3             │
│ S1-102811957      ┆ S3-928796641      ┆ S3             │
└───────────────────┴───────────────────┴

In [26]:
# ============================================================
# AMLC 2026 — CELL 24
# Deterministic 100k S1 evaluation sample
# ============================================================

EVAL_N = 100_000
SEED = 2026

# Sample S1 IDs directly from the already-loaded ground truth.
eval_gt_ids = (
    gt
    .sample(
        n=EVAL_N,
        seed=SEED
    )
    .select("source1_entity_id")
)

eval_id_list = eval_gt_ids["source1_entity_id"].to_list()

print("Selected S1 evaluation entities:", len(eval_id_list))

# Pull the corresponding source-1 records.
s1_eval = (
    scan_tsv(files["train_source1"])
    .filter(
        pl.col("entity_id").is_in(eval_id_list)
    )
    .collect()
)

print("\nS1 evaluation shape:", s1_eval.shape)
print("\nSample:")
print(s1_eval.head(10))

Selected S1 evaluation entities: 100000

S1 evaluation shape: (100000, 4)

Sample:
shape: (10, 4)
┌──────────────┬─────────────────────────────────┬─────────────────────────────────┬─────────┐
│ entity_id    ┆ business_name                   ┆ business_address                ┆ country │
│ ---          ┆ ---                             ┆ ---                             ┆ ---     │
│ str          ┆ str                             ┆ str                             ┆ str     │
╞══════════════╪═════════════════════════════════╪═════════════════════════════════╪═════════╡
│ S1-913301865 ┆ Technologies Pratyaksh Finserv… ┆ Village Kurali, Sabhapur Road … ┆ India   │
│ S1-424529930 ┆ Griffin International Center    ┆ 536 College Avenue, City Of Wa… ┆ US      │
│ S1-846529766 ┆ Pocius, Herrera & Lima Connect… ┆ 4831 Beaumont Street, Simi Val… ┆ US      │
│ S1-8092452   ┆ Urban Nails!                    ┆ 1262 Grandstaff Avenue, Lancas… ┆ US      │
│ S1-831843558 ┆ Hernandez Regional Aktiengesel

In [27]:
# ============================================================
# AMLC 2026 — CELL 25
# Evaluation subset ground truth
# ============================================================

eval_gt = (
    gt_edges
    .filter(
        pl.col("source1_entity_id")
        .is_in(eval_id_list)
    )
)

print("S1 eval entities :", s1_eval.height)
print("Positive edges   :", eval_gt.height)

print("\nPositive edges by source:")
print(
    eval_gt
    .group_by("matched_source")
    .agg(pl.len().alias("edges"))
    .sort("matched_source")
)

S1 eval entities : 100000
Positive edges   : 345980

Positive edges by source:
shape: (2, 2)
┌────────────────┬────────┐
│ matched_source ┆ edges  │
│ ---            ┆ ---    │
│ str            ┆ u32    │
╞════════════════╪════════╡
│ S2             ┆ 167274 │
│ S3             ┆ 178706 │
└────────────────┴────────┘


In [28]:
# ============================================================
# AMLC 2026 — CELL 26
# Prepare source-2/source-3 name tables for blocking experiment
# ============================================================

s2_eval_pool = (
    scan_tsv(files["train_source2"])
    .select([
        "entity_id",
        "business_name",
        "country",
    ])
    .collect()
)

s3_eval_pool = (
    scan_tsv(files["train_source3"])
    .select([
        "entity_id",
        "business_name",
        "country",
    ])
    .collect()
)

print("S2:", s2_eval_pool.shape)
print("S3:", s3_eval_pool.shape)

print("\nRAM:")
!free -h

S2: (5034616, 3)
S3: (5285603, 3)

RAM:
               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.1Gi       129Mi       4.7Mi       7.0Gi        10Gi
Swap:             0B          0B          0B


In [29]:
# ============================================================
# AMLC 2026 — CELL 27
# Memory cleanup before large-scale indexing
# ============================================================

import gc

for name in ["s2_eval_pool", "s3_eval_pool"]:
    if name in globals():
        del globals()[name]

gc.collect()

print("Memory after cleanup:")
!free -h

Memory after cleanup:
               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.1Gi       133Mi       4.7Mi       7.0Gi        10Gi
Swap:             0B          0B          0B


In [30]:
# ============================================================
# AMLC 2026 — CELL 28
# Fast vectorized name normalization
#
# This is ONLY the first blocking oracle.
# It is NOT our final normalizer.
# ============================================================

import polars as pl

LEGAL_SUFFIX_PATTERN = (
    r"(?i)\b("
    r"private\s+limited|"
    r"private|"
    r"pvt\s+ltd|"
    r"pvt|"
    r"limited|"
    r"ltd|"
    r"corporation|"
    r"corp|"
    r"company|"
    r"co|"
    r"incorporated|"
    r"inc|"
    r"llc|"
    r"llp|"
    r"sarl|"
    r"sasu|"
    r"sas|"
    r"plc|"
    r"gmbh"
    r")\b"
)


def fast_name_expr(column="business_name"):
    """
    Vectorized normalization.

    Keeps Unicode letters/digits.
    Removes punctuation.
    Normalizes ampersand.
    Removes common legal suffixes.
    Collapses whitespace.

    We preserve Unicode because cross-script handling
    is a separate experiment.
    """

    return (
        pl.col(column)
        .fill_null("")
        .str.to_lowercase()
        .str.replace_all("&", " and ")
        .str.replace_all(r"[^\p{L}\p{N}\s]", " ")
        .str.replace_all(LEGAL_SUFFIX_PATTERN, " ")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )


# Quick sanity check
test_names = pl.DataFrame({
    "business_name": [
        "ABC Private Limited",
        "ABC Pvt. Ltd.",
        "A&B Corporation",
        "Fractales Amis Groupe S.A.S",
        "श्री साईं इन्फ्राटेक",
    ]
})

print(
    test_names.with_columns(
        fast_name_expr().alias("name_norm")
    )
)

shape: (5, 2)
┌─────────────────────────────┬─────────────────────────────┐
│ business_name               ┆ name_norm                   │
│ ---                         ┆ ---                         │
│ str                         ┆ str                         │
╞═════════════════════════════╪═════════════════════════════╡
│ ABC Private Limited         ┆ abc                         │
│ ABC Pvt. Ltd.               ┆ abc                         │
│ A&B Corporation             ┆ a and b                     │
│ Fractales Amis Groupe S.A.S ┆ fractales amis groupe s a s │
│ श्री साईं इन्फ्राटेक             ┆ श र स ई इन फ र ट क          │
└─────────────────────────────┴─────────────────────────────┘


In [31]:
# ============================================================
# AMLC 2026 — CELL 29
# Normalize evaluation S1 names
# ============================================================

s1_eval_norm = (
    s1_eval
    .select([
        "entity_id",
        "business_name",
        "country",
    ])
    .with_columns(
        fast_name_expr().alias("name_norm")
    )
)

print(s1_eval_norm.head(20))

print("\nEmpty normalized names:")
print(
    s1_eval_norm
    .filter(pl.col("name_norm") == "")
    .height
)

shape: (20, 4)
┌──────────────┬─────────────────────────────────┬─────────┬─────────────────────────────────┐
│ entity_id    ┆ business_name                   ┆ country ┆ name_norm                       │
│ ---          ┆ ---                             ┆ ---     ┆ ---                             │
│ str          ┆ str                             ┆ str     ┆ str                             │
╞══════════════╪═════════════════════════════════╪═════════╪═════════════════════════════════╡
│ S1-913301865 ┆ Technologies Pratyaksh Finserv… ┆ India   ┆ technologies pratyaksh finserv… │
│ S1-424529930 ┆ Griffin International Center    ┆ US      ┆ griffin international center    │
│ S1-846529766 ┆ Pocius, Herrera & Lima Connect… ┆ US      ┆ pocius herrera and lima connec… │
│ S1-8092452   ┆ Urban Nails!                    ┆ US      ┆ urban nails                     │
│ S1-831843558 ┆ Hernandez Regional Aktiengesel… ┆ US      ┆ hernandez regional aktiengesel… │
│ …            ┆ …                 

In [32]:
# ============================================================
# AMLC 2026 — CELL 30
# Streaming S2 normalized-name index
# ============================================================

from pathlib import Path

BLOCKING_DIR = Path("/content/AMLC2026/cache/blocking")
BLOCKING_DIR.mkdir(parents=True, exist_ok=True)

S2_NAME_INDEX = BLOCKING_DIR / "train_s2_name_norm.parquet"

print("Building S2 normalized-name index...")
print("Output:", S2_NAME_INDEX)

(
    scan_tsv(files["train_source2"])
    .select([
        "entity_id",
        "country",
        "business_name",
    ])
    .with_columns(
        fast_name_expr().alias("name_norm")
    )
    .select([
        "entity_id",
        "country",
        "name_norm",
    ])
    .sink_parquet(
        S2_NAME_INDEX,
        compression="zstd",
        statistics=True,
    )
)

print("\n✅ S2 index written.")
print(f"Size: {S2_NAME_INDEX.stat().st_size / (1024**2):.1f} MB")

Building S2 normalized-name index...
Output: /content/AMLC2026/cache/blocking/train_s2_name_norm.parquet

✅ S2 index written.
Size: 77.2 MB


In [33]:
# ============================================================
# AMLC 2026 — CELL 31
# Streaming S3 normalized-name index
# ============================================================

S3_NAME_INDEX = BLOCKING_DIR / "train_s3_name_norm.parquet"

print("Building S3 normalized-name index...")
print("Output:", S3_NAME_INDEX)

(
    scan_tsv(files["train_source3"])
    .select([
        "entity_id",
        "country",
        "business_name",
    ])
    .with_columns(
        fast_name_expr().alias("name_norm")
    )
    .select([
        "entity_id",
        "country",
        "name_norm",
    ])
    .sink_parquet(
        S3_NAME_INDEX,
        compression="zstd",
        statistics=True,
    )
)

print("\n✅ S3 index written.")
print(f"Size: {S3_NAME_INDEX.stat().st_size / (1024**2):.1f} MB")

Building S3 normalized-name index...
Output: /content/AMLC2026/cache/blocking/train_s3_name_norm.parquet

✅ S3 index written.
Size: 80.4 MB


In [34]:
# ============================================================
# AMLC 2026 — CELL 32
# Verify normalized indexes
# ============================================================

print("S2 index:")
s2_idx = pl.scan_parquet(S2_NAME_INDEX)

print(s2_idx.collect_schema())

print(
    s2_idx
    .select(pl.len())
    .collect()
)

print("\nS3 index:")
s3_idx = pl.scan_parquet(S3_NAME_INDEX)

print(s3_idx.collect_schema())

print(
    s3_idx
    .select(pl.len())
    .collect()
)

print("\nDisk:")
!df -h /content

S2 index:
Schema({'entity_id': String, 'country': String, 'name_norm': String})
shape: (1, 1)
┌─────────┐
│ len     │
│ ---     │
│ u32     │
╞═════════╡
│ 5034616 │
└─────────┘

S3 index:
Schema({'entity_id': String, 'country': String, 'name_norm': String})
shape: (1, 1)
┌─────────┐
│ len     │
│ ---     │
│ u32     │
╞═════════╡
│ 5285603 │
└─────────┘

Disk:
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   53G   60G  47% /


In [36]:
# ============================================================
# AMLC 2026 — CELL 33
# Exact normalized-name blocker recall
# ============================================================

# Convert the small evaluation-side tables to LazyFrames.
s1_norm_lookup_lf = s1_norm_lookup.lazy()
eval_gt_lf = eval_gt.lazy()

# -------------------------
# S2 positive edges
# -------------------------

s2_edge_eval = (
    eval_gt_lf
    .filter(pl.col("matched_source") == "S2")
    .join(
        s2_idx.select([
            pl.col("entity_id").alias("matched_entity_id"),
            pl.col("name_norm").alias("matched_name_norm"),
        ]),
        on="matched_entity_id",
        how="left",
    )
    .join(
        s1_norm_lookup_lf,
        on="source1_entity_id",
        how="left",
    )
    .with_columns(
        (
            (pl.col("name_norm") != "")
            & (pl.col("name_norm") == pl.col("matched_name_norm"))
        ).alias("name_hit")
    )
)

# -------------------------
# S3 positive edges
# -------------------------

s3_edge_eval = (
    eval_gt_lf
    .filter(pl.col("matched_source") == "S3")
    .join(
        s3_idx.select([
            pl.col("entity_id").alias("matched_entity_id"),
            pl.col("name_norm").alias("matched_name_norm"),
        ]),
        on="matched_entity_id",
        how="left",
    )
    .join(
        s1_norm_lookup_lf,
        on="source1_entity_id",
        how="left",
    )
    .with_columns(
        (
            (pl.col("name_norm") != "")
            & (pl.col("name_norm") == pl.col("matched_name_norm"))
        ).alias("name_hit")
    )
)

edge_eval = pl.concat([
    s2_edge_eval,
    s3_edge_eval,
]).collect()

print("=" * 80)
print("EDGE RECALL")
print("=" * 80)

print(
    edge_eval
    .group_by("matched_source")
    .agg([
        pl.len().alias("true_edges"),
        pl.col("name_hit").sum().alias("recovered_edges"),
        pl.col("name_hit").mean().alias("edge_recall"),
    ])
)

print("\nALL SOURCES")

print(
    edge_eval
    .select([
        pl.len().alias("true_edges"),
        pl.col("name_hit").sum().alias("recovered_edges"),
        pl.col("name_hit").mean().alias("edge_recall"),
    ])
)

NameError: name 's1_norm_lookup' is not defined

In [37]:
# ============================================================
# AMLC 2026 — CELL 34
# Per-S1 full candidate recall
# ============================================================

per_s1_recall = (
    edge_eval
    .group_by("source1_entity_id")
    .agg([
        pl.len().alias("true_match_count"),
        pl.col("name_hit").sum().alias("recovered_match_count"),
    ])
    .with_columns(
        (
            pl.col("recovered_match_count")
            == pl.col("true_match_count")
        ).alias("fully_recovered")
    )
)

print("=" * 80)
print("S1 FULL-SET RECALL")
print("=" * 80)

print(
    per_s1_recall
    .select([
        pl.len().alias("evaluated_s1"),
        pl.col("fully_recovered").sum().alias("fully_recovered_s1"),
        pl.col("fully_recovered").mean().alias("full_recall"),
    ])
)

print("\nFull recall by true cardinality:")

print(
    per_s1_recall
    .group_by("true_match_count")
    .agg([
        pl.len().alias("s1_count"),
        pl.col("fully_recovered").mean().alias("full_recall"),
    ])
    .sort("true_match_count")
)

NameError: name 'edge_eval' is not defined

In [38]:
# ============================================================
# AMLC 2026 — CELL 35
# Candidate-volume statistics for exact normalized-name blocking
# ============================================================

s1_norm_for_counts_lf = s1_norm_lookup.lazy()


def candidate_count_by_name_lazy(
    s1_norm_df_lf,
    source_index_lf,
    prefix,
):
    counts = (
        source_index_lf
        .filter(pl.col("name_norm") != "")
        .group_by("name_norm")
        .agg(
            pl.len().alias(f"{prefix}_candidate_count")
        )
    )

    return (
        s1_norm_df_lf
        .select([
            "source1_entity_id",
            "name_norm",
        ])
        .join(
            counts,
            on="name_norm",
            how="left",
        )
        .with_columns(
            pl.col(f"{prefix}_candidate_count")
            .fill_null(0)
        )
    )


s2_counts = candidate_count_by_name_lazy(
    s1_norm_for_counts_lf,
    s2_idx,
    "s2",
)

s3_counts = candidate_count_by_name_lazy(
    s1_norm_for_counts_lf,
    s3_idx,
    "s3",
)

candidate_counts = (
    s2_counts
    .join(
        s3_counts.select([
            "source1_entity_id",
            "s3_candidate_count",
        ]),
        on="source1_entity_id",
    )
    .with_columns(
        (
            pl.col("s2_candidate_count")
            + pl.col("s3_candidate_count")
        ).alias("total_candidate_count")
    )
)

print("=" * 80)
print("EXACT NORMALIZED-NAME CANDIDATE VOLUME")
print("=" * 80)

result = (
    candidate_counts
    .select([
        pl.col("total_candidate_count")
            .mean()
            .alias("mean"),

        pl.col("total_candidate_count")
            .median()
            .alias("median"),

        pl.col("total_candidate_count")
            .quantile(0.90)
            .alias("p90"),

        pl.col("total_candidate_count")
            .quantile(0.95)
            .alias("p95"),

        pl.col("total_candidate_count")
            .quantile(0.99)
            .alias("p99"),

        pl.col("total_candidate_count")
            .max()
            .alias("max"),

        (
            pl.col("total_candidate_count") == 0
        )
        .mean()
        .alias("zero_candidate_fraction"),
    ])
    .collect()
)

print(result)

NameError: name 's1_norm_lookup' is not defined

In [ ]:
# ============================================================
# AMLC 2026 — CELL 36
# Inspect missed positive matches
# ============================================================

MISSED_SAMPLE_N = 1000

missed_edges = (
    edge_eval
    .filter(
        ~pl.col("name_hit")
    )
    .sample(
        n=min(MISSED_SAMPLE_N, edge_eval.filter(~pl.col("name_hit")).height),
        seed=2026,
    )
)

print("Missed positive edges sampled:", missed_edges.height)
print(missed_edges.head(20))

In [ ]:
# ============================================================
# AMLC 2026 — CELL 37
# Pull S1 + matched source records for missed edges
# ============================================================

miss_ids = missed_edges["source1_entity_id"].to_list()
matched_ids = missed_edges["matched_entity_id"].to_list()

# S1 records
s1_missed = (
    scan_tsv(files["train_source1"])
    .filter(
        pl.col("entity_id").is_in(miss_ids)
    )
    .select([
        pl.col("entity_id").alias("source1_entity_id"),
        pl.col("business_name").alias("s1_name"),
        pl.col("business_address").alias("s1_address"),
        pl.col("country").alias("s1_country"),
    ])
    .collect()
)

# S2 records
s2_missed = (
    scan_tsv(files["train_source2"])
    .filter(
        pl.col("entity_id").is_in(matched_ids)
    )
    .select([
        pl.col("entity_id").alias("matched_entity_id"),
        pl.col("business_name").alias("matched_name"),
        pl.col("business_address").alias("matched_address"),
        pl.col("country").alias("matched_country"),
    ])
    .collect()
)

# S3 records
s3_missed = (
    scan_tsv(files["train_source3"])
    .filter(
        pl.col("entity_id").is_in(matched_ids)
    )
    .select([
        pl.col("entity_id").alias("matched_entity_id"),
        pl.col("business_name").alias("matched_name"),
        pl.col("business_address").alias("matched_address"),
        pl.col("country").alias("matched_country"),
    ])
    .collect()
)

source_records = pl.concat([
    s2_missed,
    s3_missed,
])

missed_inspection = (
    missed_edges
    .join(
        s1_missed,
        on="source1_entity_id",
        how="left",
    )
    .join(
        source_records,
        on="matched_entity_id",
        how="left",
    )
)

print(
    missed_inspection
    .select([
        "source1_entity_id",
        "matched_entity_id",
        "matched_source",
        "s1_country",
        "matched_country",
        "s1_name",
        "matched_name",
        "s1_address",
        "matched_address",
    ])
    .head(50)
)

In [ ]:
# ============================================================
# AMLC 2026 — CELL 38
# Country consistency on true matches
# ============================================================

# Get all S1 countries for the sampled evaluation IDs.
s1_country_eval = (
    scan_tsv(files["train_source1"])
    .filter(
        pl.col("entity_id").is_in(eval_id_list)
    )
    .select([
        pl.col("entity_id").alias("source1_entity_id"),
        pl.col("country").alias("s1_country"),
    ])
    .collect()
)

# Get country of matched S2/S3 IDs.
s2_country = (
    scan_tsv(files["train_source2"])
    .filter(
        pl.col("entity_id").is_in(
            eval_gt
            .filter(pl.col("matched_source") == "S2")
            ["matched_entity_id"]
            .to_list()
        )
    )
    .select([
        pl.col("entity_id").alias("matched_entity_id"),
        pl.col("country").alias("matched_country"),
    ])
    .collect()
)

s3_country = (
    scan_tsv(files["train_source3"])
    .filter(
        pl.col("entity_id").is_in(
            eval_gt
            .filter(pl.col("matched_source") == "S3")
            ["matched_entity_id"]
            .to_list()
        )
    )
    .select([
        pl.col("entity_id").alias("matched_entity_id"),
        pl.col("country").alias("matched_country"),
    ])
    .collect()
)

country_eval = (
    eval_gt
    .join(
        s1_country_eval,
        on="source1_entity_id",
        how="left",
    )
    .join(
        pl.concat([s2_country, s3_country]),
        on="matched_entity_id",
        how="left",
    )
    .with_columns(
        (
            pl.col("s1_country")
            == pl.col("matched_country")
        ).alias("same_country")
    )
)

print(
    country_eval
    .group_by("matched_source")
    .agg([
        pl.len().alias("true_edges"),
        pl.col("same_country").sum().alias("same_country_edges"),
        pl.col("same_country").mean().alias("country_consistency"),
    ])
)

print("\nOverall:")
print(
    country_eval
    .select([
        pl.len().alias("true_edges"),
        pl.col("same_country").mean().alias("country_consistency"),
    ])
)

In [ ]:
# ============================================================
# AMLC 2026 — CELL 39
# PERSISTENT CHECKPOINT SETUP
# ============================================================

from pathlib import Path

DRIVE_PROJECT = Path("/content/drive/MyDrive/AMLC2026")

DRIVE_CACHE = DRIVE_PROJECT / "cache"
DRIVE_BLOCKING = DRIVE_CACHE / "blocking"
DRIVE_STATE = DRIVE_CACHE / "state"
DRIVE_ER_EDA = DRIVE_STATE / "er_eda"

for p in [
    DRIVE_CACHE,
    DRIVE_BLOCKING,
    DRIVE_STATE,
    DRIVE_ER_EDA,
]:
    p.mkdir(parents=True, exist_ok=True)

print("Persistent directories ready:")
print("PROJECT :", DRIVE_PROJECT)
print("CACHE   :", DRIVE_CACHE)
print("BLOCKING:", DRIVE_BLOCKING)
print("STATE   :", DRIVE_STATE)
print("ER EDA  :", DRIVE_ER_EDA)

In [ ]:
# ============================================================
# AMLC 2026 — CELL 40
# BACKUP NORMALIZED NAME INDEXES TO GOOGLE DRIVE
# ============================================================

import subprocess

LOCAL_BLOCKING = Path("/content/AMLC2026/cache/blocking")

print("Syncing blocking indexes to Google Drive...\n")

result = subprocess.run(
    [
        "rsync",
        "-ah",
        "--info=progress2",
        str(LOCAL_BLOCKING) + "/",
        str(DRIVE_BLOCKING) + "/",
    ],
    check=True,
)

print("\n✅ Blocking indexes backed up.")

for p in sorted(DRIVE_BLOCKING.glob("*")):
    print(
        f"{p.name:45s}"
        f"{p.stat().st_size / (1024**2):9.1f} MB"
    )

In [ ]:
# ============================================================
# AMLC 2026 — CELL 41
# SAVE CURRENT EDA / VALIDATION STATE
# LazyFrame-safe version
# ============================================================

from pathlib import Path
import polars as pl

DRIVE_PROJECT = Path("/content/drive/MyDrive/AMLC2026")

DRIVE_ER_EDA = (
    DRIVE_PROJECT
    / "cache"
    / "state"
    / "er_eda"
)

DRIVE_ER_EDA.mkdir(parents=True, exist_ok=True)


def save_polars(obj, path):
    """
    Save either a Polars DataFrame or LazyFrame.
    LazyFrames are collected at checkpoint time.
    """

    if isinstance(obj, pl.LazyFrame):
        print(f"Collecting LazyFrame for checkpoint: {path.name}")
        obj = obj.collect()

    elif not isinstance(obj, pl.DataFrame):
        raise TypeError(
            f"Unsupported object type for {path.name}: "
            f"{type(obj)}"
        )

    obj.write_parquet(
        path,
        compression="zstd"
    )

    print(
        f"✅ {path.name:32s}"
        f"{path.stat().st_size / (1024**2):9.2f} MB"
    )


artifacts = {
    "gt.parquet": gt,
    "gt_edges.parquet": gt_edges,
    "s1_eval.parquet": s1_eval,
    "eval_gt.parquet": eval_gt,
    "edge_eval.parquet": edge_eval,
    "per_s1_recall.parquet": per_s1_recall,
    "candidate_counts.parquet": candidate_counts,
    "country_eval.parquet": country_eval,
    "missed_edges.parquet": missed_edges,
    "missed_inspection.parquet": missed_inspection,
}


print("=" * 80)
print("SAVING AMLC 2026 EDA CHECKPOINT")
print("=" * 80)

for filename, obj in artifacts.items():

    save_polars(
        obj,
        DRIVE_ER_EDA / filename
    )

print("\n✅ ALL EDA ARTIFACTS SAVED.")

In [ ]:
# ============================================================
# AMLC 2026 — CELL 42
# SAVE EXPERIMENT SUMMARY / FINDINGS
# ============================================================

import json

summary = {
    "checkpoint": "ER_EDA_001",
    "timestamp_note": "AMLC 2026 live run",

    "dataset": {
        "train_source1": 2206821,
        "train_source2": 5034616,
        "train_source3": 5285603,
        "train_ground_truth": 2206821,
        "test_source1": 1732544,
        "test_source2": 4887273,
        "test_source3": 5082316,
    },

    "ground_truth": {
        "positive_edges": 7638365,
        "singleton_count": 123247,
        "singleton_fraction": 0.055848,
        "mean_matches_per_s1": 7638365 / 2206821,
        "s2_positive_edges": 3693619,
        "s3_positive_edges": 3944746,
        "max_matches_per_s1": 11,
    },

    "match_cardinality": {
        "0": 123247,
        "1": 119157,
        "2": 375212,
        "3": 530841,
        "4": 484115,
        "5": 321957,
        "6": 164868,
        "7": 63968,
        "8": 18680,
        "9": 4205,
        "10": 534,
        "11": 37,
    },

    "composition": {
        "both_s2_s3": 1776047,
        "s2_only": 143029,
        "s3_only": 164498,
        "none": 123247,
    },

    "training_exclusivity_observed": {
        "positive_s2_s3_records_with_multiple_s1_assignments": 0,
        "maximum_s1_count_per_positive_s2_s3_record": 1,
    },

    "name_blocking_100k_eval": {
        "eval_s1": 100000,
        "positive_edges": 345980,
        "s2_edge_recall": 0.426468,
        "s3_edge_recall": 0.410389,
        "overall_edge_recall": 0.418163,
        "full_s1_recovery": 0.095907,
    },

    "country_consistency_100k_eval": {
        "overall": 1.0,
        "s2": 1.0,
        "s3": 1.0,
        "note": "Empirical training invariant; not stated as an explicit official rule."
    },

    "engineering": {
        "s2_normalized_index": "train_s2_name_norm.parquet",
        "s3_normalized_index": "train_s3_name_norm.parquet",
        "index_type": "Parquet / ZSTD",
    },

    "next_stage": [
        "Exact normalized-name candidate volume analysis",
        "Missed-positive error taxonomy",
        "Country hard-block validation on broader sample",
        "Character n-gram retrieval",
        "Token / rare-token blocking",
        "Address / numeric blocking",
        "Cross-script transliteration",
        "Semantic ANN only after lexical blockers are measured",
    ]
}

summary_path = DRIVE_ER_EDA / "checkpoint_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("✅ Summary saved:")
print(summary_path)

In [ ]:
# ============================================================
# AMLC 2026 — CELL 43
# SAVE ENVIRONMENT MANIFEST
# ============================================================

import sys
import subprocess
import json
from pathlib import Path

manifest = {
    "python": sys.version,
    "packages": {}
}

packages_to_record = [
    "polars",
    "pandas",
    "numpy",
    "duckdb",
    "rapidfuzz",
    "scikit-learn",
    "lightgbm",
    "xgboost",
    "faiss-cpu",
    "anyascii",
]

for package in packages_to_record:
    try:
        output = subprocess.check_output(
            [sys.executable, "-m", "pip", "show", package],
            text=True
        )

        version = None

        for line in output.splitlines():
            if line.startswith("Version:"):
                version = line.split(":", 1)[1].strip()
                break

        manifest["packages"][package] = version

    except Exception:
        manifest["packages"][package] = None

manifest_path = DRIVE_PROJECT / "cache" / "environment_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print("\n✅ Environment manifest saved.")

In [ ]:
# ============================================================
# AMLC 2026 — CELL 44
# 🚨 AMLC 2026 WAR-ROOM RESUME CELL
# Robust checkpoint restore
# ============================================================

from pathlib import Path
import json
import polars as pl

DRIVE_PROJECT = Path("/content/drive/MyDrive/AMLC2026")

DRIVE_BLOCKING = (
    DRIVE_PROJECT
    / "cache"
    / "blocking"
)

DRIVE_ER_EDA = (
    DRIVE_PROJECT
    / "cache"
    / "state"
    / "er_eda"
)

TRAIN_DIR = DRIVE_PROJECT / "dataset" / "train"
TEST_DIR = DRIVE_PROJECT / "dataset" / "test"

files = {
    "train_source1": TRAIN_DIR / "train_source1.tsv",
    "train_source2": TRAIN_DIR / "train_source2.tsv",
    "train_source3": TRAIN_DIR / "train_source3.tsv",
    "train_ground_truth": TRAIN_DIR / "train_ground_truth.tsv",
    "test_source1": TEST_DIR / "test_source1.tsv",
    "test_source2": TEST_DIR / "test_source2.tsv",
    "test_source3": TEST_DIR / "test_source3.tsv",
}


# ------------------------------------------------------------
# Loader
# ------------------------------------------------------------

def clean_column_name(name: str) -> str:
    return str(name).replace("\ufeff", "").strip()


def scan_tsv(path: Path) -> pl.LazyFrame:

    lf = pl.scan_csv(
        path,
        separator="\t",
        infer_schema_length=1000,
        null_values=["", "null", "NULL", "None"],
    )

    schema = lf.collect_schema()

    rename_map = {
        col: clean_column_name(col)
        for col in schema.names()
        if clean_column_name(col) != col
    }

    if rename_map:
        lf = lf.rename(rename_map)

    return lf


# ------------------------------------------------------------
# Normalize indexes
# ------------------------------------------------------------

S2_NAME_INDEX = DRIVE_BLOCKING / "train_s2_name_norm.parquet"
S3_NAME_INDEX = DRIVE_BLOCKING / "train_s3_name_norm.parquet"

assert S2_NAME_INDEX.exists(), f"Missing: {S2_NAME_INDEX}"
assert S3_NAME_INDEX.exists(), f"Missing: {S3_NAME_INDEX}"

s2_idx = pl.scan_parquet(S2_NAME_INDEX)
s3_idx = pl.scan_parquet(S3_NAME_INDEX)


# ------------------------------------------------------------
# Restore saved artifacts
# ------------------------------------------------------------

artifact_names = [
    "gt",
    "gt_edges",
    "s1_eval",
    "eval_gt",
    "edge_eval",
    "per_s1_recall",
    "candidate_counts",
    "country_eval",
    "missed_edges",
    "missed_inspection",
]

restored = []
missing = []

for name in artifact_names:

    path = DRIVE_ER_EDA / f"{name}.parquet"

    if path.exists():

        globals()[name] = pl.read_parquet(path)
        restored.append(name)

    else:

        missing.append(name)


# ------------------------------------------------------------
# Restore summary
# ------------------------------------------------------------

summary_path = DRIVE_ER_EDA / "checkpoint_summary.json"

if summary_path.exists():

    with open(summary_path, "r", encoding="utf-8") as f:
        checkpoint_summary = json.load(f)

else:

    checkpoint_summary = None


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 80)
print("AMLC 2026 WAR-ROOM CHECKPOINT")
print("=" * 80)

print("\n✅ Restored artifacts:")

for name in restored:
    obj = globals()[name]
    print(f"   {name:25s} {obj.shape}")

if missing:

    print("\n⚠️ Missing optional artifacts:")

    for name in missing:
        print(f"   {name}")

else:

    print("\n✅ No checkpoint artifacts missing.")

print("\nIndexes:")
print("   S2:", S2_NAME_INDEX)
print("   S3:", S3_NAME_INDEX)

if checkpoint_summary:

    print("\nKey findings:")

    print(
        "   Name edge recall:",
        checkpoint_summary[
            "name_blocking_100k_eval"
        ]["overall_edge_recall"]
    )

    print(
        "   Full S1 recall:",
        checkpoint_summary[
            "name_blocking_100k_eval"
        ]["full_s1_recovery"]
    )

    print(
        "   Country consistency:",
        checkpoint_summary[
            "country_consistency_100k_eval"
        ]["overall"]
    )

print("\n✅ RESUME STATE READY.")

In [ ]:
# ============================================================
# AMLC 2026 — CELL 45
# FINAL CHECKPOINT INTEGRITY TEST
# ============================================================

required_files = [
    DRIVE_BLOCKING / "train_s2_name_norm.parquet",
    DRIVE_BLOCKING / "train_s3_name_norm.parquet",

    DRIVE_ER_EDA / "gt.parquet",
    DRIVE_ER_EDA / "gt_edges.parquet",
    DRIVE_ER_EDA / "s1_eval.parquet",
    DRIVE_ER_EDA / "eval_gt.parquet",
    DRIVE_ER_EDA / "edge_eval.parquet",
    DRIVE_ER_EDA / "per_s1_recall.parquet",
    DRIVE_ER_EDA / "candidate_counts.parquet",
    DRIVE_ER_EDA / "country_eval.parquet",
    DRIVE_ER_EDA / "missed_edges.parquet",
    DRIVE_ER_EDA / "missed_inspection.parquet",
    DRIVE_ER_EDA / "checkpoint_summary.json",

    DRIVE_PROJECT / "cache" / "environment_manifest.json",
]

print("=" * 80)
print("CHECKPOINT INTEGRITY")
print("=" * 80)

all_ok = True

for path in required_files:

    exists = path.exists()

    print(
        ("✅" if exists else "❌"),
        path
    )

    if not exists:
        all_ok = False

print("\n" + "=" * 80)

if all_ok:
    print("✅ ALL CHECKPOINT ARTIFACTS ARE SAFE ON GOOGLE DRIVE.")
    print("✅ SAFE TO TAKE A BREAK / DISCONNECT.")
else:
    print("❌ CHECKPOINT INCOMPLETE — DO NOT DISCONNECT YET.")

print("=" * 80)

In [54]:
# ============================================================
# AMLC 2026 — CELL 46
# WAR-ROOM RESUME / NEXT-STAGE STATE
#
# Purpose:
#   Reconstruct only the small state needed for the next stage.
#   Do NOT reload 12.5M source rows into RAM.
# ============================================================

from pathlib import Path
import json
import re
import unicodedata
import gc

import numpy as np
import polars as pl
from rapidfuzz import fuzz

# ------------------------------------------------------------
# Persistent paths
# ------------------------------------------------------------

DRIVE_PROJECT = Path("/content/drive/MyDrive/AMLC2026")

DRIVE_BLOCKING = (
    DRIVE_PROJECT / "cache" / "blocking"
)

DRIVE_ER_EDA = (
    DRIVE_PROJECT / "cache" / "state" / "er_eda"
)

TRAIN_DIR = DRIVE_PROJECT / "dataset" / "train"
TEST_DIR = DRIVE_PROJECT / "dataset" / "test"

files = {
    "train_source1": TRAIN_DIR / "train_source1.tsv",
    "train_source2": TRAIN_DIR / "train_source2.tsv",
    "train_source3": TRAIN_DIR / "train_source3.tsv",
    "train_ground_truth": TRAIN_DIR / "train_ground_truth.tsv",
    "test_source1": TEST_DIR / "test_source1.tsv",
    "test_source2": TEST_DIR / "test_source2.tsv",
    "test_source3": TEST_DIR / "test_source3.tsv",
}

# ------------------------------------------------------------
# Loader
# ------------------------------------------------------------

def clean_column_name(name: str) -> str:
    return str(name).replace("\ufeff", "").strip()


def scan_tsv(path: Path) -> pl.LazyFrame:
    lf = pl.scan_csv(
        path,
        separator="\t",
        infer_schema_length=1000,
        null_values=["", "null", "NULL", "None"],
    )

    schema = lf.collect_schema()

    rename_map = {
        col: clean_column_name(col)
        for col in schema.names()
        if clean_column_name(col) != col
    }

    if rename_map:
        lf = lf.rename(rename_map)

    return lf


# ------------------------------------------------------------
# Restore checkpoint artifacts
# ------------------------------------------------------------

artifact_names = [
    "gt",
    "gt_edges",
    "s1_eval",
    "eval_gt",
    "edge_eval",
    "per_s1_recall",
    "candidate_counts",
    "country_eval",
    "missed_edges",
    "missed_inspection",
]

for name in artifact_names:

    path = DRIVE_ER_EDA / f"{name}.parquet"

    if path.exists():
        globals()[name] = pl.read_parquet(path)

# ------------------------------------------------------------
# Restore normalized indexes
# ------------------------------------------------------------

S2_NAME_INDEX = DRIVE_BLOCKING / "train_s2_name_norm.parquet"
S3_NAME_INDEX = DRIVE_BLOCKING / "train_s3_name_norm.parquet"

assert S2_NAME_INDEX.exists()
assert S3_NAME_INDEX.exists()

s2_idx = pl.scan_parquet(S2_NAME_INDEX)
s3_idx = pl.scan_parquet(S3_NAME_INDEX)

# ------------------------------------------------------------
# Re-create the exact normalizer used by the notebook
# ------------------------------------------------------------

LEGAL_SUFFIX_PATTERN = (
    r"(?i)\b("
    r"private\s+limited|"
    r"private|"
    r"pvt\s+ltd|"
    r"pvt|"
    r"limited|"
    r"ltd|"
    r"corporation|"
    r"corp|"
    r"company|"
    r"co|"
    r"incorporated|"
    r"inc|"
    r"llc|"
    r"llp|"
    r"sarl|"
    r"sasu|"
    r"sas|"
    r"plc|"
    r"gmbh"
    r")\b"
)


def fast_name_expr(column="business_name"):

    return (
        pl.col(column)
        .fill_null("")
        .str.to_lowercase()
        .str.replace_all("&", " and ")
        .str.replace_all(
            r"[^\p{L}\p{N}\s]",
            " "
        )
        .str.replace_all(
            LEGAL_SUFFIX_PATTERN,
            " "
        )
        .str.replace_all(
            r"\s+",
            " "
        )
        .str.strip_chars()
    )


# ------------------------------------------------------------
# IMPORTANT FIX:
# Build the variable that Cell 33/35 expected.
# ------------------------------------------------------------

if "s1_norm_lookup" not in globals():

    s1_norm_lookup = (
        s1_eval
        .select([
            "entity_id",
            "business_name",
            "country",
        ])
        .rename({
            "entity_id": "source1_entity_id"
        })
        .with_columns(
            fast_name_expr().alias("name_norm")
        )
    )

# ------------------------------------------------------------
# Evaluation universe
#
# IMPORTANT:
# This remains the full 100k S1 evaluation population,
# INCLUDING the singleton S1s.
# ------------------------------------------------------------

EVAL_S1_IDS = s1_eval["entity_id"].to_list()

assert len(EVAL_S1_IDS) == 100_000

print("=" * 90)
print("AMLC 2026 — NEXT-STAGE STATE READY")
print("=" * 90)

print("Evaluation S1:", len(EVAL_S1_IDS))
print("Positive edges:", eval_gt.height)
print("S2 name index:", S2_NAME_INDEX)
print("S3 name index:", S3_NAME_INDEX)

print("\nRestored:")
for name in artifact_names:
    obj = globals().get(name)
    if obj is not None:
        print(f"  ✅ {name:22s} {obj.shape}")

print("\n✅ Ready for blocker-oracle analysis.")

AMLC 2026 — NEXT-STAGE STATE READY
Evaluation S1: 100000
Positive edges: 345980
S2 name index: /content/drive/MyDrive/AMLC2026/cache/blocking/train_s2_name_norm.parquet
S3 name index: /content/drive/MyDrive/AMLC2026/cache/blocking/train_s3_name_norm.parquet

Restored:
  ✅ gt                     (2206821, 2)
  ✅ gt_edges               (7638365, 3)
  ✅ s1_eval                (100000, 4)
  ✅ eval_gt                (345980, 3)
  ✅ edge_eval              (345980, 6)
  ✅ per_s1_recall          (94404, 4)
  ✅ candidate_counts       (100000, 5)
  ✅ country_eval           (345980, 6)
  ✅ missed_edges           (1000, 6)
  ✅ missed_inspection      (1000, 12)

✅ Ready for blocker-oracle analysis.


In [55]:
# ============================================================
# AMLC 2026 — CELL 47
# FULL POSITIVE-EDGE ORACLE
#
# Materializes only the 345,980 known positive evaluation edges.
# This is NOT candidate generation.
# It is used to measure blocker ceilings.
# ============================================================

print("=" * 90)
print("BUILDING POSITIVE-EDGE ORACLE")
print("=" * 90)

# ------------------------------------------------------------
# S1 records
# ------------------------------------------------------------

s1_truth = (
    s1_eval
    .select([
        pl.col("entity_id").alias("source1_entity_id"),
        pl.col("business_name").alias("s1_name"),
        pl.col("business_address").alias("s1_address"),
        pl.col("country").alias("s1_country"),
    ])
)

# ------------------------------------------------------------
# Positive S2 IDs
# ------------------------------------------------------------

s2_ids = (
    eval_gt
    .filter(pl.col("matched_source") == "S2")
    ["matched_entity_id"]
    .to_list()
)

s3_ids = (
    eval_gt
    .filter(pl.col("matched_source") == "S3")
    ["matched_entity_id"]
    .to_list()
)

print("S2 positive records:", len(s2_ids))
print("S3 positive records:", len(s3_ids))

# ------------------------------------------------------------
# Pull S2 truth records
# ------------------------------------------------------------

s2_truth = (
    scan_tsv(files["train_source2"])
    .filter(
        pl.col("entity_id").is_in(s2_ids)
    )
    .select([
        pl.col("entity_id").alias("matched_entity_id"),
        pl.col("business_name").alias("matched_name"),
        pl.col("business_address").alias("matched_address"),
        pl.col("country").alias("matched_country"),
    ])
    .collect()
)

# ------------------------------------------------------------
# Pull S3 truth records
# ------------------------------------------------------------

s3_truth = (
    scan_tsv(files["train_source3"])
    .filter(
        pl.col("entity_id").is_in(s3_ids)
    )
    .select([
        pl.col("entity_id").alias("matched_entity_id"),
        pl.col("business_name").alias("matched_name"),
        pl.col("business_address").alias("matched_address"),
        pl.col("country").alias("matched_country"),
    ])
    .collect()
)

truth_records = pl.concat([
    s2_truth,
    s3_truth,
])

# ------------------------------------------------------------
# Join everything
# ------------------------------------------------------------

positive_pairs = (
    eval_gt
    .join(
        s1_truth,
        on="source1_entity_id",
        how="left",
    )
    .join(
        truth_records,
        on="matched_entity_id",
        how="left",
    )
)

print("\nShape:", positive_pairs.shape)

print(
    positive_pairs
    .select([
        "source1_entity_id",
        "matched_entity_id",
        "matched_source",
        "s1_country",
        "matched_country",
        "s1_name",
        "matched_name",
        "s1_address",
        "matched_address",
    ])
    .head(10)
)

assert positive_pairs.height == 345_980

print("\n✅ Positive-edge oracle ready.")

BUILDING POSITIVE-EDGE ORACLE
S2 positive records: 167274
S3 positive records: 178706

Shape: (345980, 9)
shape: (10, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ source1_e ┆ matched_e ┆ matched_s ┆ s1_countr ┆ … ┆ s1_name   ┆ matched_n ┆ s1_addres ┆ matched_ │
│ ntity_id  ┆ ntity_id  ┆ ource     ┆ y         ┆   ┆ ---       ┆ ame       ┆ s         ┆ address  │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ str       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ str       ┆ str       ┆ str       ┆   ┆           ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ S1-869891 ┆ S3-274817 ┆ S3        ┆ India     ┆ … ┆ Laxmi     ┆ Laxmi Gbn ┆ New       ┆ New      │
│ 37        ┆ 120       ┆           ┆           ┆   ┆ Golden    ┆ lnvestmen ┆ Bridge    ┆ Bridge   │
│           ┆           ┆           ┆           ┆   ┆ Investmen ┆ ts   

In [56]:
# ============================================================
# AMLC 2026 — CELL 48
# EXACT VARIANT ORACLE
#
# Measures positive-edge recall ceilings for:
#   - native normalized name
#   - transliterated name
#   - normalized address
#   - transliterated address
#   - numeric address signature
#
# IMPORTANT:
# These are recall ceilings only.
# They tell us which blockers are worth indexing.
# ============================================================

from anyascii import anyascii

# ------------------------------------------------------------
# Python normalizers
# ------------------------------------------------------------

PY_SUFFIXES = {
    "private",
    "private limited",
    "pvt",
    "pvt ltd",
    "limited",
    "ltd",
    "corporation",
    "corp",
    "company",
    "co",
    "incorporated",
    "inc",
    "llc",
    "llp",
    "sarl",
    "sasu",
    "sas",
    "plc",
    "gmbh",
}


def normalize_ascii_name(text):

    if text is None:
        return ""

    x = anyascii(str(text))
    x = unicodedata.normalize("NFKC", x).casefold()

    x = x.replace("&", " and ")

    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    tokens = [
        t for t in x.split()
        if t not in PY_SUFFIXES
    ]

    return " ".join(tokens)


# Common address standardizations.
# Conservative enough for a blocker experiment.

ADDRESS_ABBR = [
    (r"\bstreet\b", " st "),
    (r"\broad\b", " rd "),
    (r"\bavenue\b", " ave "),
    (r"\bboulevard\b", " blvd "),
    (r"\bdrive\b", " dr "),
    (r"\blane\b", " ln "),
    (r"\bcourt\b", " ct "),
    (r"\bhighway\b", " hwy "),
    (r"\bparkway\b", " pkwy "),
    (r"\bapartment\b", " apt "),
    (r"\bsuite\b", " ste "),
    (r"\bbuilding\b", " bldg "),
]


def normalize_address_native(text):

    if text is None:
        return ""

    x = unicodedata.normalize(
        "NFKC",
        str(text)
    ).casefold()

    x = x.replace("&", " and ")

    x = re.sub(
        r"[^\w\s]",
        " ",
        x,
        flags=re.UNICODE,
    )

    for pattern, replacement in ADDRESS_ABBR:
        x = re.sub(
            pattern,
            replacement,
            x,
        )

    x = re.sub(r"\s+", " ", x).strip()

    return x


def normalize_address_ascii(text):

    if text is None:
        return ""

    x = anyascii(str(text))

    x = unicodedata.normalize(
        "NFKC",
        x
    ).casefold()

    x = x.replace("&", " and ")

    x = re.sub(
        r"[^a-z0-9\s]",
        " ",
        x,
    )

    for pattern, replacement in ADDRESS_ABBR:
        x = re.sub(
            pattern,
            replacement,
            x,
        )

    x = re.sub(r"\s+", " ", x).strip()

    return x


def numeric_signature(text):

    if text is None:
        return ""

    tokens = re.findall(
        r"\d+[a-zA-Z]?",
        str(text)
    )

    if not tokens:
        return ""

    return "|".join(
        sorted(set(tokens))
    )


# ------------------------------------------------------------
# Native name normalization using the notebook's exact logic
# ------------------------------------------------------------

positive_pairs = positive_pairs.with_columns([
    fast_name_expr("s1_name").alias("s1_name_native"),
    fast_name_expr("matched_name").alias("matched_name_native"),
])

# ------------------------------------------------------------
# Transliteration
# ------------------------------------------------------------

print("Computing transliterated name/address views...")

positive_pairs = positive_pairs.with_columns([
    pl.col("s1_name")
      .map_elements(
          normalize_ascii_name,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("s1_name_ascii"),

    pl.col("matched_name")
      .map_elements(
          normalize_ascii_name,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("matched_name_ascii"),

    pl.col("s1_address")
      .map_elements(
          normalize_address_native,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("s1_addr_native"),

    pl.col("matched_address")
      .map_elements(
          normalize_address_native,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("matched_addr_native"),

    pl.col("s1_address")
      .map_elements(
          normalize_address_ascii,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("s1_addr_ascii"),

    pl.col("matched_address")
      .map_elements(
          normalize_address_ascii,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("matched_addr_ascii"),

    pl.col("s1_address")
      .map_elements(
          numeric_signature,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("s1_addr_nums"),

    pl.col("matched_address")
      .map_elements(
          numeric_signature,
          return_dtype=pl.String,
          skip_nulls=False,
      )
      .alias("matched_addr_nums"),
])

# ------------------------------------------------------------
# Exact conditions
# ------------------------------------------------------------

positive_pairs = positive_pairs.with_columns([

    (
        pl.col("s1_name_native")
        == pl.col("matched_name_native")
    ).alias("name_native_exact"),

    (
        (pl.col("s1_name_ascii") != "")
        & (
            pl.col("s1_name_ascii")
            == pl.col("matched_name_ascii")
        )
    ).alias("name_ascii_exact"),

    (
        (pl.col("s1_addr_native") != "")
        & (
            pl.col("s1_addr_native")
            == pl.col("matched_addr_native")
        )
    ).alias("address_native_exact"),

    (
        (pl.col("s1_addr_ascii") != "")
        & (
            pl.col("s1_addr_ascii")
            == pl.col("matched_addr_ascii")
        )
    ).alias("address_ascii_exact"),

    (
        (pl.col("s1_addr_nums") != "")
        & (
            pl.col("s1_addr_nums")
            == pl.col("matched_addr_nums")
        )
    ).alias("numeric_exact"),
])

# ------------------------------------------------------------
# Recall helper
# ------------------------------------------------------------

def oracle_summary(df, condition_cols):

    rows = []

    total = df.height

    for col in condition_cols:

        rows.append({
            "signal": col,
            "recovered_edges": int(
                df[col].sum()
            ),
            "edge_recall": float(
                df[col].mean()
            ),
        })

    return pl.DataFrame(rows).sort(
        "edge_recall",
        descending=True,
    )


signals = [
    "name_native_exact",
    "name_ascii_exact",
    "address_native_exact",
    "address_ascii_exact",
    "numeric_exact",
]

print("=" * 90)
print("POSITIVE-EDGE EXACT ORACLE")
print("=" * 90)

print(
    oracle_summary(
        positive_pairs,
        signals,
    )
)

print("\nBy source:")

for source in ["S2", "S3"]:

    print(f"\n--- {source} ---")

    print(
        oracle_summary(
            positive_pairs.filter(
                pl.col("matched_source") == source
            ),
            signals,
        )
    )

# ------------------------------------------------------------
# Cumulative unions
# ------------------------------------------------------------

positive_pairs = positive_pairs.with_columns([

    (
        pl.col("name_native_exact")
        | pl.col("name_ascii_exact")
    ).alias("name_union_exact"),

    (
        pl.col("name_native_exact")
        | pl.col("name_ascii_exact")
        | pl.col("address_native_exact")
        | pl.col("address_ascii_exact")
    ).alias("name_address_union_exact"),

    (
        pl.col("name_native_exact")
        | pl.col("name_ascii_exact")
        | pl.col("address_native_exact")
        | pl.col("address_ascii_exact")
        | pl.col("numeric_exact")
    ).alias("exact_union"),

])

print("\nCUMULATIVE")

print(
    oracle_summary(
        positive_pairs,
        [
            "name_union_exact",
            "name_address_union_exact",
            "exact_union",
        ],
    )
)

Computing transliterated name/address views...
POSITIVE-EDGE EXACT ORACLE
shape: (5, 3)
┌──────────────────────┬─────────────────┬─────────────┐
│ signal               ┆ recovered_edges ┆ edge_recall │
│ ---                  ┆ ---             ┆ ---         │
│ str                  ┆ i64             ┆ f64         │
╞══════════════════════╪═════════════════╪═════════════╡
│ numeric_exact        ┆ 200277          ┆ 0.578869    │
│ name_ascii_exact     ┆ 161460          ┆ 0.466674    │
│ name_native_exact    ┆ 144681          ┆ 0.418177    │
│ address_ascii_exact  ┆ 39930           ┆ 0.115411    │
│ address_native_exact ┆ 39717           ┆ 0.114796    │
└──────────────────────┴─────────────────┴─────────────┘

By source:

--- S2 ---
shape: (5, 3)
┌──────────────────────┬─────────────────┬─────────────┐
│ signal               ┆ recovered_edges ┆ edge_recall │
│ ---                  ┆ ---             ┆ ---         │
│ str                  ┆ i64             ┆ f64         │
╞══════════════════

In [42]:
# ============================================================
# AMLC 2026 — CELL 49
# FUZZY MISSED-POSITIVE TAXONOMY
#
# Only a sample is scored here.
# We do NOT run expensive RapidFuzz over the entire 346k edges.
# ============================================================

FUZZY_SAMPLE_N = 10_000

fuzzy_sample = (
    positive_pairs
    .filter(
        ~pl.col("name_union_exact")
    )
    .sample(
        n=min(
            FUZZY_SAMPLE_N,
            positive_pairs.filter(
                ~pl.col("name_union_exact")
            ).height,
        ),
        seed=2026,
    )
)

def safe_ratio(a, b):
    return fuzz.ratio(
        str(a),
        str(b),
    )

def safe_token_set(a, b):
    return fuzz.token_set_ratio(
        str(a),
        str(b),
    )


rows = []

for row in fuzzy_sample.iter_rows(named=True):

    name_native_ratio = safe_ratio(
        row["s1_name_native"],
        row["matched_name_native"],
    )

    name_ascii_ratio = safe_ratio(
        row["s1_name_ascii"],
        row["matched_name_ascii"],
    )

    name_token_ratio = safe_token_set(
        row["s1_name_ascii"],
        row["matched_name_ascii"],
    )

    addr_ascii_ratio = safe_ratio(
        row["s1_addr_ascii"],
        row["matched_addr_ascii"],
    )

    rows.append({
        "source1_entity_id":
            row["source1_entity_id"],

        "matched_entity_id":
            row["matched_entity_id"],

        "matched_source":
            row["matched_source"],

        "name_native_ratio":
            name_native_ratio,

        "name_ascii_ratio":
            name_ascii_ratio,

        "name_token_ratio":
            name_token_ratio,

        "address_ascii_ratio":
            addr_ascii_ratio,

        "name_ascii_exact":
            row["name_ascii_exact"],

        "address_ascii_exact":
            row["address_ascii_exact"],

        "numeric_exact":
            row["numeric_exact"],

        "s1_name":
            row["s1_name"],

        "matched_name":
            row["matched_name"],

        "s1_address":
            row["s1_address"],

        "matched_address":
            row["matched_address"],
    })

fuzzy_df = pl.DataFrame(rows)

print("=" * 90)
print("FUZZY MISSED-POSITIVE DISTRIBUTIONS")
print("=" * 90)

for col in [
    "name_native_ratio",
    "name_ascii_ratio",
    "name_token_ratio",
    "address_ascii_ratio",
]:

    print(f"\n{col}")

    print(
        fuzzy_df
        .select([
            pl.col(col).quantile(0.05).alias("p05"),
            pl.col(col).quantile(0.25).alias("p25"),
            pl.col(col).median().alias("p50"),
            pl.col(col).quantile(0.75).alias("p75"),
            pl.col(col).quantile(0.90).alias("p90"),
            pl.col(col).quantile(0.95).alias("p95"),
            pl.col(col).max().alias("max"),
        ])
    )

print("\nExamples — strongest transliterated name similarity among misses:")

print(
    fuzzy_df
    .sort("name_ascii_ratio", descending=True)
    .select([
        "matched_source",
        "name_ascii_ratio",
        "name_token_ratio",
        "s1_name",
        "matched_name",
        "s1_address",
        "matched_address",
    ])
    .head(30)
)

FUZZY MISSED-POSITIVE DISTRIBUTIONS

name_native_ratio
shape: (1, 7)
┌──────────┬───────────┬───────────┬──────┬───────────┬──────┬───────────┐
│ p05      ┆ p25       ┆ p50       ┆ p75  ┆ p90       ┆ p95  ┆ max       │
│ ---      ┆ ---       ┆ ---       ┆ ---  ┆ ---       ┆ ---  ┆ ---       │
│ f64      ┆ f64       ┆ f64       ┆ f64  ┆ f64       ┆ f64  ┆ f64       │
╞══════════╪═══════════╪═══════════╪══════╪═══════════╪══════╪═══════════╡
│ 4.651163 ┆ 60.606061 ┆ 77.777778 ┆ 87.5 ┆ 93.023256 ┆ 95.0 ┆ 98.701299 │
└──────────┴───────────┴───────────┴──────┴───────────┴──────┴───────────┘

name_ascii_ratio
shape: (1, 7)
┌──────┬──────┬───────────┬──────┬───────────┬───────────┬───────────┐
│ p05  ┆ p25  ┆ p50       ┆ p75  ┆ p90       ┆ p95       ┆ max       │
│ ---  ┆ ---  ┆ ---       ┆ ---  ┆ ---       ┆ ---       ┆ ---       │
│ f64  ┆ f64  ┆ f64       ┆ f64  ┆ f64       ┆ f64       ┆ f64       │
╞══════╪══════╪═══════════╪══════╪═══════════╪═══════════╪═══════════╡
│ 40.0 ┆ 65.0 ┆ 78.

In [43]:
# ============================================================
# AMLC 2026 — CELL 50
# BLOCK-KEY ORACLE
#
# Tests cheap retrieval keys WITHOUT building their indexes yet.
#
# Keys:
#   - native prefix/suffix
#   - transliterated prefix/suffix
#   - first / last token
#   - exact normalized address
#   - numeric signature
#
# We measure cumulative positive recall.
# ============================================================

# ------------------------------------------------------------
# Key expressions
# ------------------------------------------------------------

for prefix in [
    "s1_name_native",
    "matched_name_native",
    "s1_name_ascii",
    "matched_name_ascii",
]:

    positive_pairs = positive_pairs.with_columns([

        pl.col(prefix)
          .str.slice(0, 4)
          .alias(f"{prefix}_p4"),

        pl.col(prefix)
          .str.slice(-4)
          .alias(f"{prefix}_s4"),

        pl.col(prefix)
          .str.split(" ")
          .list.first()
          .fill_null("")
          .alias(f"{prefix}_first"),

        pl.col(prefix)
          .str.split(" ")
          .list.last()
          .fill_null("")
          .alias(f"{prefix}_last"),
    ])

# ------------------------------------------------------------
# Key matches
# ------------------------------------------------------------

positive_pairs = positive_pairs.with_columns([

    (
        (pl.col("s1_name_native_p4") != "")
        & (
            pl.col("s1_name_native_p4")
            == pl.col("matched_name_native_p4")
        )
    ).alias("native_prefix4"),

    (
        (pl.col("s1_name_native_s4") != "")
        & (
            pl.col("s1_name_native_s4")
            == pl.col("matched_name_native_s4")
        )
    ).alias("native_suffix4"),

    (
        (pl.col("s1_name_ascii_p4") != "")
        & (
            pl.col("s1_name_ascii_p4")
            == pl.col("matched_name_ascii_p4")
        )
    ).alias("ascii_prefix4"),

    (
        (pl.col("s1_name_ascii_s4") != "")
        & (
            pl.col("s1_name_ascii_s4")
            == pl.col("matched_name_ascii_s4")
        )
    ).alias("ascii_suffix4"),

    (
        (pl.col("s1_name_ascii_first") != "")
        & (
            pl.col("s1_name_ascii_first")
            == pl.col("matched_name_ascii_first")
        )
    ).alias("ascii_first_token"),

    (
        (pl.col("s1_name_ascii_last") != "")
        & (
            pl.col("s1_name_ascii_last")
            == pl.col("matched_name_ascii_last")
        )
    ).alias("ascii_last_token"),

])

# ------------------------------------------------------------
# Combined blocker unions
# ------------------------------------------------------------

positive_pairs = positive_pairs.with_columns([

    (
        pl.col("name_union_exact")
        | pl.col("native_prefix4")
        | pl.col("native_suffix4")
        | pl.col("ascii_prefix4")
        | pl.col("ascii_suffix4")
    ).alias("name_key_union"),

    (
        pl.col("name_union_exact")
        | pl.col("native_prefix4")
        | pl.col("native_suffix4")
        | pl.col("ascii_prefix4")
        | pl.col("ascii_suffix4")
        | pl.col("ascii_first_token")
        | pl.col("ascii_last_token")
    ).alias("name_plus_token_union"),

    (
        pl.col("name_union_exact")
        | pl.col("native_prefix4")
        | pl.col("native_suffix4")
        | pl.col("ascii_prefix4")
        | pl.col("ascii_suffix4")
        | pl.col("ascii_first_token")
        | pl.col("ascii_last_token")
        | pl.col("address_native_exact")
        | pl.col("address_ascii_exact")
        | pl.col("numeric_exact")
    ).alias("cheap_union"),

])

print("=" * 90)
print("BLOCK-KEY POSITIVE-EDGE ORACLE")
print("=" * 90)

print(
    oracle_summary(
        positive_pairs,
        [
            "name_union_exact",
            "name_key_union",
            "name_plus_token_union",
            "cheap_union",
        ],
    )
)

print("\nBy source:")

for source in ["S2", "S3"]:

    print(f"\n--- {source} ---")

    print(
        oracle_summary(
            positive_pairs.filter(
                pl.col("matched_source") == source
            ),
            [
                "name_union_exact",
                "name_key_union",
                "name_plus_token_union",
                "cheap_union",
            ],
        )
    )

print("\n✅ This is the decision gate for the next blocker implementation.")

BLOCK-KEY POSITIVE-EDGE ORACLE
shape: (4, 3)
┌───────────────────────┬─────────────────┬─────────────┐
│ signal                ┆ recovered_edges ┆ edge_recall │
│ ---                   ┆ ---             ┆ ---         │
│ str                   ┆ i64             ┆ f64         │
╞═══════════════════════╪═════════════════╪═════════════╡
│ cheap_union           ┆ 331342          ┆ 0.957691    │
│ name_plus_token_union ┆ 309611          ┆ 0.894881    │
│ name_key_union        ┆ 308063          ┆ 0.890407    │
│ name_union_exact      ┆ 161465          ┆ 0.466689    │
└───────────────────────┴─────────────────┴─────────────┘

By source:

--- S2 ---
shape: (4, 3)
┌───────────────────────┬─────────────────┬─────────────┐
│ signal                ┆ recovered_edges ┆ edge_recall │
│ ---                   ┆ ---             ┆ ---         │
│ str                   ┆ i64             ┆ f64         │
╞═══════════════════════╪═════════════════╪═════════════╡
│ cheap_union           ┆ 159149          ┆ 0.9

In [44]:
# ============================================================
# AMLC 2026 — CELL 51
# COMPOSITE BLOCKER ORACLE + MARGINAL CONTRIBUTION
#
# Goal:
#   1. Find which blocking-key combinations recover the
#      remaining positives without relying on single common keys.
#   2. Measure marginal contribution of each blocker.
#   3. Establish the exact residual-positive population that
#      the approximate retrieval stage must recover.
#
# IMPORTANT:
# This is still ORACLE ANALYSIS.
# We are NOT yet generating millions of candidate pairs.
# ============================================================

print("=" * 95)
print("COMPOSITE BLOCKER ORACLE")
print("=" * 95)

# ------------------------------------------------------------
# Base blocker conditions already produced in Cell 50
# ------------------------------------------------------------

base_blockers = {
    "native_name_exact":
        "name_native_exact",

    "ascii_name_exact":
        "name_ascii_exact",

    "numeric_exact":
        "numeric_exact",

    "address_native_exact":
        "address_native_exact",

    "address_ascii_exact":
        "address_ascii_exact",

    "native_prefix4":
        "native_prefix4",

    "native_suffix4":
        "native_suffix4",

    "ascii_prefix4":
        "ascii_prefix4",

    "ascii_suffix4":
        "ascii_suffix4",

    "ascii_first_token":
        "ascii_first_token",

    "ascii_last_token":
        "ascii_last_token",
}

# ------------------------------------------------------------
# Composite blockers
#
# These are much safer candidates for actual indexing because
# they combine signals instead of allowing one common token
# to explode the block.
# ------------------------------------------------------------

positive_pairs = positive_pairs.with_columns([

    # Transliteration-safe prefix + token
    (
        pl.col("ascii_prefix4")
        & pl.col("ascii_first_token")
    ).alias("ascii_prefix_first"),

    (
        pl.col("ascii_suffix4")
        & pl.col("ascii_last_token")
    ).alias("ascii_suffix_last"),

    (
        pl.col("ascii_first_token")
        & pl.col("ascii_last_token")
    ).alias("ascii_first_last"),

    # Native equivalents
    (
        pl.col("native_prefix4")
        & (
            pl.col("s1_name_native_first")
            == pl.col("matched_name_native_first")
        )
    ).alias("native_prefix_first"),

    (
        pl.col("native_suffix4")
        & (
            pl.col("s1_name_native_last")
            == pl.col("matched_name_native_last")
        )
    ).alias("native_suffix_last"),

    # Numeric + name anchors.
    # Much safer than numeric alone.
    (
        pl.col("numeric_exact")
        & pl.col("ascii_prefix4")
    ).alias("numeric_ascii_prefix"),

    (
        pl.col("numeric_exact")
        & pl.col("ascii_suffix4")
    ).alias("numeric_ascii_suffix"),

    (
        pl.col("numeric_exact")
        & pl.col("ascii_first_token")
    ).alias("numeric_ascii_first"),

    (
        pl.col("numeric_exact")
        & pl.col("ascii_last_token")
    ).alias("numeric_ascii_last"),

    # Address + name anchors.
    (
        pl.col("address_ascii_exact")
        & pl.col("ascii_first_token")
    ).alias("address_ascii_first"),

    (
        pl.col("address_ascii_exact")
        & pl.col("ascii_last_token")
    ).alias("address_ascii_last"),

    (
        pl.col("address_ascii_exact")
        & pl.col("ascii_prefix4")
    ).alias("address_ascii_prefix"),

    (
        pl.col("address_ascii_exact")
        & pl.col("ascii_suffix4")
    ).alias("address_ascii_suffix"),
])

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def recall_of(df, col):
    return float(df[col].mean())


def print_signal_table(df, signals, title):

    rows = []

    for signal in signals:

        rows.append({
            "signal": signal,
            "recovered_edges": int(df[signal].sum()),
            "edge_recall": float(df[signal].mean()),
        })

    out = (
        pl.DataFrame(rows)
        .sort(
            "edge_recall",
            descending=True,
        )
    )

    print(f"\n{title}")
    print("-" * 95)
    print(out)

    return out


composite_signals = [
    "ascii_prefix_first",
    "ascii_suffix_last",
    "ascii_first_last",
    "native_prefix_first",
    "native_suffix_last",

    "numeric_ascii_prefix",
    "numeric_ascii_suffix",
    "numeric_ascii_first",
    "numeric_ascii_last",

    "address_ascii_first",
    "address_ascii_last",
    "address_ascii_prefix",
    "address_ascii_suffix",
]

composite_summary = print_signal_table(
    positive_pairs,
    composite_signals,
    "COMPOSITE BLOCKER RECALL",
)

# ------------------------------------------------------------
# Marginal contribution analysis.
#
# We add blockers one at a time to an existing high-recall
# base and count newly recovered positives.
# ------------------------------------------------------------

# Start from exact normalized names.
current = (
    pl.col("name_union_exact")
)

marginal_rows = []

ordered_steps = [

    (
        "exact_name_union",
        pl.col("name_union_exact"),
    ),

    (
        "+ascii_prefix_first",
        pl.col("ascii_prefix_first"),
    ),

    (
        "+ascii_suffix_last",
        pl.col("ascii_suffix_last"),
    ),

    (
        "+ascii_first_last",
        pl.col("ascii_first_last"),
    ),

    (
        "+numeric_ascii_prefix",
        pl.col("numeric_ascii_prefix"),
    ),

    (
        "+numeric_ascii_suffix",
        pl.col("numeric_ascii_suffix"),
    ),

    (
        "+numeric_ascii_first",
        pl.col("numeric_ascii_first"),
    ),

    (
        "+numeric_ascii_last",
        pl.col("numeric_ascii_last"),
    ),

    (
        "+address_ascii_first",
        pl.col("address_ascii_first"),
    ),

    (
        "+address_ascii_last",
        pl.col("address_ascii_last"),
    ),

    (
        "+address_ascii_prefix",
        pl.col("address_ascii_prefix"),
    ),

    (
        "+address_ascii_suffix",
        pl.col("address_ascii_suffix"),
    ),

    # Keep the original cheap single-key signals at the end
    # so we can see exactly how much recall they add.
    (
        "+native_prefix4",
        pl.col("native_prefix4"),
    ),

    (
        "+native_suffix4",
        pl.col("native_suffix4"),
    ),

    (
        "+ascii_prefix4",
        pl.col("ascii_prefix4"),
    ),

    (
        "+ascii_suffix4",
        pl.col("ascii_suffix4"),
    ),

    (
        "+ascii_first_token",
        pl.col("ascii_first_token"),
    ),

    (
        "+ascii_last_token",
        pl.col("ascii_last_token"),
    ),

    (
        "+numeric_exact",
        pl.col("numeric_exact"),
    ),

    (
        "+address_ascii_exact",
        pl.col("address_ascii_exact"),
    ),
]

current_expr = pl.col("name_union_exact")

for step_name, step_expr in ordered_steps:

    previous_name = (
        f"__previous_{len(marginal_rows)}"
    )

    new_name = (
        f"__current_{len(marginal_rows)}"
    )

    if step_name == "exact_name_union":

        marginal_rows.append({
            "step": step_name,
            "new_edges": int(
                positive_pairs["name_union_exact"].sum()
            ),
            "cumulative_edges": int(
                positive_pairs["name_union_exact"].sum()
            ),
            "cumulative_recall": float(
                positive_pairs["name_union_exact"].mean()
            ),
        })

        continue

    previous_mask = current_expr

    current_expr = (
        current_expr
        | step_expr
    )

    tmp = positive_pairs.select([

        previous_mask.alias("__previous"),
        current_expr.alias("__current"),
    ])

    new_edges = int(
        (
            (~tmp["__previous"])
            & tmp["__current"]
        ).sum()
    )

    cumulative_edges = int(
        tmp["__current"].sum()
    )

    marginal_rows.append({
        "step": step_name,
        "new_edges": new_edges,
        "cumulative_edges": cumulative_edges,
        "cumulative_recall":
            cumulative_edges / positive_pairs.height,
    })


marginal_summary = pl.DataFrame(
    marginal_rows
)

print("\n")
print("=" * 95)
print("MARGINAL RECALL — ORDERED BLOCKER STACK")
print("=" * 95)

print(marginal_summary)

# ------------------------------------------------------------
# True residual after the full cheap union
# ------------------------------------------------------------

residual = (
    positive_pairs
    .filter(
        ~pl.col("cheap_union")
    )
)

print("\n")
print("=" * 95)
print("RESIDUAL POSITIVE EDGES")
print("=" * 95)

print(
    f"Residual edges: {residual.height:,}"
)

print(
    f"Residual fraction: "
    f"{residual.height / positive_pairs.height:.4%}"
)

print("\nBy source:")

print(
    residual
    .group_by("matched_source")
    .agg(
        pl.len().alias("residual_edges")
    )
    .with_columns(
        (
            pl.col("residual_edges")
            / positive_pairs.height
        ).alias("fraction_of_all_edges")
    )
    .sort("matched_source")
)

# ------------------------------------------------------------
# Residual by original S1 cardinality
# ------------------------------------------------------------

print("\nResidual by true S1 match cardinality:")

residual_cardinality = (
    residual
    .join(
        eval_gt
        .group_by("source1_entity_id")
        .agg(
            pl.len().alias("true_match_count")
        ),
        on="source1_entity_id",
        how="left",
    )
    .group_by("true_match_count")
    .agg(
        pl.len().alias("residual_edges")
    )
    .sort("true_match_count")
)

print(residual_cardinality)

print("\n✅ Cell 51 complete.")

COMPOSITE BLOCKER ORACLE

COMPOSITE BLOCKER RECALL
-----------------------------------------------------------------------------------------------
shape: (13, 3)
┌──────────────────────┬─────────────────┬─────────────┐
│ signal               ┆ recovered_edges ┆ edge_recall │
│ ---                  ┆ ---             ┆ ---         │
│ str                  ┆ i64             ┆ f64         │
╞══════════════════════╪═════════════════╪═════════════╡
│ ascii_prefix_first   ┆ 264883          ┆ 0.765602    │
│ native_prefix_first  ┆ 260330          ┆ 0.752442    │
│ ascii_suffix_last    ┆ 203902          ┆ 0.589346    │
│ native_suffix_last   ┆ 190237          ┆ 0.54985     │
│ ascii_first_last     ┆ 178870          ┆ 0.516995    │
│ …                    ┆ …               ┆ …           │
│ numeric_ascii_last   ┆ 118575          ┆ 0.342722    │
│ address_ascii_prefix ┆ 28917           ┆ 0.08358     │
│ address_ascii_first  ┆ 27274           ┆ 0.078831    │
│ address_ascii_suffix ┆ 24754          

In [45]:
# ============================================================
# AMLC 2026 — CELL 52
# REAL BLOCKER VOLUME AUDIT
#
# GOAL
# ----
# Measure how large each blocker actually is on the FULL
# Source-2 / Source-3 population.
#
# We do NOT generate the final candidate graph yet.
# Instead we:
#
#   1. Build compact blocking keys from the existing normalized
#      name indexes.
#   2. Compute source-side key frequencies.
#   3. Join the 100k evaluation S1s against those frequencies.
#   4. Estimate the candidate volume produced by each blocker.
#
# This lets us decide which blockers are safe enough to use
# in the actual candidate generator.
# ============================================================

import gc
import time
import numpy as np
import polars as pl
from anyascii import anyascii

print("=" * 100)
print("AMLC 2026 — REAL BLOCKER VOLUME AUDIT")
print("=" * 100)

# ------------------------------------------------------------
# Compact ASCII normalization
#
# IMPORTANT:
# s2_idx/s3_idx already contain the native normalized name.
# We transliterate THAT normalized representation rather than
# rescanning the raw 10M+ names.
#
# This is sufficient for blocker construction because the
# legal-suffix / punctuation normalization has already happened.
# ------------------------------------------------------------

def ascii_name_from_native(x):
    if x is None:
        return ""
    return anyascii(str(x)).casefold().strip()


def add_name_blocker_keys(lf):

    return (
        lf
        .with_columns(
            pl.col("name_norm")
            .map_elements(
                ascii_name_from_native,
                return_dtype=pl.String,
                skip_nulls=False,
            )
            .alias("name_ascii")
        )
        .with_columns([

            # Exact transliterated normalized name
            pl.col("name_ascii")
            .alias("k_ascii_exact"),

            # Prefix / suffix
            pl.col("name_ascii")
            .str.slice(0, 4)
            .alias("k_ascii_p4"),

            pl.col("name_ascii")
            .str.slice(-4)
            .alias("k_ascii_s4"),

            # First / last token
            pl.col("name_ascii")
            .str.split(" ")
            .list.first()
            .fill_null("")
            .alias("k_ascii_first"),

            pl.col("name_ascii")
            .str.split(" ")
            .list.last()
            .fill_null("")
            .alias("k_ascii_last"),
        ])
        .with_columns([

            # Composite keys that performed well in Cell 51
            pl.concat_str([
                pl.col("k_ascii_p4"),
                pl.lit("|"),
                pl.col("k_ascii_first"),
            ]).alias("k_p4_first"),

            pl.concat_str([
                pl.col("k_ascii_s4"),
                pl.lit("|"),
                pl.col("k_ascii_last"),
            ]).alias("k_s4_last"),

            pl.concat_str([
                pl.col("k_ascii_first"),
                pl.lit("|"),
                pl.col("k_ascii_last"),
            ]).alias("k_first_last"),
        ])
    )


print("\nBuilding source-side blocking-key frequency tables...")

t0 = time.time()

# ------------------------------------------------------------
# S2 frequencies
# ------------------------------------------------------------

s2_keys = add_name_blocker_keys(
    s2_idx.select([
        "entity_id",
        "country",
        "name_norm",
    ])
)

S2_KEY_FREQ = (
    s2_keys
    .select([
        "country",
        "k_ascii_exact",
        "k_p4_first",
        "k_s4_last",
        "k_first_last",
        "k_ascii_p4",
        "k_ascii_s4",
        "k_ascii_first",
        "k_ascii_last",
    ])
    .collect()
)

print(
    f"S2 key table built in "
    f"{time.time() - t0:.1f}s"
)

# ------------------------------------------------------------
# S3 frequencies
# ------------------------------------------------------------

t0 = time.time()

s3_keys = add_name_blocker_keys(
    s3_idx.select([
        "entity_id",
        "country",
        "name_norm",
    ])
)

S3_KEY_FREQ = (
    s3_keys
    .select([
        "country",
        "k_ascii_exact",
        "k_p4_first",
        "k_s4_last",
        "k_first_last",
        "k_ascii_p4",
        "k_ascii_s4",
        "k_ascii_first",
        "k_ascii_last",
    ])
    .collect()
)

print(
    f"S3 key table built in "
    f"{time.time() - t0:.1f}s"
)

# ------------------------------------------------------------
# Evaluation S1 keys
# ------------------------------------------------------------

EVAL_KEYS = (
    s1_norm_lookup
    .select([
        "source1_entity_id",
        "country",
        "name_norm",
    ])
)

EVAL_KEYS = add_name_blocker_keys(
    EVAL_KEYS.rename({
        "source1_entity_id": "entity_id"
    })
)

EVAL_KEYS = (
    EVAL_KEYS
    .rename({
        "entity_id": "source1_entity_id"
    })
)

print("\nEvaluation S1 key rows:", EVAL_KEYS.height)

# ------------------------------------------------------------
# Frequency lookup helper
# ------------------------------------------------------------

def attach_frequency(
    s1_df,
    src_freq,
    key,
    out_name,
):

    freq = (
        src_freq
        .group_by([
            "country",
            key,
        ])
        .agg(
            pl.len().alias(out_name)
        )
    )

    return (
        s1_df
        .join(
            freq,
            on=[
                "country",
                key,
            ],
            how="left",
        )
        .with_columns(
            pl.col(out_name)
            .fill_null(0)
            .cast(pl.Int64)
        )
    )


# ------------------------------------------------------------
# Build volume table
# ------------------------------------------------------------

KEYS = [
    ("k_ascii_exact", "ascii_exact"),
    ("k_p4_first", "prefix4_first"),
    ("k_s4_last", "suffix4_last"),
    ("k_first_last", "first_last"),
    ("k_ascii_p4", "prefix4_only"),
    ("k_ascii_s4", "suffix4_only"),
    ("k_ascii_first", "first_token_only"),
    ("k_ascii_last", "last_token_only"),
]

volume_s2 = EVAL_KEYS

for key, name in KEYS:

    volume_s2 = attach_frequency(
        volume_s2,
        S2_KEY_FREQ,
        key,
        f"s2_{name}",
    )

volume_s3 = EVAL_KEYS

for key, name in KEYS:

    volume_s3 = attach_frequency(
        volume_s3,
        S3_KEY_FREQ,
        key,
        f"s3_{name}",
    )

# ------------------------------------------------------------
# Combine S2 + S3 candidate volume
# ------------------------------------------------------------

volume = (
    volume_s2
    .select([
        "source1_entity_id",
        "country",
        *[
            f"s2_{name}"
            for _, name in KEYS
        ],
    ])
    .join(
        volume_s3.select([
            "source1_entity_id",
            *[
                f"s3_{name}"
                for _, name in KEYS
            ],
        ]),
        on="source1_entity_id",
        how="left",
    )
)

# Total candidate estimate across sources
for _, name in KEYS:

    volume = volume.with_columns(
        (
            pl.col(f"s2_{name}")
            + pl.col(f"s3_{name}")
        ).alias(f"total_{name}")
    )

# ------------------------------------------------------------
# Statistics helper
# ------------------------------------------------------------

rows = []

for _, name in KEYS:

    col = f"total_{name}"

    stats = (
        volume
        .select([
            pl.col(col).mean().alias("mean"),
            pl.col(col).median().alias("median"),
            pl.col(col).quantile(0.90).alias("p90"),
            pl.col(col).quantile(0.95).alias("p95"),
            pl.col(col).quantile(0.99).alias("p99"),
            pl.col(col).max().alias("max"),
            (
                pl.col(col) == 0
            ).mean().alias("zero_fraction"),
            (
                pl.col(col) <= 10
            ).mean().alias("le_10"),
            (
                pl.col(col) <= 50
            ).mean().alias("le_50"),
            (
                pl.col(col) <= 100
            ).mean().alias("le_100"),
        ])
        .row(0)
    )

    rows.append({
        "blocker": name,
        "mean": stats[0],
        "median": stats[1],
        "p90": stats[2],
        "p95": stats[3],
        "p99": stats[4],
        "max": stats[5],
        "zero_fraction": stats[6],
        "fraction_le_10": stats[7],
        "fraction_le_50": stats[8],
        "fraction_le_100": stats[9],
    })

volume_summary = (
    pl.DataFrame(rows)
    .sort("median")
)

print("\n")
print("=" * 100)
print("FULL-DATA BLOCKER VOLUME — 100k EVAL S1")
print("=" * 100)
print(volume_summary)

# ------------------------------------------------------------
# Identify dangerous blockers
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("DANGEROUS / BROAD BLOCKER DIAGNOSTICS")
print("=" * 100)

for _, name in KEYS:

    col = f"total_{name}"

    print(f"\n{name}")

    print(
        volume
        .sort(col, descending=True)
        .select([
            "source1_entity_id",
            "country",
            col,
        ])
        .head(10)
    )

# ------------------------------------------------------------
# Persist compact audit
# ------------------------------------------------------------

audit_path = (
    DRIVE_ER_EDA /
    "blocker_volume_audit.parquet"
)

volume_summary.write_parquet(
    audit_path,
    compression="zstd",
)

print(
    f"\n✅ Saved blocker-volume audit:"
    f"\n{audit_path}"
)

# Free some temporary state
del S2_KEY_FREQ, S3_KEY_FREQ
gc.collect()

print("\n✅ Cell 52 complete.")

AMLC 2026 — REAL BLOCKER VOLUME AUDIT

Building source-side blocking-key frequency tables...
S2 key table built in 8.5s
S3 key table built in 10.4s

Evaluation S1 key rows: 100000


FULL-DATA BLOCKER VOLUME — 100k EVAL S1
shape: (8, 11)
┌────────────┬────────────┬─────────┬─────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ blocker    ┆ mean       ┆ median  ┆ p90     ┆ … ┆ zero_fract ┆ fraction_ ┆ fraction_ ┆ fraction_ │
│ ---        ┆ ---        ┆ ---     ┆ ---     ┆   ┆ ion        ┆ le_10     ┆ le_50     ┆ le_100    │
│ str        ┆ f64        ┆ f64     ┆ f64     ┆   ┆ ---        ┆ ---       ┆ ---       ┆ ---       │
│            ┆            ┆         ┆         ┆   ┆ f64        ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════════╪═════════╪═════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ ascii_exac ┆ 37.59576   ┆ 3.0     ┆ 75.0    ┆ … ┆ 0.0889     ┆ 0.71777   ┆ 0.81241   ┆ 0.9196    │
│ t          ┆            ┆         ┆         ┆   ┆     

In [46]:
# ============================================================
# AMLC 2026 — CELL 53
# FREQUENCY-CAPPED BLOCKER ORACLE
#
# We now determine which blocking keys are actually safe to use.
#
# A key is NOT automatically usable just because it has high
# positive-edge recall. If the same key occurs thousands of
# times, it creates an enormous candidate block.
#
# For each key we measure:
#
#   recall when block size <= 10
#   recall when block size <= 25
#   recall when block size <= 50
#   recall when block size <= 100
#   recall when block size <= 250
#   recall when block size <= 500
#   recall when block size <= 1000
#
# The block size is COUNTRY + KEY across S2 + S3.
# ============================================================

import gc
import time
import numpy as np
import polars as pl

print("=" * 100)
print("FREQUENCY-CAPPED BLOCKER ORACLE")
print("=" * 100)

# ------------------------------------------------------------
# Rebuild only the key columns we need.
# ------------------------------------------------------------

def build_name_keys(lf):

    return (
        lf
        .select([
            "entity_id",
            "country",
            "name_norm",
        ])
        .with_columns([

            pl.col("name_norm")
            .map_elements(
                ascii_name_from_native,
                return_dtype=pl.String,
                skip_nulls=False,
            )
            .alias("name_ascii"),

        ])
        .with_columns([

            pl.col("name_ascii")
            .str.slice(0, 4)
            .alias("p4"),

            pl.col("name_ascii")
            .str.slice(-4)
            .alias("s4"),

            pl.col("name_ascii")
            .str.split(" ")
            .list.first()
            .fill_null("")
            .alias("first"),

            pl.col("name_ascii")
            .str.split(" ")
            .list.last()
            .fill_null("")
            .alias("last"),

        ])
        .with_columns([

            pl.concat_str([
                pl.col("p4"),
                pl.lit("|"),
                pl.col("first"),
            ]).alias("p4_first"),

            pl.concat_str([
                pl.col("s4"),
                pl.lit("|"),
                pl.col("last"),
            ]).alias("s4_last"),

            pl.concat_str([
                pl.col("first"),
                pl.lit("|"),
                pl.col("last"),
            ]).alias("first_last"),

        ])
    )


KEY_COLS = [
    "name_ascii",
    "p4_first",
    "s4_last",
    "first_last",
]

# ------------------------------------------------------------
# Source frequency table
# ------------------------------------------------------------

print("\nBuilding source key frequencies...")

t0 = time.time()

s2_keys = (
    build_name_keys(s2_idx)
    .collect()
)

s3_keys = (
    build_name_keys(s3_idx)
    .collect()
)

print(
    f"Source key tables ready in "
    f"{time.time() - t0:.1f}s"
)

# ------------------------------------------------------------
# Combined country + key frequencies
#
# We DO NOT need the entity IDs here.
# ------------------------------------------------------------

def build_frequency_table(df, key):

    return (
        df
        .filter(
            pl.col(key).is_not_null()
            & (pl.col(key) != "")
        )
        .group_by([
            "country",
            key,
        ])
        .agg(
            pl.len().alias("freq")
        )
    )


freq_tables = {}

for key in KEY_COLS:

    f2 = build_frequency_table(
        s2_keys,
        key,
    )

    f3 = build_frequency_table(
        s3_keys,
        key,
    )

    freq_tables[key] = (
        pl.concat([f2, f3])
        .group_by([
            "country",
            key,
        ])
        .agg(
            pl.col("freq").sum().alias("block_size")
        )
    )

# ------------------------------------------------------------
# Attach source-side block size to every true positive edge.
#
# This is the critical oracle:
#
# For a known TRUE match, how large was the blocker that would
# have retrieved it?
# ------------------------------------------------------------

oracle = (
    positive_pairs
    .select([
        "source1_entity_id",
        "matched_entity_id",
        "matched_source",
        "s1_country",

        "s1_name_ascii",
        "matched_name_ascii",

        "s1_name_native",
        "matched_name_native",
    ])
)

oracle = oracle.with_columns([

    pl.col("s1_name_ascii")
    .alias("name_ascii_key"),

    pl.concat_str([
        pl.col("s1_name_ascii").str.slice(0, 4),
        pl.lit("|"),
        pl.col("s1_name_ascii").str.split(" ").list.first().fill_null(""),
    ]).alias("p4_first_key"),

    pl.concat_str([
        pl.col("s1_name_ascii").str.slice(-4),
        pl.lit("|"),
        pl.col("s1_name_ascii").str.split(" ").list.last().fill_null(""),
    ]).alias("s4_last_key"),

    pl.concat_str([
        pl.col("s1_name_ascii").str.split(" ").list.first().fill_null(""),
        pl.lit("|"),
        pl.col("s1_name_ascii").str.split(" ").list.last().fill_null(""),
    ]).alias("first_last_key"),
])

# ------------------------------------------------------------
# Join each key's true-match block size.
# ------------------------------------------------------------

for key_name, oracle_key in [
    ("name_ascii", "name_ascii_key"),
    ("p4_first", "p4_first_key"),
    ("s4_last", "s4_last_key"),
    ("first_last", "first_last_key"),
]:

    freq = freq_tables[key_name]

    oracle = (
        oracle
        .join(
            freq,
            left_on=[
                "s1_country",
                oracle_key,
            ],
            right_on=[
                "country",
                key_name,
            ],
            how="left",
        )
        .rename({
            "block_size":
                f"{key_name}_block_size"
        })
    )

# ------------------------------------------------------------
# Print raw block-size distributions on TRUE positives
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("TRUE-POSITIVE BLOCK-SIZE DISTRIBUTIONS")
print("=" * 100)

for key_name in KEY_COLS:

    col = f"{key_name}_block_size"

    print(f"\n{key_name}")

    print(
        oracle
        .select([
            pl.col(col).quantile(0.01).alias("p01"),
            pl.col(col).quantile(0.05).alias("p05"),
            pl.col(col).quantile(0.10).alias("p10"),
            pl.col(col).median().alias("p50"),
            pl.col(col).quantile(0.75).alias("p75"),
            pl.col(col).quantile(0.90).alias("p90"),
            pl.col(col).quantile(0.95).alias("p95"),
            pl.col(col).quantile(0.99).alias("p99"),
            pl.col(col).max().alias("max"),
        ])
    )

# ------------------------------------------------------------
# Frequency thresholds
# ------------------------------------------------------------

THRESHOLDS = [
    5,
    10,
    25,
    50,
    100,
    250,
    500,
    1000,
    2500,
    5000,
]

rows = []

for key_name in KEY_COLS:

    col = f"{key_name}_block_size"

    for threshold in THRESHOLDS:

        valid = (
            oracle[col].fill_null(10**18)
            <= threshold
        )

        rows.append({
            "blocker":
                key_name,

            "max_block_size":
                threshold,

            "recovered_edges":
                int(valid.sum()),

            "edge_recall":
                float(valid.mean()),
        })

cap_recall = (
    pl.DataFrame(rows)
)

print("\n")
print("=" * 100)
print("RECALL VS MAXIMUM ALLOWED BLOCK SIZE")
print("=" * 100)

for key_name in KEY_COLS:

    print(f"\n--- {key_name} ---")

    print(
        cap_recall
        .filter(
            pl.col("blocker") == key_name
        )
        .select([
            "max_block_size",
            "recovered_edges",
            "edge_recall",
        ])
    )

# ------------------------------------------------------------
# Combined candidate-blocker recall
#
# We test safe caps for:
#
#   exact transliterated name
#   first+last token
#
# and then progressively allow larger composite blocks.
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("COMBINED SAFE-BLOCKER RECALL")
print("=" * 100)

for threshold in [
    10,
    25,
    50,
    100,
    250,
    500,
    1000,
]:

    exact_ok = (
        oracle["name_ascii_block_size"]
        .fill_null(10**18)
        <= threshold
    )

    first_last_ok = (
        oracle["first_last_block_size"]
        .fill_null(10**18)
        <= threshold
    )

    combined = (
        exact_ok
        | first_last_ok
    )

    print(
        f"max_block={threshold:4d}  "
        f"combined_recall={combined.mean():.6f}  "
        f"edges={combined.sum():,}"
    )

# ------------------------------------------------------------
# Also compare first+last with exact name at asymmetric caps.
# This matters because exact names are usually much safer.
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("ASYMMETRIC EXACT + FIRST/LAST CAPS")
print("=" * 100)

CONFIGS = [
    (100, 25),
    (250, 25),
    (500, 25),
    (1000, 25),

    (100, 50),
    (250, 50),
    (500, 50),
    (1000, 50),

    (250, 100),
    (500, 100),
    (1000, 100),
]

for exact_cap, fl_cap in CONFIGS:

    exact_ok = (
        oracle["name_ascii_block_size"]
        .fill_null(10**18)
        <= exact_cap
    )

    first_last_ok = (
        oracle["first_last_block_size"]
        .fill_null(10**18)
        <= fl_cap
    )

    combined = (
        exact_ok
        | first_last_ok
    )

    print(
        f"exact<={exact_cap:4d} | "
        f"first_last<={fl_cap:3d} | "
        f"recall={combined.mean():.6f} | "
        f"edges={combined.sum():,}"
    )

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

cap_recall.write_parquet(
    DRIVE_ER_EDA / "frequency_capped_blocker_oracle.parquet",
    compression="zstd",
)

print("\n✅ Saved frequency-capped blocker oracle.")

# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

del s2_keys
del s3_keys
del freq_tables
gc.collect()

print("✅ Cell 53 complete.")

FREQUENCY-CAPPED BLOCKER ORACLE

Building source key frequencies...
Source key tables ready in 19.5s


TRUE-POSITIVE BLOCK-SIZE DISTRIBUTIONS

name_ascii
shape: (1, 9)
┌─────┬─────┬─────┬─────┬───┬──────┬───────┬───────┬──────┐
│ p01 ┆ p05 ┆ p10 ┆ p50 ┆ … ┆ p90  ┆ p95   ┆ p99   ┆ max  │
│ --- ┆ --- ┆ --- ┆ --- ┆   ┆ ---  ┆ ---   ┆ ---   ┆ ---  │
│ f64 ┆ f64 ┆ f64 ┆ f64 ┆   ┆ f64  ┆ f64   ┆ f64   ┆ u32  │
╞═════╪═════╪═════╪═════╪═══╪══════╪═══════╪═══════╪══════╡
│ 1.0 ┆ 1.0 ┆ 1.0 ┆ 4.0 ┆ … ┆ 79.0 ┆ 173.0 ┆ 637.0 ┆ 1393 │
└─────┴─────┴─────┴─────┴───┴──────┴───────┴───────┴──────┘

p4_first
shape: (1, 9)
┌─────┬──────┬──────┬────────┬───┬─────────┬─────────┬─────────┬───────┐
│ p01 ┆ p05  ┆ p10  ┆ p50    ┆ … ┆ p90     ┆ p95     ┆ p99     ┆ max   │
│ --- ┆ ---  ┆ ---  ┆ ---    ┆   ┆ ---     ┆ ---     ┆ ---     ┆ ---   │
│ f64 ┆ f64  ┆ f64  ┆ f64    ┆   ┆ f64     ┆ f64     ┆ f64     ┆ u32   │
╞═════╪══════╪══════╪════════╪═══╪═════════╪═════════╪═════════╪═══════╡
│ 5.0 ┆ 13.0 ┆ 24.0 ┆ 1

In [47]:
# ==================================================================================================
# AMLC 2026 — CELL 54
# ACTUAL FREQUENCY-CAPPED CANDIDATE GENERATOR — FINAL ROBUST VERSION
#
# Tier 1:
#   country + exact ASCII-normalized name
#
# Tier 2:
#   country + first-token + last-token
#
# Frequency caps:
#   EXACT_CAP      = 1000
#   FIRST_LAST_CAP = 1000
#
# Robust source-index loading:
#   1. Existing in-memory s2_idx / s3_idx
#   2. Local normalized parquet
#   3. Drive normalized parquet
#   4. Rebuild from raw train TSV
#
# Outputs:
#   eval_candidates_tier12.parquet
#   eval_candidate_residual.parquet
#   eval_candidate_counts_tier12.parquet
# ==================================================================================================

import os
import re
import time
import numpy as np
import polars as pl

from anyascii import anyascii

print("=" * 100)
print("AMLC 2026 — ACTUAL CANDIDATE GENERATOR")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# CONFIG
# --------------------------------------------------------------------------------------------------

EXACT_CAP = 1000
FIRST_LAST_CAP = 1000

DRIVE_PROJECT = "/content/drive/MyDrive/AMLC2026"

LOCAL_PROJECT = "/content/AMLC2026"

DRIVE_BLOCKING_DIR = os.path.join(
    DRIVE_PROJECT,
    "cache",
    "blocking",
)

LOCAL_BLOCKING_DIR = os.path.join(
    LOCAL_PROJECT,
    "cache",
    "blocking",
)

STATE_DIR = os.path.join(
    DRIVE_PROJECT,
    "cache",
    "state",
    "er_eda",
)

os.makedirs(STATE_DIR, exist_ok=True)

# Normalized source indexes
LOCAL_S2_NORM = os.path.join(
    LOCAL_BLOCKING_DIR,
    "train_s2_name_norm.parquet",
)

LOCAL_S3_NORM = os.path.join(
    LOCAL_BLOCKING_DIR,
    "train_s3_name_norm.parquet",
)

DRIVE_S2_NORM = os.path.join(
    DRIVE_BLOCKING_DIR,
    "train_s2_name_norm.parquet",
)

DRIVE_S3_NORM = os.path.join(
    DRIVE_BLOCKING_DIR,
    "train_s3_name_norm.parquet",
)

# Raw source files
TRAIN_DIR = os.path.join(
    DRIVE_PROJECT,
    "dataset",
    "train",
)

S2_RAW = os.path.join(
    TRAIN_DIR,
    "train_source2.tsv",
)

S3_RAW = os.path.join(
    TRAIN_DIR,
    "train_source3.tsv",
)

# State artifacts
S1_EVAL_PATH = os.path.join(
    STATE_DIR,
    "s1_eval.parquet",
)

EDGE_EVAL_PATH = os.path.join(
    STATE_DIR,
    "edge_eval.parquet",
)

CANDIDATE_OUT = os.path.join(
    STATE_DIR,
    "eval_candidates_tier12.parquet",
)

RESIDUAL_OUT = os.path.join(
    STATE_DIR,
    "eval_candidate_residual.parquet",
)

COUNTS_OUT = os.path.join(
    STATE_DIR,
    "eval_candidate_counts_tier12.parquet",
)


# --------------------------------------------------------------------------------------------------
# HELPERS
# --------------------------------------------------------------------------------------------------

def collect_if_lazy(x):
    return x.collect() if isinstance(x, pl.LazyFrame) else x


def resolve_col(df, candidates, label):
    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(
        f"Could not resolve {label}. "
        f"Tried {candidates}. Available columns: {df.columns}"
    )


LEGAL_SUFFIXES = [
    "private limited",
    "private",
    "pvt ltd",
    "pvt",
    "limited",
    "ltd",
    "corporation",
    "corp",
    "company",
    "co",
    "incorporated",
    "inc",
    "llc",
    "llp",
    "sarl",
    "sasu",
    "sas",
    "plc",
    "gmbh",
]


def normalize_name_python(s):
    """
    Canonical name normalization matching the earlier EDA normalization family.
    Used only as a fallback when rebuilding from raw TSVs.
    """

    if s is None:
        return ""

    s = str(s).lower()

    # Normalize ampersand.
    s = s.replace("&", " and ")

    # Keep Unicode letters / digits / whitespace.
    s = "".join(
        ch if (ch.isalnum() or ch.isspace()) else " "
        for ch in s
    )

    s = re.sub(r"\s+", " ", s).strip()

    # Strip legal suffixes from the end.
    changed = True

    while changed and s:
        changed = False

        for suffix in LEGAL_SUFFIXES:

            if s == suffix:
                s = ""
                changed = True
                break

            if s.endswith(" " + suffix):
                s = s[:-(len(suffix) + 1)].rstrip()
                changed = True
                break

    return s


def add_block_keys(df):
    """
    Adds:
        name_ascii
        first_token
        last_token
        first_last
    """

    df = collect_if_lazy(df)

    # AnyAscii is intentionally only an approximate retrieval representation.
    ascii_values = [
        anyascii(x) if x else ""
        for x in df["name_norm"].to_list()
    ]

    df = df.with_columns(
        pl.Series(
            "name_ascii",
            ascii_values,
            dtype=pl.String,
        )
    )

    df = (
        df
        .with_columns(
            pl.col("name_ascii")
            .str.extract(r"^(\S+)", 1)
            .fill_null("")
            .alias("first_token"),

            pl.col("name_ascii")
            .str.extract(r"(\S+)$", 1)
            .fill_null("")
            .alias("last_token"),
        )
        .with_columns(
            (
                pl.col("first_token")
                + pl.lit("\x1f")
                + pl.col("last_token")
            ).alias("first_last")
        )
    )

    return df


def standardize_normalized_index(df, source_label):
    """
    Accept an existing normalized parquet and make its schema deterministic.
    """

    df = collect_if_lazy(df)

    if "entity_id" not in df.columns:
        raise ValueError(
            f"{source_label} normalized index is missing entity_id. "
            f"Columns: {df.columns}"
        )

    if "country" not in df.columns:
        raise ValueError(
            f"{source_label} normalized index is missing country. "
            f"Columns: {df.columns}"
        )

    if "name_norm" not in df.columns:
        raise ValueError(
            f"{source_label} normalized index is missing name_norm. "
            f"Columns: {df.columns}"
        )

    return (
        df
        .select([
            pl.col("entity_id"),
            pl.col("country"),
            pl.col("name_norm")
            .fill_null("")
            .cast(pl.String),
        ])
        .unique(subset=["entity_id"])
    )


def rebuild_source_index_from_raw(raw_path, source_label):
    """
    Last-resort rebuild from raw TSV.
    This is only used if neither memory nor local/Drive normalized caches exist.
    """

    if not os.path.exists(raw_path):
        raise FileNotFoundError(
            f"{source_label} raw TSV also not found:\n{raw_path}"
        )

    print(
        f"\n[{source_label}] Normalized cache unavailable."
        f"\n[{source_label}] Rebuilding compact index from raw TSV:"
        f"\n[{source_label}] {raw_path}"
    )

    t0 = time.time()

    raw = (
        pl.scan_csv(
            raw_path,
            separator="\t",
            has_header=True,
            infer_schema_length=10000,
            ignore_errors=False,
        )
        .rename({
            " entity_id": "entity_id",
        })
        if raw_path.endswith("train_source3.tsv")
        else
        pl.scan_csv(
            raw_path,
            separator="\t",
            has_header=True,
            infer_schema_length=10000,
            ignore_errors=False,
        )
    )

    raw_cols = raw.collect_schema().names()

    name_col = None
    for c in ["business_name", "name", "business"]:
        if c in raw_cols:
            name_col = c
            break

    if name_col is None:
        raise ValueError(
            f"[{source_label}] Could not find business-name column. "
            f"Columns: {raw_cols}"
        )

    out = (
        raw
        .select([
            pl.col("entity_id"),
            pl.col("country"),
            pl.col(name_col)
            .cast(pl.String)
            .fill_null("")
            .alias("_raw_name"),
        ])
        .with_columns(
            pl.col("_raw_name")
            .map_elements(
                normalize_name_python,
                return_dtype=pl.String,
            )
            .alias("name_norm")
        )
        .drop("_raw_name")
        .collect()
    )

    out = out.unique(subset=["entity_id"])

    print(
        f"[{source_label}] Rebuilt {out.height:,} rows "
        f"in {time.time() - t0:.1f}s"
    )

    return out


def load_source_index(source_label, memory_name, local_path, drive_path, raw_path):
    """
    Robust source-index loader:

    1. existing variable in notebook
    2. local cache
    3. Drive cache
    4. raw TSV rebuild
    """

    # ------------------------------------------------------------------------------------------------
    # 1. Existing in-memory dataframe
    # ------------------------------------------------------------------------------------------------

    if memory_name in globals():

        obj = globals()[memory_name]

        try:
            df = collect_if_lazy(obj)

            required = {"entity_id", "country"}

            if required.issubset(set(df.columns)):

                # Already normalized.
                if "name_norm" in df.columns:
                    print(
                        f"[{source_label}] Using existing in-memory index: "
                        f"{memory_name}"
                    )

                    return standardize_normalized_index(
                        df,
                        source_label,
                    )

                # Raw-ish dataframe with business_name.
                name_col = None

                for c in ["business_name", "name", "business"]:
                    if c in df.columns:
                        name_col = c
                        break

                if name_col is not None:

                    print(
                        f"[{source_label}] Using existing in-memory raw index: "
                        f"{memory_name}"
                    )

                    out = (
                        df
                        .select([
                            pl.col("entity_id"),
                            pl.col("country"),
                            pl.col(name_col)
                            .cast(pl.String)
                            .fill_null("")
                            .alias("_raw_name"),
                        ])
                        .with_columns(
                            pl.col("_raw_name")
                            .map_elements(
                                normalize_name_python,
                                return_dtype=pl.String,
                            )
                            .alias("name_norm")
                        )
                        .drop("_raw_name")
                        .unique(subset=["entity_id"])
                    )

                    return out

        except Exception as e:

            print(
                f"[{source_label}] Existing in-memory index unavailable: "
                f"{type(e).__name__}: {e}"
            )


    # ------------------------------------------------------------------------------------------------
    # 2. Local cache
    # ------------------------------------------------------------------------------------------------

    if os.path.exists(local_path):

         print(
             f"[{source_label}] Using LOCAL normalized cache:\n"
             f"  {local_path}"
         )

         return standardize_normalized_index(
             pl.read_parquet(local_path),
             source_label,
         )


    # ------------------------------------------------------------------------------------------------
    # 3. Drive cache
    # ------------------------------------------------------------------------------------------------

    if os.path.exists(drive_path):

         print(
             f"[{source_label}] Using DRIVE normalized cache:\n"
             f"  {drive_path}"
         )

         return standardize_normalized_index(
             pl.read_parquet(drive_path),
             source_label,
         )


    # ------------------------------------------------------------------------------------------------
    # 4. Rebuild from raw
    # ------------------------------------------------------------------------------------------------

    return rebuild_source_index_from_raw(
        raw_path,
        source_label,
    )


# --------------------------------------------------------------------------------------------------
# LOAD S1 EVALUATION SET
# --------------------------------------------------------------------------------------------------

print("\nLoading S1 evaluation set...")

if "s1_eval" in globals():

    s1_base = collect_if_lazy(s1_eval)

else:

    if not os.path.exists(S1_EVAL_PATH):

        raise FileNotFoundError(
            "Could not find s1_eval in memory or checkpoint:\n"
            f"{S1_EVAL_PATH}"
        )

    s1_base = pl.read_parquet(S1_EVAL_PATH)


print(
    f"S1 evaluation rows: "
    f"{s1_base.height:,}"
)


# --------------------------------------------------------------------------------------------------
# LOAD S2 / S3 INDEXES ROBUSTLY
# --------------------------------------------------------------------------------------------------

print("\nCollecting S2/S3 compact indexes...")

t_sources = time.time()

s2_idx = load_source_index(
    source_label="S2",
    memory_name="s2_idx",
    local_path=LOCAL_S2_NORM,
    drive_path=DRIVE_S2_NORM,
    raw_path=S2_RAW,
)

s3_idx = load_source_index(
    source_label="S3",
    memory_name="s3_idx",
    local_path=LOCAL_S3_NORM,
    drive_path=DRIVE_S3_NORM,
    raw_path=S3_RAW,
)

print(
    f"\nS2 rows: {s2_idx.height:,}"
)

print(
    f"S3 rows: {s3_idx.height:,}"
)

print(
    f"Source indexes ready in "
    f"{time.time() - t_sources:.1f}s"
)


# --------------------------------------------------------------------------------------------------
# ADD BLOCK KEYS
# --------------------------------------------------------------------------------------------------

print("\nPreparing block keys...")

s2_idx = add_block_keys(s2_idx)
s3_idx = add_block_keys(s3_idx)


# Keep only columns needed downstream.
s2_idx = (
    s2_idx
    .select([
        "entity_id",
        "country",
        "name_ascii",
        "first_token",
        "last_token",
        "first_last",
    ])
    .unique(subset=["entity_id"])
)

s3_idx = (
    s3_idx
    .select([
        "entity_id",
        "country",
        "name_ascii",
        "first_token",
        "last_token",
        "first_last",
    ])
    .unique(subset=["entity_id"])
)


# --------------------------------------------------------------------------------------------------
# PREP S1 KEYS
# --------------------------------------------------------------------------------------------------

s1_name_col = resolve_col(
    s1_base,
    ["business_name", "name", "business"],
    "S1 business-name column",
)

if "name_norm" in s1_base.columns:

    s1_idx = (
        s1_base
        .select([
            pl.col("entity_id").alias("s1_entity_id"),
            pl.col("country"),
            pl.col("name_norm")
            .fill_null("")
            .cast(pl.String),
        ])
        .unique(subset=["s1_entity_id"])
    )

else:

    s1_idx = (
        s1_base
        .select([
            pl.col("entity_id").alias("s1_entity_id"),
            pl.col("country"),
            pl.col(s1_name_col)
            .cast(pl.String)
            .fill_null("")
            .alias("_raw_name"),
        ])
        .with_columns(
            pl.col("_raw_name")
            .map_elements(
                normalize_name_python,
                return_dtype=pl.String,
            )
            .alias("name_norm")
        )
        .drop("_raw_name")
        .unique(subset=["s1_entity_id"])
    )

s1_idx = add_block_keys(
    s1_idx.rename({
        "s1_entity_id": "entity_id"
    })
)

s1_idx = (
    s1_idx
    .rename({
        "entity_id": "s1_entity_id"
    })
    .select([
        "s1_entity_id",
        "country",
        "name_ascii",
        "first_token",
        "last_token",
        "first_last",
    ])
)


if s1_idx.height != s1_base.height:

    print(
        "WARNING:"
        f" S1 row count changed "
        f"{s1_base.height:,} -> {s1_idx.height:,}"
    )


# --------------------------------------------------------------------------------------------------
# SOURCE BLOCK FREQUENCIES
# --------------------------------------------------------------------------------------------------

print("\nComputing source block frequencies...")

t_freq = time.time()


# Exact-name frequency across S2 + S3.
s2_exact_freq = (
    s2_idx
    .filter(pl.col("name_ascii") != "")
    .group_by(["country", "name_ascii"])
    .len()
    .rename({"len": "freq_exact_s2"})
)

s3_exact_freq = (
    s3_idx
    .filter(pl.col("name_ascii") != "")
    .group_by(["country", "name_ascii"])
    .len()
    .rename({"len": "freq_exact_s3"})
)

exact_freq = (
    s2_exact_freq
    .join(
        s3_exact_freq,
        on=["country", "name_ascii"],
        how="full",
        coalesce=True,
    )
    .with_columns([
        pl.col("freq_exact_s2").fill_null(0),
        pl.col("freq_exact_s3").fill_null(0),
    ])
    .with_columns(
        (
            pl.col("freq_exact_s2")
            + pl.col("freq_exact_s3")
        ).alias("freq_exact")
    )
    .select([
        "country",
        "name_ascii",
        "freq_exact",
    ])
)


# First+last frequency across S2 + S3.
s2_fl_freq = (
    s2_idx
    .filter(
        (pl.col("first_token") != "")
        & (pl.col("last_token") != "")
    )
    .group_by(["country", "first_last"])
    .len()
    .rename({"len": "freq_fl_s2"})
)

s3_fl_freq = (
    s3_idx
    .filter(
        (pl.col("first_token") != "")
        & (pl.col("last_token") != "")
    )
    .group_by(["country", "first_last"])
    .len()
    .rename({"len": "freq_fl_s3"})
)

fl_freq = (
    s2_fl_freq
    .join(
        s3_fl_freq,
        on=["country", "first_last"],
        how="full",
        coalesce=True,
    )
    .with_columns([
        pl.col("freq_fl_s2").fill_null(0),
        pl.col("freq_fl_s3").fill_null(0),
    ])
    .with_columns(
        (
            pl.col("freq_fl_s2")
            + pl.col("freq_fl_s3")
        ).alias("freq_first_last")
    )
    .select([
        "country",
        "first_last",
        "freq_first_last",
    ])
)


# Attach frequencies.
s2_idx = (
    s2_idx
    .join(
        exact_freq,
        on=["country", "name_ascii"],
        how="left",
    )
    .join(
        fl_freq,
        on=["country", "first_last"],
        how="left",
    )
    .with_columns([
        pl.col("freq_exact").fill_null(0),
        pl.col("freq_first_last").fill_null(0),
    ])
)

s3_idx = (
    s3_idx
    .join(
        exact_freq,
        on=["country", "name_ascii"],
        how="left",
    )
    .join(
        fl_freq,
        on=["country", "first_last"],
        how="left",
    )
    .with_columns([
        pl.col("freq_exact").fill_null(0),
        pl.col("freq_first_last").fill_null(0),
    ])
)


# Mark rows whose blockers are within caps.
s2_idx = s2_idx.with_columns([
    (
        (pl.col("freq_exact") > 0)
        & (pl.col("freq_exact") <= EXACT_CAP)
    ).alias("usable_exact"),

    (
        (pl.col("freq_first_last") > 0)
        & (pl.col("freq_first_last") <= FIRST_LAST_CAP)
    ).alias("usable_first_last"),
])

s3_idx = s3_idx.with_columns([
    (
        (pl.col("freq_exact") > 0)
        & (pl.col("freq_exact") <= EXACT_CAP)
    ).alias("usable_exact"),

    (
        (pl.col("freq_first_last") > 0)
        & (pl.col("freq_first_last") <= FIRST_LAST_CAP)
    ).alias("usable_first_last"),
])


print(
    f"Frequency tables ready in "
    f"{time.time() - t_freq:.1f}s"
)


print("\nUsable source rows:")

s2_usable_count = s2_idx.filter(pl.col("usable_exact") | pl.col("usable_first_last")).height
s3_usable_count = s3_idx.filter(pl.col("usable_exact") | pl.col("usable_first_last")).height

print(f"S2: {s2_usable_count:,} / {s2_idx.height:,}")
print(f"S3: {s3_usable_count:,} / {s3_idx.height:,}")


# --------------------------------------------------------------------------------------------------
# GENERATE CANDIDATES
# --------------------------------------------------------------------------------------------------

print("\nGenerating capped exact-name candidates...")

t_gen = time.time()

s1_exact = (
    s1_idx
    .filter(pl.col("name_ascii") != "")
    .join(
        exact_freq,
        on=["country", "name_ascii"],
        how="left",
    )
    .with_columns(
        pl.col("freq_exact").fill_null(0)
    )
    .filter(
        (pl.col("freq_exact") > 0)
        & (pl.col("freq_exact") <= EXACT_CAP)
    )
    .select([
        "s1_entity_id",
        "country",
        "name_ascii",
    ])
)


# S2 exact
c_s2_exact = (
    s1_exact
    .join(
        s2_idx
        .filter(pl.col("usable_exact"))
        .select([
            "entity_id",
            "country",
            "name_ascii",
        ]),
        on=["country", "name_ascii"],
        how="inner",
    )
    .select([
        "s1_entity_id",
        pl.col("entity_id").alias("candidate_entity_id"),
        pl.lit("S2").alias("source"),
        pl.lit(1).cast(pl.UInt8).alias("block_exact"),
        pl.lit(0).cast(pl.UInt8).alias("block_first_last"),
    ])
)


# S3 exact
c_s3_exact = (
    s1_exact
    .join(
        s3_idx
        .filter(pl.col("usable_exact"))
        .select([
            "entity_id",
            "country",
            "name_ascii",
        ]),
        on=["country", "name_ascii"],
        how="inner",
    )
    .select([
        "s1_entity_id",
        pl.col("entity_id").alias("candidate_entity_id"),
        pl.lit("S3").alias("source"),
        pl.lit(1).cast(pl.UInt8).alias("block_exact"),
        pl.lit(0).cast(pl.UInt8).alias("block_first_last"),
    ])
)


# --------------------------------------------------------------------------------------------------

print("Generating capped first+last candidates...")


s1_fl = (
    s1_idx
    .filter(
        (pl.col("first_token") != "")
        & (pl.col("last_token") != "")
    )
    .join(
        fl_freq,
        on=["country", "first_last"],
        how="left",
    )
    .with_columns(
        pl.col("freq_first_last").fill_null(0)
    )
    .filter(
        (pl.col("freq_first_last") > 0)
        & (pl.col("freq_first_last") <= FIRST_LAST_CAP)
    )
    .select([
        "s1_entity_id",
        "country",
        "first_last",
    ])
)


# S2 first+last
c_s2_fl = (
    s1_fl
    .join(
        s2_idx
        .filter(pl.col("usable_first_last"))
        .select([
            "entity_id",
            "country",
            "first_last",
        ]),
        on=["country", "first_last"],
        how="inner",
    )
    .select([
        "s1_entity_id",
        pl.col("entity_id").alias("candidate_entity_id"),
        pl.lit("S2").alias("source"),
        pl.lit(0).cast(pl.UInt8).alias("block_exact"),
        pl.lit(1).cast(pl.UInt8).alias("block_first_last"),
    ])
)


# S3 first+last
c_s3_fl = (
    s1_fl
    .join(
        s3_idx
        .filter(pl.col("usable_first_last"))
        .select([
            "entity_id",
            "country",
            "first_last",
        ]),
        on=["country", "first_last"],
        how="inner",
    )
    .select([
        "s1_entity_id",
        pl.col("entity_id").alias("candidate_entity_id"),
        pl.lit("S3").alias("source"),
        pl.lit(0).cast(pl.UInt8).alias("block_exact"),
        pl.lit(1).cast(pl.UInt8).alias("block_first_last"),
    ])
)


# --------------------------------------------------------------------------------------------------
# UNION + DEDUP
# --------------------------------------------------------------------------------------------------

candidate_pairs = pl.concat(
    [
        c_s2_exact,
        c_s3_exact,
        c_s2_fl,
        c_s3_fl,
    ],
    how="vertical",
)


# Same pair may have been retrieved by BOTH blockers.
candidate_pairs = (
    candidate_pairs
    .group_by([
        "s1_entity_id",
        "candidate_entity_id",
        "source",
    ])
    .agg([
        pl.max("block_exact").alias("block_exact"),
        pl.max("block_first_last").alias("block_first_last"),
    ])
    .sort([
        "s1_entity_id",
        "source",
        "candidate_entity_id",
    ])
)


generation_time = time.time() - t_gen

print(
    f"\nCandidate generation took "
    f"{generation_time:.1f}s"
)

print(
    f"Total candidate pairs: "
    f"{candidate_pairs.height:,}"
)


# --------------------------------------------------------------------------------------------------
# ACTUAL CANDIDATE VOLUME
# IMPORTANT: left join retains ZERO-CANDIDATE S1 rows.
# --------------------------------------------------------------------------------------------------

candidate_counts = (
    s1_idx
    .select("s1_entity_id")
    .join(
        candidate_pairs
        .group_by("s1_entity_id")
        .len()
        .rename({"len": "candidate_count"}),
        on="s1_entity_id",
        how="left",
    )
    .with_columns(
        pl.col("candidate_count")
        .fill_null(0)
        .cast(pl.UInt32)
    )
)


print("\n" + "=" * 100)
print("ACTUAL CANDIDATE VOLUME")
print("=" * 100)


volume_summary = (
    candidate_counts
    .select([
        pl.col("candidate_count")
        .mean()
        .alias("mean"),

        pl.col("candidate_count")
        .median()
        .alias("median"),

        pl.col("candidate_count")
        .quantile(0.90)
        .alias("p90"),

        pl.col("candidate_count")
        .quantile(0.95)
        .alias("p95"),

        pl.col("candidate_count")
        .quantile(0.99)
        .alias("p99"),

        pl.col("candidate_count")
        .max()
        .alias("max"),

        (
            pl.col("candidate_count") == 0
        )
        .mean()
        .alias("zero_rate"),
    ])
)

print(volume_summary)


# --------------------------------------------------------------------------------------------------
# FULL COUNT DISTRIBUTION
# --------------------------------------------------------------------------------------------------

print("\nCandidate count distribution:")

candidate_dist = (
    candidate_counts
    .group_by("candidate_count")
    .len()
    .rename({"len": "s1_rows"})
    .sort("candidate_count")
)

print(
    candidate_dist.head(50)
)


# --------------------------------------------------------------------------------------------------
# BUCKETED DISTRIBUTION
# --------------------------------------------------------------------------------------------------

print("\nCandidate count buckets:")

bucketed = (
    candidate_counts
    .with_columns(
        pl.when(pl.col("candidate_count") == 0)
        .then(pl.lit("0"))

        .when(pl.col("candidate_count") <= 10)
        .then(pl.lit("1-10"))

        .when(pl.col("candidate_count") <= 25)
        .then(pl.lit("11-25"))

        .when(pl.col("candidate_count") <= 50)
        .then(pl.lit("26-50"))

        .when(pl.col("candidate_count") <= 100)
        .then(pl.lit("51-100"))

        .when(pl.col("candidate_count") <= 250)
        .then(pl.lit("101-250"))

        .when(pl.col("candidate_count") <= 500)
        .then(pl.lit("251-500"))

        .when(pl.col("candidate_count") <= 1000)
        .then(pl.lit("501-1000"))

        .otherwise(pl.lit(">1000"))
        .alias("bucket")
    )
    .group_by("bucket")
    .len()
    .rename({"len": "s1_rows"})
    .sort("bucket")
)

print(bucketed)


# --------------------------------------------------------------------------------------------------
# LOAD POSITIVE EDGE ORACLE
# --------------------------------------------------------------------------------------------------

print("\nLoading positive edge oracle...")

if "edge_eval" in globals():

    positive_pairs = collect_if_lazy(edge_eval)

else:

    if not os.path.exists(EDGE_EVAL_PATH):

        raise FileNotFoundError(
            "Could not find edge_eval in memory or checkpoint:\n"
            f"{EDGE_EVAL_PATH}"
        )

    positive_pairs = pl.read_parquet(
        EDGE_EVAL_PATH
    )


print(
    f"Positive edges: "
    f"{positive_pairs.height:,}"
)

print(
    f"Oracle columns: "
    f"{positive_pairs.columns}"
)


# --------------------------------------------------------------------------------------------------
# NORMALIZE POSITIVE-EDGE SCHEMA
# --------------------------------------------------------------------------------------------------

s1_gt_col = resolve_col(
    positive_pairs,
    [
        "source1_entity_id",
        "s1_entity_id",
        "entity_id_s1",
    ],
    "positive-edge S1 column",
)

matched_gt_col = resolve_col(
    positive_pairs,
    [
        "matched_entity_id",
        "candidate_entity_id",
        "matched_id",
        "entity_id",
    ],
    "positive-edge matched entity column",
)


positive_pairs = (
    positive_pairs
    .select([
        pl.col(s1_gt_col).alias("s1_entity_id"),
        pl.col(matched_gt_col).alias("candidate_entity_id"),
    ])
    .unique()
)


# --------------------------------------------------------------------------------------------------
# ACTUAL EDGE RECALL
# --------------------------------------------------------------------------------------------------

positive_hits = (
    positive_pairs
    .join(
        candidate_pairs.select([
            "s1_entity_id",
            "candidate_entity_id",
        ]).unique(),
        on=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        how="inner",
    )
    .unique()
)


total_positive_edges = positive_pairs.height
hit_positive_edges = positive_hits.height

overall_edge_recall = (
    hit_positive_edges / total_positive_edges
    if total_positive_edges
    else 0.0
)


print("\n" + "=" * 100)
print("ACTUAL CANDIDATE RECALL")
print("=" * 100)

print(
    f"Positive edges found : "
    f"{hit_positive_edges:,} / {total_positive_edges:,}"
)

print(
    f"Overall edge recall  : "
    f"{overall_edge_recall:.4%}"
)


# --------------------------------------------------------------------------------------------------
# SOURCE-WISE RECALL
# Reattach source from candidate table where necessary.
# --------------------------------------------------------------------------------------------------

positive_with_source = (
    positive_pairs
    .join(
        candidate_pairs
        .select([
            "s1_entity_id",
            "candidate_entity_id",
            "source",
        ])
        .unique(),
        on=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        how="left",
    )
)


hit_with_source = (
    positive_hits
    .join(
        candidate_pairs
        .select([
            "s1_entity_id",
            "candidate_entity_id",
            "source",
        ])
        .unique(),
        on=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        how="left",
    )
)


gt_source_counts = (
    positive_with_source
    .filter(pl.col("source").is_not_null())
    .group_by("source")
    .len()
    .rename({"len": "positive_edges"})
)


hit_source_counts = (
    hit_with_source
    .filter(pl.col("source").is_not_null())
    .group_by("source")
    .len()
    .rename({"len": "hit_edges"})
)


source_recall = (
    gt_source_counts
    .join(
        hit_source_counts,
        on="source",
        how="left",
    )
    .with_columns(
        pl.col("hit_edges")
        .fill_null(0)
    )
    .with_columns(
        (
            pl.col("hit_edges")
            / pl.col("positive_edges")
        ).alias("edge_recall")
    )
    .sort("source")
)


print("\nSource-wise edge recall:")
print(source_recall)


# --------------------------------------------------------------------------------------------------
# FULL-SET RECOVERY
# Only S1 entities with at least one true match.
# --------------------------------------------------------------------------------------------------

gt_set_counts = (
    positive_pairs
    .group_by("s1_entity_id")
    .len()
    .rename({"len": "true_match_count"})
)


hit_set_counts = (
    positive_hits
    .group_by("s1_entity_id")
    .len()
    .rename({"len": "candidate_hit_count"})
)


full_set_check = (
    gt_set_counts
    .join(
        hit_set_counts,
        on="s1_entity_id",
        how="left",
    )
    .with_columns(
        pl.col("candidate_hit_count")
        .fill_null(0)
    )
    .with_columns(
        (
            pl.col("true_match_count")
            == pl.col("candidate_hit_count")
        )
        .alias("full_set_recovered")
    )
)


positive_s1_count = full_set_check.height

full_set_recovered_count = (
    full_set_check
    .filter(pl.col("full_set_recovered"))
    .height
)


full_set_recovery = (
    full_set_recovered_count / positive_s1_count
    if positive_s1_count
    else 0.0
)


print(
    f"\nPositive S1 entities : "
    f"{positive_s1_count:,}"
)

print(
    f"Full-set recovered   : "
    f"{full_set_recovered_count:,}"
)

print(
    f"Full-set recovery    : "
    f"{full_set_recovery:.4%}"
)


# --------------------------------------------------------------------------------------------------
# RESIDUAL POSITIVE EDGES
# --------------------------------------------------------------------------------------------------

eval_candidate_residual = (
    positive_pairs
    .join(
        candidate_pairs
        .select([
            "s1_entity_id",
            "candidate_entity_id",
        ])
        .unique(),
        on=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        how="anti",
    )
)


print("\n" + "=" * 100)
print("RESIDUAL POSITIVE EDGES")
print("=" * 100)

print(
    f"Missed positive edges : "
    f"{eval_candidate_residual.height:,}"
)

print(
    f"Residual edge rate    : "
    f"{(
        eval_candidate_residual.height / total_positive_edges
        if total_positive_edges
        else 0.0
    ):.4%}"
)


if eval_candidate_residual.height > 0:

    print("\nSample residual positives:")

    print(
        eval_candidate_residual
        .head(25)
    )

else:

    print(
        "\nNONE — candidate generator recovered "
        "every positive edge."
    )


# --------------------------------------------------------------------------------------------------
# RESIDUAL SOURCE BREAKDOWN
# --------------------------------------------------------------------------------------------------

if eval_candidate_residual.height > 0:

    residual_with_source = (
        eval_candidate_residual
        .join(
            positive_with_source,
            on=[
                "s1_entity_id",
                "candidate_entity_id",
            ],
            how="left",
        )
    )

    print("\nResidual positives by source:")

    print(
        residual_with_source
        .filter(pl.col("source").is_not_null())
        .group_by("source")
        .len()
        .rename({"len": "missed_edges"})
        .sort("source")
    )


# --------------------------------------------------------------------------------------------------
# SAVE ARTIFACTS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SAVING ARTIFACTS")
print("=" * 100)

t_save = time.time()


candidate_pairs.write_parquet(
    CANDIDATE_OUT,
    compression="zstd",
)


eval_candidate_residual.write_parquet(
    RESIDUAL_OUT,
    compression="zstd",
)


candidate_counts.write_parquet(
    COUNTS_OUT,
    compression="zstd",
)


print(
    f"Saved candidate pairs    : {CANDIDATE_OUT}"
)

print(
    f"Saved residual positives : {RESIDUAL_OUT}"
)

print(
    f"Saved candidate counts   : {COUNTS_OUT}"
)

print(
    f"Save time                : "
    f"{time.time() - t_save:.1f}s"
)


# --------------------------------------------------------------------------------------------------
# FINAL SUMMARY
# --------------------------------------------------------------------------------------------------

zero_candidate_count = (
    candidate_counts
    .filter(pl.col("candidate_count") == 0)
    .height
)

print("\n" + "=" * 100)
print("CELL 54 COMPLETE")
print("=" * 100)

print(
    f"S1 evaluation rows      : {s1_idx.height:,}"
)

print(
    f"Candidate pairs         : {candidate_pairs.height:,}"
)

print(
    f"Mean candidates / S1    : "
    f"{candidate_counts['candidate_count'].mean():.3f}"
)

print(
    f"Median candidates / S1  : "
    f"{candidate_counts['candidate_count'].median():.3f}"
)

print(
    f"P90 candidates / S1     : "
    f"{candidate_counts['candidate_count'].quantile(0.90):.3f}"
)

print(
    f"P95 candidates / S1     : "
    f"{candidate_counts['candidate_count'].quantile(0.95):.3f}"
)

print(
    f"P99 candidates / S1     : "
    f"{candidate_counts['candidate_count'].quantile(0.99):.3f}"
)

print(
    f"Max candidates / S1     : "
    f"{candidate_counts['candidate_count'].max():,}"
)

print(
    f"Zero-candidate S1s      : "
    f"{zero_candidate_count:,}"
)

print(
    f"Positive edge recall    : "
    f"{overall_edge_recall:.4%}"
)

print(
    f"Full-set recovery       : "
    f"{full_set_recovery:.4%}"
)

print(
    f"Residual positive edges : "
    f"{eval_candidate_residual.height:,}"
)

print("=" * 100)

AMLC 2026 — ACTUAL CANDIDATE GENERATOR

Loading S1 evaluation set...
S1 evaluation rows: 100,000

[S2] Using existing in-memory index: s2_idx
[S3] Using existing in-memory index: s3_idx

S2 rows: 5,034,616
S3 rows: 5,285,603
Source indexes ready in 5.0s

Preparing block keys...

Computing source block frequencies...
Frequency tables ready in 18.5s

Usable source rows:
S2: 5,030,167 / 5,034,616
S3: 5,281,198 / 5,285,603

Generating capped exact-name candidates...
Generating capped first+last candidates...

Candidate generation took 15.9s
Total candidate pairs: 6,568,168

ACTUAL CANDIDATE VOLUME
shape: (1, 7)
┌──────────┬────────┬───────┬───────┬───────┬─────┬───────────┐
│ mean     ┆ median ┆ p90   ┆ p95   ┆ p99   ┆ max ┆ zero_rate │
│ ---      ┆ ---    ┆ ---   ┆ ---   ┆ ---   ┆ --- ┆ ---       │
│ f64      ┆ f64    ┆ f64   ┆ f64   ┆ f64   ┆ u32 ┆ f64       │
╞══════════╪════════╪═══════╪═══════╪═══════╪═════╪═══════════╡
│ 65.68168 ┆ 7.0    ┆ 144.0 ┆ 355.0 ┆ 853.0 ┆ 993 ┆ 0.03916   │
└

In [49]:
# ==================================================================================================
# AMLC 2026 — CELL 55
# BLOCKER RECALL FORENSICS
#
# Purpose:
#   Explain exactly why Cell 54 lost ~49.5% of positive edges.
#
# Tests:
#   1. Actual recall by blocker
#   2. True source of every positive edge
#   3. Country agreement
#   4. Source-specific blocker frequency
#   5. Combined S2+S3 blocker frequency
#   6. How many positives are lost specifically because of combined-frequency caps
# ==================================================================================================

import polars as pl
import time

print("=" * 100)
print("AMLC 2026 — BLOCKER RECALL FORENSICS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. LOAD TRUE POSITIVE EDGES WITH THEIR REAL SOURCE
# --------------------------------------------------------------------------------------------------

print("\nLoading true positive edges...")

if "edge_eval" in globals():
    gt = edge_eval.collect() if isinstance(edge_eval, pl.LazyFrame) else edge_eval
else:
    gt = pl.read_parquet(EDGE_EVAL_PATH)

gt = (
    gt
    .select([
        "source1_entity_id",
        "matched_entity_id",
        "matched_source",
    ])
    .rename({
        "source1_entity_id": "s1_entity_id",
        "matched_entity_id": "candidate_entity_id",
        "matched_source": "true_source",
    })
    .unique()
)

print(f"Positive edges: {gt.height:,}")


# --------------------------------------------------------------------------------------------------
# 2. ACTUAL HIT STATUS + WHICH BLOCKER FOUND IT
# --------------------------------------------------------------------------------------------------

hit_diag = (
    gt
    .join(
        candidate_pairs.select([
            "s1_entity_id",
            "candidate_entity_id",
            "source",
            "block_exact",
            "block_first_last",
        ]),
        on=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        how="left",
    )
    .with_columns([
        pl.col("source").is_not_null().alias("retrieved"),
        pl.col("block_exact").fill_null(0),
        pl.col("block_first_last").fill_null(0),
    ])
)

print("\n" + "=" * 100)
print("ACTUAL BLOCKER RECALL")
print("=" * 100)

total = gt.height

exact_hits = hit_diag.filter(
    pl.col("block_exact") == 1
).height

fl_hits = hit_diag.filter(
    pl.col("block_first_last") == 1
).height

either_hits = hit_diag.filter(
    (pl.col("block_exact") == 1)
    | (pl.col("block_first_last") == 1)
).height

print(
    f"Exact-name blocker hits : "
    f"{exact_hits:,} / {total:,} = {exact_hits / total:.4%}"
)

print(
    f"First+last blocker hits : "
    f"{fl_hits:,} / {total:,} = {fl_hits / total:.4%}"
)

print(
    f"Either blocker          : "
    f"{either_hits:,} / {total:,} = {either_hits / total:.4%}"
)


# --------------------------------------------------------------------------------------------------
# 3. TRUE SOURCE BREAKDOWN
# --------------------------------------------------------------------------------------------------

print("\nTrue-source recall:")

source_breakdown = (
    hit_diag
    .group_by("true_source")
    .agg([
        pl.len().alias("positive_edges"),

        (
            pl.col("retrieved")
            .cast(pl.Int64)
            .sum()
        ).alias("retrieved_edges"),

        (
            pl.col("block_exact")
            .sum()
        ).alias("exact_hits"),

        (
            pl.col("block_first_last")
            .sum()
        ).alias("first_last_hits"),
    ])
    .with_columns(
        (
            pl.col("retrieved_edges")
            / pl.col("positive_edges")
        ).alias("actual_recall")
    )
    .sort("true_source")
)

print(source_breakdown)


# --------------------------------------------------------------------------------------------------
# 4. BUILD SOURCE-SPECIFIC FREQUENCY TABLES
# --------------------------------------------------------------------------------------------------

print("\nComputing SOURCE-SPECIFIC frequencies...")

t0 = time.time()


s2_exact_freq_src = (
    s2_idx
    .filter(pl.col("name_ascii") != "")
    .group_by(["country", "name_ascii"])
    .len()
    .rename({"len": "src_exact_freq"})
)

s3_exact_freq_src = (
    s3_idx
    .filter(pl.col("name_ascii") != "")
    .group_by(["country", "name_ascii"])
    .len()
    .rename({"len": "src_exact_freq"})
)


s2_fl_freq_src = (
    s2_idx
    .filter(
        (pl.col("first_token") != "")
        & (pl.col("last_token") != "")
    )
    .group_by(["country", "first_last"])
    .len()
    .rename({"len": "src_fl_freq"})
)

s3_fl_freq_src = (
    s3_idx
    .filter(
        (pl.col("first_token") != "")
        & (pl.col("last_token") != "")
    )
    .group_by(["country", "first_last"])
    .len()
    .rename({"len": "src_fl_freq"})
)

print(
    f"Source-specific frequency tables ready in "
    f"{time.time() - t0:.1f}s"
)


# --------------------------------------------------------------------------------------------------
# 5. LOOK UP TRUE MATCH RECORDS
# --------------------------------------------------------------------------------------------------

print("\nLooking up true matched source records...")

s2_true = (
    s2_idx
    .join(
        s2_exact_freq_src,
        on=["country", "name_ascii"],
        how="left",
    )
    .join(
        s2_fl_freq_src,
        on=["country", "first_last"],
        how="left",
    )
    .with_columns([
        pl.lit("S2").alias("_lookup_source"),
        pl.col("src_exact_freq").fill_null(0),
        pl.col("src_fl_freq").fill_null(0),
    ])
)

s3_true = (
    s3_idx
    .join(
        s3_exact_freq_src,
        on=["country", "name_ascii"],
        how="left",
    )
    .join(
        s3_fl_freq_src,
        on=["country", "first_last"],
        how="left",
    )
    .with_columns([
        pl.lit("S3").alias("_lookup_source"),
        pl.col("src_exact_freq").fill_null(0),
        pl.col("src_fl_freq").fill_null(0),
    ])
)


source_lookup = pl.concat([
    s2_true,
    s3_true,
], how="vertical")


# --------------------------------------------------------------------------------------------------
# 6. ADD S1 COUNTRY — FIXED VERSION
# --------------------------------------------------------------------------------------------------

print("\nAttaching S1 country...")

s1_country = (
    s1_idx
    .select([
        pl.col("s1_entity_id"),
        pl.col("country").alias("country_s1"),
    ])
)


# --------------------------------------------------------------------------------------------------
# 7. JOIN TRUE POSITIVES TO THEIR ACTUAL SOURCE RECORD
#    Explicit aliases avoid Polars suffix collisions.
# --------------------------------------------------------------------------------------------------

print("Joining positives to source records...")

source_lookup_small = (
    source_lookup
    .select([
        pl.col("entity_id").alias("lookup_entity_id"),
        pl.col("country").alias("country_source"),
        pl.col("name_ascii").alias("source_name_ascii"),
        pl.col("first_last").alias("source_first_last"),
        pl.col("src_exact_freq"),
        pl.col("src_fl_freq"),
    ])
)


forensics = (
    hit_diag
    .join(
        s1_country,
        on="s1_entity_id",
        how="left",
    )
    .join(
        source_lookup_small,
        left_on="candidate_entity_id",
        right_on="lookup_entity_id",
        how="left",
    )
    .with_columns([
        (
            pl.col("country_s1")
            == pl.col("country_source")
        )
        .alias("country_equal"),

        (
            pl.col("src_exact_freq")
            .fill_null(0)
            <= EXACT_CAP
        )
        .alias("source_exact_within_cap"),

        (
            pl.col("src_fl_freq")
            .fill_null(0)
            <= FIRST_LAST_CAP
        )
        .alias("source_fl_within_cap"),
    ])
)


# --------------------------------------------------------------------------------------------------
# 8. MISSED POSITIVES ONLY
# --------------------------------------------------------------------------------------------------

missed = (
    forensics
    .filter(~pl.col("retrieved"))
)

print("\n" + "=" * 100)
print("MISSED-POSITIVE FORENSICS")
print("=" * 100)

print(
    f"Missed positives: {missed.height:,}"
)


# --------------------------------------------------------------------------------------------------
# 9. COUNTRY AGREEMENT
# --------------------------------------------------------------------------------------------------

country_stats = (
    missed
    .select([
        pl.len().alias("missed_edges"),

        (
            pl.col("country_equal")
            .fill_null(False)
            .cast(pl.Int64)
            .sum()
        )
        .alias("country_equal"),

        (
            pl.col("country_equal")
            .fill_null(False)
            .cast(pl.Float64)
            .mean()
        )
        .alias("country_equal_rate"),
    ])
)

print("\nCountry agreement among missed positives:")
print(country_stats)


# --------------------------------------------------------------------------------------------------
# 10. SOURCE-SPECIFIC CAP VS COMBINED-CAP DIAGNOSIS
# --------------------------------------------------------------------------------------------------

print("\nSource-specific blocker eligibility among MISSED positives:")

cap_diag = (
    missed
    .select([
        pl.len().alias("missed_edges"),

        (
            pl.col("source_exact_within_cap")
            .fill_null(False)
            .cast(pl.Int64)
            .sum()
        )
        .alias("exact_source_allowed"),

        (
            pl.col("source_fl_within_cap")
            .fill_null(False)
            .cast(pl.Int64)
            .sum()
        )
        .alias("first_last_source_allowed"),

        (
            (
                pl.col("source_exact_within_cap")
                .fill_null(False)
            )
            |
            (
                pl.col("source_fl_within_cap")
                .fill_null(False)
            )
        )
        .cast(pl.Int64)
        .sum()
        .alias("either_source_allowed"),
    ])
)

print(cap_diag)


# --------------------------------------------------------------------------------------------------
# 11. MISSED POSITIVES BY TRUE SOURCE
# --------------------------------------------------------------------------------------------------

print("\nMissed positives by TRUE SOURCE:")

missed_source = (
    missed
    .group_by("true_source")
    .agg([
        pl.len().alias("missed_edges"),

        (
            pl.col("source_exact_within_cap")
            .fill_null(False)
            .cast(pl.Int64)
            .sum()
        )
        .alias("exact_allowed_source_specific"),

        (
            pl.col("source_fl_within_cap")
            .fill_null(False)
            .cast(pl.Int64)
            .sum()
        )
        .alias("first_last_allowed_source_specific"),

        (
            (
                pl.col("source_exact_within_cap")
                .fill_null(False)
            )
            |
            (
                pl.col("source_fl_within_cap")
                .fill_null(False)
            )
        )
        .cast(pl.Int64)
        .sum()
        .alias("either_allowed_source_specific"),
    ])
    .sort("true_source")
)

print(missed_source)


# --------------------------------------------------------------------------------------------------
# 12. SAMPLE MISSED POSITIVES
# --------------------------------------------------------------------------------------------------

print("\nSample of 30 missed positives with blocker frequencies:")

print(
    missed
    .select([
        "s1_entity_id",
        "candidate_entity_id",
        "true_source",
        "country_s1",
        "country_source",
        "country_equal",
        "src_exact_freq",
        "src_fl_freq",
        "source_exact_within_cap",
        "source_fl_within_cap",
    ])
    .head(30)
)


# --------------------------------------------------------------------------------------------------
# 13. EXTRA: HOW MANY MISSES WOULD BE RECOVERED BY SOURCE-SPECIFIC CAPS?
#
# This is the critical number.
# --------------------------------------------------------------------------------------------------

source_specific_recoverable = (
    missed
    .filter(
        (
            pl.col("source_exact_within_cap")
            .fill_null(False)
        )
        |
        (
            pl.col("source_fl_within_cap")
            .fill_null(False)
        )
    )
    .height
)

print("\n" + "=" * 100)
print("SOURCE-SPECIFIC CAP RECOVERY")
print("=" * 100)

print(
    f"Missed positives currently            : "
    f"{missed.height:,}"
)

print(
    f"Recoverable with source-specific caps : "
    f"{source_specific_recoverable:,}"
)

print(
    f"Fraction of current misses rescued    : "
    f"{(
        source_specific_recoverable / missed.height
        if missed.height else 0.0
    ):.4%}"
)

print(
    f"Potential overall edge recall         : "
    f"{(
        (
            total
            - missed.height
            + source_specific_recoverable
        ) / total
        if total else 0.0
    ):.4%}"
)


# --------------------------------------------------------------------------------------------------
# 14. CELL COMPLETE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("CELL 55 COMPLETE")
print("=" * 100)

print(
    """
Interpretation:

1. If `Recoverable with source-specific caps` is LARGE:
   the combined S2+S3 frequency cap is the main culprit.

2. If it is SMALL:
   the problem is not primarily the combined frequency cap.
   We then inspect normalization / key construction / country filtering.

3. Do NOT build the LightGBM pairwise feature matrix until this
   candidate-recall issue is resolved.
"""
)

AMLC 2026 — BLOCKER RECALL FORENSICS

Loading true positive edges...
Positive edges: 345,980

ACTUAL BLOCKER RECALL
Exact-name blocker hits : 157,993 / 345,980 = 45.6654%
First+last blocker hits : 169,359 / 345,980 = 48.9505%
Either blocker          : 174,631 / 345,980 = 50.4743%

True-source recall:
shape: (2, 6)
┌─────────────┬────────────────┬─────────────────┬────────────┬─────────────────┬───────────────┐
│ true_source ┆ positive_edges ┆ retrieved_edges ┆ exact_hits ┆ first_last_hits ┆ actual_recall │
│ ---         ┆ ---            ┆ ---             ┆ ---        ┆ ---             ┆ ---           │
│ str         ┆ u32            ┆ i64             ┆ i64        ┆ i64             ┆ f64           │
╞═════════════╪════════════════╪═════════════════╪════════════╪═════════════════╪═══════════════╡
│ S2          ┆ 167274         ┆ 85990           ┆ 77715      ┆ 83353           ┆ 0.514067      │
│ S3          ┆ 178706         ┆ 88641           ┆ 80278      ┆ 86006           ┆ 0.496016      

In [50]:
# ==================================================================================================
# AMLC 2026 — CELL 56
# SOURCE-SPECIFIC FREQUENCY-CAPPED CANDIDATE GENERATOR
#
# FIX:
#   Frequency caps are applied independently to S2 and S3.
#
# WRONG:
#   freq(country, key, S2 + S3) <= cap
#
# CORRECT:
#   freq(country, key, S2) <= cap
#   freq(country, key, S3) <= cap
#
# Tier 1:
#   country + exact ASCII-normalized name
#
# Tier 2:
#   country + first-token + last-token
#
# ==================================================================================================

import os
import time
import polars as pl


print("=" * 100)
print("AMLC 2026 — SOURCE-SPECIFIC CAPPED CANDIDATE GENERATOR")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# CONFIG
# --------------------------------------------------------------------------------------------------

EXACT_CAP = 1000
FIRST_LAST_CAP = 1000

STATE_DIR = "/content/drive/MyDrive/AMLC2026/cache/state/er_eda"

os.makedirs(STATE_DIR, exist_ok=True)

CANDIDATE_OUT = os.path.join(
    STATE_DIR,
    "eval_candidates_tier12_source_capped.parquet",
)

RESIDUAL_OUT = os.path.join(
    STATE_DIR,
    "eval_candidate_residual_source_capped.parquet",
)

COUNTS_OUT = os.path.join(
    STATE_DIR,
    "eval_candidate_counts_source_capped.parquet",
)


# --------------------------------------------------------------------------------------------------
# REQUIRED IN-MEMORY TABLES
# --------------------------------------------------------------------------------------------------

required = [
    "s1_idx",
    "s2_idx",
    "s3_idx",
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required in-memory tables: "
        + ", ".join(missing)
        + "\nRun the earlier source-index preparation cells first."
    )


# --------------------------------------------------------------------------------------------------
# LOAD TABLES
# --------------------------------------------------------------------------------------------------

s1 = s1_idx.collect() if isinstance(s1_idx, pl.LazyFrame) else s1_idx
s2 = s2_idx.collect() if isinstance(s2_idx, pl.LazyFrame) else s2_idx
s3 = s3_idx.collect() if isinstance(s3_idx, pl.LazyFrame) else s3_idx

print(f"S1 evaluation rows : {s1.height:,}")
print(f"S2 source rows      : {s2.height:,}")
print(f"S3 source rows      : {s3.height:,}")


# --------------------------------------------------------------------------------------------------
# VERIFY REQUIRED COLUMNS
# --------------------------------------------------------------------------------------------------

required_s1_cols = {
    "s1_entity_id",
    "country",
    "name_ascii",
    "first_last",
}

required_src_cols = {
    "entity_id",
    "country",
    "name_ascii",
    "first_last",
}

missing_s1 = required_s1_cols - set(s1.columns)
missing_s2 = required_src_cols - set(s2.columns)
missing_s3 = required_src_cols - set(s3.columns)

if missing_s1:
    raise ValueError(
        f"S1 index missing columns: {sorted(missing_s1)}"
    )

if missing_s2:
    raise ValueError(
        f"S2 index missing columns: {sorted(missing_s2)}"
    )

if missing_s3:
    raise ValueError(
        f"S3 index missing columns: {sorted(missing_s3)}"
    )


# --------------------------------------------------------------------------------------------------
# SOURCE-SPECIFIC FREQUENCY TABLES
# --------------------------------------------------------------------------------------------------

print("\nComputing SOURCE-SPECIFIC blocker frequencies...")

t_freq = time.time()


# -------------------------------
# S2
# -------------------------------

s2_exact_freq = (
    s2
    .filter(pl.col("name_ascii") != "")
    .group_by([
        "country",
        "name_ascii",
    ])
    .len()
    .rename({
        "len": "freq_exact_s2",
    })
)

s2_fl_freq = (
    s2
    .filter(
        (pl.col("first_last") != "")
        & (pl.col("first_last") != "\x1f")
    )
    .group_by([
        "country",
        "first_last",
    ])
    .len()
    .rename({
        "len": "freq_fl_s2",
    })
)


# -------------------------------
# S3
# -------------------------------

s3_exact_freq = (
    s3
    .filter(pl.col("name_ascii") != "")
    .group_by([
        "country",
        "name_ascii",
    ])
    .len()
    .rename({
        "len": "freq_exact_s3",
    })
)

s3_fl_freq = (
    s3
    .filter(
        (pl.col("first_last") != "")
        & (pl.col("first_last") != "\x1f")
    )
    .group_by([
        "country",
        "first_last",
    ])
    .len()
    .rename({
        "len": "freq_fl_s3",
    })
)


print(
    f"Frequency tables ready in "
    f"{time.time() - t_freq:.1f}s"
)


# --------------------------------------------------------------------------------------------------
# SOURCE-SPECIFIC USABILITY
#
# IMPORTANT:
# Do NOT reuse the previous `usable_exact` / `usable_first_last`
# columns because those were generated from the WRONG combined frequency.
# --------------------------------------------------------------------------------------------------

s2 = (
    s2
    .join(
        s2_exact_freq,
        on=[
            "country",
            "name_ascii",
        ],
        how="left",
    )
    .join(
        s2_fl_freq,
        on=[
            "country",
            "first_last",
        ],
        how="left",
    )
    .with_columns([
        pl.col("freq_exact_s2")
        .fill_null(0),

        pl.col("freq_fl_s2")
        .fill_null(0),
    ])
    .with_columns([
        (
            (pl.col("freq_exact_s2") > 0)
            & (pl.col("freq_exact_s2") <= EXACT_CAP)
        )
        .alias("usable_exact_s2"),

        (
            (pl.col("freq_fl_s2") > 0)
            & (pl.col("freq_fl_s2") <= FIRST_LAST_CAP)
        )
        .alias("usable_fl_s2"),
    ])
)


s3 = (
    s3
    .join(
        s3_exact_freq,
        on=[
            "country",
            "name_ascii",
        ],
        how="left",
    )
    .join(
        s3_fl_freq,
        on=[
            "country",
            "first_last",
        ],
        how="left",
    )
    .with_columns([
        pl.col("freq_exact_s3")
        .fill_null(0),

        pl.col("freq_fl_s3")
        .fill_null(0),
    ])
    .with_columns([
        (
            (pl.col("freq_exact_s3") > 0)
            & (pl.col("freq_exact_s3") <= EXACT_CAP)
        )
        .alias("usable_exact_s3"),

        (
            (pl.col("freq_fl_s3") > 0)
            & (pl.col("freq_fl_s3") <= FIRST_LAST_CAP)
        )
        .alias("usable_fl_s3"),
    ])
)


print("\nUsable source rows:")

s2_usable = s2.filter(
    pl.col("usable_exact_s2")
    | pl.col("usable_fl_s2")
).height

s3_usable = s3.filter(
    pl.col("usable_exact_s3")
    | pl.col("usable_fl_s3")
).height

print(
    f"S2: {s2_usable:,} / {s2.height:,}"
)

print(
    f"S3: {s3_usable:,} / {s3.height:,}"
)


# --------------------------------------------------------------------------------------------------
# S1 BLOCK KEYS
# --------------------------------------------------------------------------------------------------

s1_exact = (
    s1
    .filter(pl.col("name_ascii") != "")
    .select([
        "s1_entity_id",
        "country",
        "name_ascii",
    ])
)

s1_fl = (
    s1
    .filter(
        (pl.col("first_last") != "")
        & (pl.col("first_last") != "\x1f")
    )
    .select([
        "s1_entity_id",
        "country",
        "first_last",
    ])
)


# --------------------------------------------------------------------------------------------------
# EXACT NAME CANDIDATES
# --------------------------------------------------------------------------------------------------

print("\nGenerating SOURCE-SPECIFIC exact-name candidates...")

t_gen = time.time()


# S2 exact
c_s2_exact = (
    s1_exact
    .join(
        s2
        .filter(pl.col("usable_exact_s2"))
        .select([
            "entity_id",
            "country",
            "name_ascii",
        ]),
        on=[
            "country",
            "name_ascii",
        ],
        how="inner",
    )
    .select([
        "s1_entity_id",
        pl.col("entity_id")
        .alias("candidate_entity_id"),
        pl.lit("S2")
        .alias("source"),
        pl.lit(1)
        .cast(pl.UInt8)
        .alias("block_exact"),
        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_first_last"),
    ])
)


# S3 exact
c_s3_exact = (
    s1_exact
    .join(
        s3
        .filter(pl.col("usable_exact_s3"))
        .select([
            "entity_id",
            "country",
            "name_ascii",
        ]),
        on=[
            "country",
            "name_ascii",
        ],
        how="inner",
    )
    .select([
        "s1_entity_id",
        pl.col("entity_id")
        .alias("candidate_entity_id"),
        pl.lit("S3")
        .alias("source"),
        pl.lit(1)
        .cast(pl.UInt8)
        .alias("block_exact"),
        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_first_last"),
    ])
)


# --------------------------------------------------------------------------------------------------
# FIRST + LAST CANDIDATES
# --------------------------------------------------------------------------------------------------

print("Generating SOURCE-SPECIFIC first+last candidates...")


# S2 first-last
c_s2_fl = (
    s1_fl
    .join(
        s2
        .filter(pl.col("usable_fl_s2"))
        .select([
            "entity_id",
            "country",
            "first_last",
        ]),
        on=[
            "country",
            "first_last",
        ],
        how="inner",
    )
    .select([
        "s1_entity_id",
        pl.col("entity_id")
        .alias("candidate_entity_id"),
        pl.lit("S2")
        .alias("source"),
        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_exact"),
        pl.lit(1)
        .cast(pl.UInt8)
        .alias("block_first_last"),
    ])
)


# S3 first-last
c_s3_fl = (
    s1_fl
    .join(
        s3
        .filter(pl.col("usable_fl_s3"))
        .select([
            "entity_id",
            "country",
            "first_last",
        ]),
        on=[
            "country",
            "first_last",
        ],
        how="inner",
    )
    .select([
        "s1_entity_id",
        pl.col("entity_id")
        .alias("candidate_entity_id"),
        pl.lit("S3")
        .alias("source"),
        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_exact"),
        pl.lit(1)
        .cast(pl.UInt8)
        .alias("block_first_last"),
    ])
)


# --------------------------------------------------------------------------------------------------
# UNION + DEDUP
# --------------------------------------------------------------------------------------------------

candidate_pairs = pl.concat(
    [
        c_s2_exact,
        c_s3_exact,
        c_s2_fl,
        c_s3_fl,
    ],
    how="vertical",
)


candidate_pairs = (
    candidate_pairs
    .group_by([
        "s1_entity_id",
        "candidate_entity_id",
        "source",
    ])
    .agg([
        pl.max("block_exact")
        .alias("block_exact"),

        pl.max("block_first_last")
        .alias("block_first_last"),
    ])
    .sort([
        "s1_entity_id",
        "source",
        "candidate_entity_id",
    ])
)


print(
    f"\nCandidate generation took "
    f"{time.time() - t_gen:.1f}s"
)

print(
    f"Total candidate pairs: "
    f"{candidate_pairs.height:,}"
)


# --------------------------------------------------------------------------------------------------
# CANDIDATE COUNT PER S1
# --------------------------------------------------------------------------------------------------

candidate_counts = (
    s1
    .select("s1_entity_id")
    .join(
        candidate_pairs
        .group_by("s1_entity_id")
        .len()
        .rename({
            "len": "candidate_count",
        }),
        on="s1_entity_id",
        how="left",
    )
    .with_columns(
        pl.col("candidate_count")
        .fill_null(0)
        .cast(pl.UInt32)
    )
)


# --------------------------------------------------------------------------------------------------
# VOLUME
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SOURCE-SPECIFIC CANDIDATE VOLUME")
print("=" * 100)


volume_summary = (
    candidate_counts
    .select([
        pl.col("candidate_count")
        .mean()
        .alias("mean"),

        pl.col("candidate_count")
        .median()
        .alias("median"),

        pl.col("candidate_count")
        .quantile(0.90)
        .alias("p90"),

        pl.col("candidate_count")
        .quantile(0.95)
        .alias("p95"),

        pl.col("candidate_count")
        .quantile(0.99)
        .alias("p99"),

        pl.col("candidate_count")
        .max()
        .alias("max"),

        (
            pl.col("candidate_count") == 0
        )
        .mean()
        .alias("zero_rate"),
    ])
)

print(volume_summary)


# --------------------------------------------------------------------------------------------------
# LOAD GROUND TRUTH
# --------------------------------------------------------------------------------------------------

if "edge_eval" in globals():

    gt = (
        edge_eval.collect()
        if isinstance(edge_eval, pl.LazyFrame)
        else edge_eval
    )

else:

    gt = pl.read_parquet(
        os.path.join(
            STATE_DIR,
            "edge_eval.parquet",
        )
    )


gt = (
    gt
    .select([
        "source1_entity_id",
        "matched_entity_id",
        "matched_source",
    ])
    .rename({
        "source1_entity_id": "s1_entity_id",
        "matched_entity_id": "candidate_entity_id",
        "matched_source": "true_source",
    })
    .unique()
)


print(
    f"\nPositive edges: {gt.height:,}"
)


# --------------------------------------------------------------------------------------------------
# ACTUAL RECALL
#
# IMPORTANT:
# JOIN ON source TOO.
# --------------------------------------------------------------------------------------------------

hits = (
    gt
    .join(
        candidate_pairs.select([
            "s1_entity_id",
            "candidate_entity_id",
            "source",
        ]),
        left_on=[
            "s1_entity_id",
            "candidate_entity_id",
            "true_source",
        ],
        right_on=[
            "s1_entity_id",
            "candidate_entity_id",
            "source",
        ],
        how="inner",
    )
    .unique([
        "s1_entity_id",
        "candidate_entity_id",
        "true_source",
    ])
)


total_edges = gt.height
hit_edges = hits.height

edge_recall = (
    hit_edges / total_edges
    if total_edges
    else 0.0
)


print("\n" + "=" * 100)
print("ACTUAL CANDIDATE RECALL")
print("=" * 100)

print(
    f"Positive edges found : "
    f"{hit_edges:,} / {total_edges:,}"
)

print(
    f"Overall edge recall  : "
    f"{edge_recall:.4%}"
)


# --------------------------------------------------------------------------------------------------
# SOURCE-WISE RECALL
# --------------------------------------------------------------------------------------------------

source_recall = (
    gt
    .group_by("true_source")
    .len()
    .rename({
        "len": "positive_edges",
    })
    .join(
        hits
        .group_by("true_source")
        .len()
        .rename({
            "len": "hit_edges",
        }),
        on="true_source",
        how="left",
    )
    .with_columns([
        pl.col("hit_edges")
        .fill_null(0),

        (
            pl.col("hit_edges")
            .fill_null(0)
            / pl.col("positive_edges")
        )
        .alias("edge_recall"),
    ])
    .sort("true_source")
)

print("\nSource-wise recall:")
print(source_recall)


# --------------------------------------------------------------------------------------------------
# FULL-SET RECOVERY
# --------------------------------------------------------------------------------------------------

gt_counts = (
    gt
    .group_by("s1_entity_id")
    .len()
    .rename({
        "len": "true_match_count",
    })
)

hit_counts = (
    hits
    .group_by("s1_entity_id")
    .len()
    .rename({
        "len": "candidate_hit_count",
    })
)

full_set = (
    gt_counts
    .join(
        hit_counts,
        on="s1_entity_id",
        how="left",
    )
    .with_columns(
        pl.col("candidate_hit_count")
        .fill_null(0)
    )
    .with_columns(
        (
            pl.col("true_match_count")
            == pl.col("candidate_hit_count")
        )
        .alias("full_set_recovered")
    )
)

positive_s1 = full_set.height

full_recovered = (
    full_set
    .filter(pl.col("full_set_recovered"))
    .height
)

full_recovery = (
    full_recovered / positive_s1
    if positive_s1
    else 0.0
)


print("\n" + "=" * 100)
print("FULL-SET RECOVERY")
print("=" * 100)

print(
    f"Positive S1 entities : {positive_s1:,}"
)

print(
    f"Fully recovered      : {full_recovered:,}"
)

print(
    f"Full-set recovery    : {full_recovery:.4%}"
)


# --------------------------------------------------------------------------------------------------
# RESIDUAL
# --------------------------------------------------------------------------------------------------

residual = (
    gt
    .join(
        candidate_pairs.select([
            "s1_entity_id",
            "candidate_entity_id",
            "source",
        ]),
        left_on=[
            "s1_entity_id",
            "candidate_entity_id",
            "true_source",
        ],
        right_on=[
            "s1_entity_id",
            "candidate_entity_id",
            "source",
        ],
        how="anti",
    )
)


print("\n" + "=" * 100)
print("RESIDUAL POSITIVE EDGES")
print("=" * 100)

print(
    f"Missed positives: {residual.height:,}"
)

print(
    f"Residual edge rate: "
    f"{(
        residual.height / total_edges
        if total_edges
        else 0.0
    ): .4%}"
)

if residual.height > 0:

    print("\nSample residuals:")

    print(
        residual
        .head(25)
    )

else:

    print(
        "\nZERO RESIDUAL POSITIVE EDGES."
    )


# --------------------------------------------------------------------------------------------------
# SAVE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SAVING SOURCE-CAPPED CANDIDATE ARTIFACTS")
print("=" * 100)

candidate_pairs.write_parquet(
    CANDIDATE_OUT,
    compression="zstd",
)

candidate_counts.write_parquet(
    COUNTS_OUT,
    compression="zstd",
)

residual.write_parquet(
    RESIDUAL_OUT,
    compression="zstd",
)


print(
    f"Candidates : {CANDIDATE_OUT}"
)

print(
    f"Counts     : {COUNTS_OUT}"
)

print(
    f"Residual   : {RESIDUAL_OUT}"
)


# --------------------------------------------------------------------------------------------------
# FINAL SUMMARY
# --------------------------------------------------------------------------------------------------

zero_count = (
    candidate_counts
    .filter(pl.col("candidate_count") == 0)
    .height
)

print("\n" + "=" * 100)
print("CELL 56 COMPLETE")
print("=" * 100)

print(
    f"S1 rows                 : {s1.height:,}"
)

print(
    f"Candidate pairs         : {candidate_pairs.height:,}"
)

print(
    f"Mean candidates / S1    : "
    f"{candidate_counts['candidate_count'].mean():.3f}"
)

print(
    f"Median candidates / S1  : "
    f"{candidate_counts['candidate_count'].median():.3f}"
)

print(
    f"P90 candidates / S1     : "
    f"{candidate_counts['candidate_count'].quantile(0.90):.3f}"
)

print(
    f"P95 candidates / S1     : "
    f"{candidate_counts['candidate_count'].quantile(0.95):.3f}"
)

print(
    f"P99 candidates / S1     : "
    f"{candidate_counts['candidate_count'].quantile(0.99):.3f}"
)

print(
    f"Max candidates / S1     : "
    f"{candidate_counts['candidate_count'].max():,}"
)

print(
    f"Zero-candidate S1s      : {zero_count:,}"
)

print(
    f"Positive edge recall    : {edge_recall:.4%}"
)

print(
    f"Full-set recovery       : {full_recovery:.4%}"
)

print(
    f"Residual positive edges : {residual.height:,}"
)

print("=" * 100)

AMLC 2026 — SOURCE-SPECIFIC CAPPED CANDIDATE GENERATOR
S1 evaluation rows : 100,000
S2 source rows      : 5,034,616
S3 source rows      : 5,285,603

Computing SOURCE-SPECIFIC blocker frequencies...
Frequency tables ready in 10.2s

Usable source rows:
S2: 5,034,602 / 5,034,616
S3: 5,285,588 / 5,285,603

Generating SOURCE-SPECIFIC exact-name candidates...
Generating SOURCE-SPECIFIC first+last candidates...

Candidate generation took 22.1s
Total candidate pairs: 9,494,493

SOURCE-SPECIFIC CANDIDATE VOLUME
shape: (1, 7)
┌──────────┬────────┬───────┬───────┬────────┬──────┬───────────┐
│ mean     ┆ median ┆ p90   ┆ p95   ┆ p99    ┆ max  ┆ zero_rate │
│ ---      ┆ ---    ┆ ---   ┆ ---   ┆ ---    ┆ ---  ┆ ---       │
│ f64      ┆ f64    ┆ f64   ┆ f64   ┆ f64    ┆ u32  ┆ f64       │
╞══════════╪════════╪═══════╪═══════╪════════╪══════╪═══════════╡
│ 94.94493 ┆ 7.0    ┆ 171.0 ┆ 626.0 ┆ 1433.0 ┆ 1945 ┆ 0.0383    │
└──────────┴────────┴───────┴───────┴────────┴──────┴───────────┘

Positive edges:

In [59]:
# ==================================================================================================
# AMLC 2026 — CELL 57
# TRUE POSITIVE FAILURE ANALYSIS — FINAL ROBUST VERSION
#
# IMPORTANT:
#   We derive source blocking keys ONLY for the ground-truth positive pairs.
#   We do NOT rebuild name_ascii for all 10.3M source rows.
#
# Purpose:
#   Understand why exact-name + first-last blocking only retrieves ~50.7%
#   of the true positive edges.
# ==================================================================================================

import os
import re
import time
import numpy as np
import polars as pl

from rapidfuzz import fuzz
from anyascii import anyascii


print("=" * 100)
print("AMLC 2026 — TRUE POSITIVE FAILURE ANALYSIS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# HELPERS
# --------------------------------------------------------------------------------------------------

def eager(x):
    return x.collect() if isinstance(x, pl.LazyFrame) else x


def make_block_keys_from_name_norm(df, id_col="entity_id"):
    """
    Build the EXACT SAME key family used by the blocker:
        name_ascii
        first_token
        last_token
        first_last

    Input must contain:
        id_col
        name_norm
    """

    df = eager(df)

    ascii_values = [
        anyascii(x) if x else ""
        for x in df["name_norm"].to_list()
    ]

    out = df.with_columns(
        pl.Series(
            "name_ascii",
            ascii_values,
            dtype=pl.String,
        )
    )

    out = (
        out
        .with_columns([
            pl.col("name_ascii")
            .str.extract(r"^(\S+)", 1)
            .fill_null("")
            .alias("first_token"),

            pl.col("name_ascii")
            .str.extract(r"(\S+)$", 1)
            .fill_null("")
            .alias("last_token"),
        ])
        .with_columns(
            (
                pl.col("first_token")
                + pl.lit("\x1f")
                + pl.col("last_token")
            )
            .alias("first_last")
        )
    )

    return out


def resolve_col(df, candidates, label):
    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(
        f"Could not find {label}. "
        f"Tried {candidates}; available={df.columns}"
    )


def normalize_address_py(s):
    if s is None:
        return ""

    s = str(s).lower()
    s = s.replace("&", " and ")

    s = "".join(
        ch if (ch.isalnum() or ch.isspace()) else " "
        for ch in s
    )

    s = re.sub(r"\s+", " ", s).strip()

    return s


# --------------------------------------------------------------------------------------------------
# PATHS
# --------------------------------------------------------------------------------------------------

DRIVE_PROJECT = "/content/drive/MyDrive/AMLC2026"

TRAIN_DIR = os.path.join(
    DRIVE_PROJECT,
    "dataset",
    "train",
)

S2_RAW = os.path.join(
    TRAIN_DIR,
    "train_source2.tsv",
)

S3_RAW = os.path.join(
    TRAIN_DIR,
    "train_source3.tsv",
)

S1_EVAL_PATH = os.path.join(
    DRIVE_PROJECT,
    "cache",
    "state",
    "er_eda",
    "s1_eval.parquet",
)

EDGE_EVAL_PATH = os.path.join(
    DRIVE_PROJECT,
    "cache",
    "state",
    "er_eda",
    "edge_eval.parquet",
)

DIAG_OUT = os.path.join(
    DRIVE_PROJECT,
    "cache",
    "state",
    "er_eda",
    "positive_pair_signal_diagnostics.parquet",
)

os.makedirs(
    os.path.dirname(DIAG_OUT),
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# MATERIALIZE ONLY WHAT WE NEED
# --------------------------------------------------------------------------------------------------

print("\nMaterializing working tables...")

if "s1_idx" in globals():
    s1_idx_e = eager(s1_idx)
else:
    raise RuntimeError("s1_idx is missing.")

if "s2_idx" in globals():
    s2_idx_e = eager(s2_idx)
else:
    raise RuntimeError("s2_idx is missing.")

if "s3_idx" in globals():
    s3_idx_e = eager(s3_idx)
else:
    raise RuntimeError("s3_idx is missing.")


print(
    f"S1 working index : {s1_idx_e.height:,}"
)

print(
    f"S2 working index : {s2_idx_e.height:,}"
)

print(
    f"S3 working index : {s3_idx_e.height:,}"
)


# --------------------------------------------------------------------------------------------------
# LOAD POSITIVE EDGES
# --------------------------------------------------------------------------------------------------

print("\nLoading positive edges...")

if "edge_eval" in globals():

    gt = eager(edge_eval)

else:

    gt = pl.read_parquet(
        EDGE_EVAL_PATH
    )


gt = (
    gt
    .select([
        "source1_entity_id",
        "matched_entity_id",
        "matched_source",
    ])
    .rename({
        "source1_entity_id": "s1_entity_id",
        "matched_entity_id": "candidate_entity_id",
        "matched_source": "true_source",
    })
    .unique()
)


print(
    f"Positive edges: {gt.height:,}"
)


# --------------------------------------------------------------------------------------------------
# BUILD S1 KEY TABLE
# --------------------------------------------------------------------------------------------------

print("\nPreparing S1 blocking keys...")

# s1_idx may already contain name_ascii/first_last.
if {
    "name_ascii",
    "first_last",
}.issubset(set(s1_idx_e.columns)):

    s1_keys = (
        s1_idx_e
        .select([
            "s1_entity_id",
            "country",
            "name_ascii",
            "first_last",
        ])
        .rename({
            "name_ascii": "s1_name_ascii",
            "first_last": "s1_first_last",
            "country": "s1_country",
        })
    )

elif {
    "entity_id",
    "name_norm",
}.issubset(set(s1_idx_e.columns)):

    s1_tmp = (
        s1_idx_e
        .select([
            "entity_id",
            "country",
            "name_norm",
        ])
    )

    s1_tmp = make_block_keys_from_name_norm(
        s1_tmp,
        id_col="entity_id",
    )

    s1_keys = (
        s1_tmp
        .select([
            pl.col("entity_id").alias("s1_entity_id"),
            pl.col("country").alias("s1_country"),
            pl.col("name_ascii").alias("s1_name_ascii"),
            pl.col("first_last").alias("s1_first_last"),
        ])
    )

else:

    raise RuntimeError(
        "S1 working index has neither blocking keys nor name_norm."
    )


# --------------------------------------------------------------------------------------------------
# PREPARE ONLY TRUE SOURCE RECORDS FOR KEY ANALYSIS
# --------------------------------------------------------------------------------------------------

print("\nPreparing true source-side blocking keys...")

s2_positive_ids = (
    gt
    .filter(pl.col("true_source") == "S2")
    .select("candidate_entity_id")
    .unique()
)


s3_positive_ids = (
    gt
    .filter(pl.col("true_source") == "S3")
    .select("candidate_entity_id")
    .unique()
)


# S2 source index currently contains name_norm.
s2_true = (
    s2_positive_ids
    .join(
        s2_idx_e.select([
            "entity_id",
            "country",
            "name_norm",
        ]),
        left_on="candidate_entity_id",
        right_on="entity_id",
        how="left",
    )
    .with_columns(
        pl.lit("S2").alias("true_source")
    )
)


s2_true = make_block_keys_from_name_norm(
    s2_true.select([
        "candidate_entity_id",
        "country",
        "name_norm",
        "true_source",
    ])
    .rename({
        "candidate_entity_id": "entity_id"
    }),
    id_col="entity_id",
)


s2_true = (
    s2_true
    .select([
        pl.col("entity_id").alias("candidate_entity_id"),
        pl.col("country").alias("source_country"),
        pl.col("name_ascii").alias("source_name_ascii"),
        pl.col("first_last").alias("source_first_last"),
        "true_source",
    ])
)


# S3
s3_true = (
    s3_positive_ids
    .join(
        s3_idx_e.select([
            "entity_id",
            "country",
            "name_norm",
        ]),
        left_on="candidate_entity_id",
        right_on="entity_id",
        how="left",
    )
    .with_columns(
        pl.lit("S3").alias("true_source")
    )
)


s3_true = make_block_keys_from_name_norm(
    s3_true.select([
        "candidate_entity_id",
        "country",
        "name_norm",
        "true_source",
    ])
    .rename({
        "candidate_entity_id": "entity_id"
    }),
    id_col="entity_id",
)


s3_true = (
    s3_true
    .select([
        pl.col("entity_id").alias("candidate_entity_id"),
        pl.col("country").alias("source_country"),
        pl.col("name_ascii").alias("source_name_ascii"),
        pl.col("first_last").alias("source_first_last"),
        "true_source",
    ])
)


source_true_keys = pl.concat(
    [
        s2_true,
        s3_true,
    ],
    how="vertical",
).unique([
    "candidate_entity_id",
    "true_source",
])


print(
    f"True source records keyed: "
    f"{source_true_keys.height:,}"
)


# --------------------------------------------------------------------------------------------------
# JOIN TRUE POSITIVE PAIRS TO KEYS
# --------------------------------------------------------------------------------------------------

print("\nJoining true positives to their actual blocker keys...")

diag = (
    gt
    .join(
        s1_keys,
        on="s1_entity_id",
        how="left",
    )
    .join(
        source_true_keys,
        on=[
            "candidate_entity_id",
            "true_source",
        ],
        how="left",
    )
    .with_columns([
        (
            pl.col("s1_name_ascii")
            == pl.col("source_name_ascii")
        )
        .alias("name_ascii_exact"),

        (
            pl.col("s1_first_last")
            == pl.col("source_first_last")
        )
        .alias("first_last_exact"),

        (
            (
                pl.col("s1_name_ascii")
                == pl.col("source_name_ascii")
            )
            |
            (
                pl.col("s1_first_last")
                == pl.col("source_first_last")
            )
        )
        .alias("either_current_key"),

        (
            pl.col("s1_country")
            == pl.col("source_country")
        )
        .alias("country_equal"),
    ])
)


# --------------------------------------------------------------------------------------------------
# CURRENT KEY RECALL
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("ACTUAL TRUE-PAIR KEY AGREEMENT")
print("=" * 100)

total = diag.height

exact_count = (
    diag
    .filter(pl.col("name_ascii_exact"))
    .height
)

first_last_count = (
    diag
    .filter(pl.col("first_last_exact"))
    .height
)

either_count = (
    diag
    .filter(pl.col("either_current_key"))
    .height
)

country_count = (
    diag
    .filter(pl.col("country_equal"))
    .height
)

print(
    f"Exact ASCII name agreement : "
    f"{exact_count:,} / {total:,} = {exact_count / total:.4%}"
)

print(
    f"First+last agreement       : "
    f"{first_last_count:,} / {total:,} = {first_last_count / total:.4%}"
)

print(
    f"Either current key         : "
    f"{either_count:,} / {total:,} = {either_count / total:.4%}"
)

print(
    f"Country agreement          : "
    f"{country_count:,} / {total:,} = {country_count / total:.4%}"
)


# --------------------------------------------------------------------------------------------------
# TRUE RESIDUAL
# --------------------------------------------------------------------------------------------------

residual = (
    diag
    .filter(~pl.col("either_current_key"))
)


print("\n" + "=" * 100)
print("TRUE POSITIVES MISSED BY BOTH CURRENT BLOCKERS")
print("=" * 100)

print(
    f"Residual positive edges: "
    f"{residual.height:,}"
)

print(
    f"Residual rate: "
    f"{residual.height / total:.4%}"
)


# --------------------------------------------------------------------------------------------------
# LOAD RAW S1 RECORDS
# --------------------------------------------------------------------------------------------------

print("\nLoading raw S1 evaluation records...")

if "s1_eval" in globals():

    s1_eval_raw = eager(s1_eval)

elif "s1_base" in globals():

    s1_eval_raw = eager(s1_base)

elif os.path.exists(S1_EVAL_PATH):

    s1_eval_raw = pl.read_parquet(
        S1_EVAL_PATH
    )

else:

    raise FileNotFoundError(
        "Could not find s1 evaluation data."
    )


s1_name_col = resolve_col(
    s1_eval_raw,
    [
        "business_name",
        "name",
        "business",
    ],
    "S1 business name",
)


s1_address_col = None

for c in [
    "business_address",
    "address",
]:
    if c in s1_eval_raw.columns:
        s1_address_col = c
        break


if s1_address_col is None:

    s1_raw = (
        s1_eval_raw
        .select([
            pl.col("entity_id")
            .alias("s1_entity_id"),

            pl.col(s1_name_col)
            .cast(pl.String)
            .fill_null("")
            .alias("s1_business_name"),

            pl.lit("")
            .alias("s1_business_address"),
        ])
    )

else:

    s1_raw = (
        s1_eval_raw
        .select([
            pl.col("entity_id")
            .alias("s1_entity_id"),

            pl.col(s1_name_col)
            .cast(pl.String)
            .fill_null("")
            .alias("s1_business_name"),

            pl.col(s1_address_col)
            .cast(pl.String)
            .fill_null("")
            .alias("s1_business_address"),
        ])
    )


# --------------------------------------------------------------------------------------------------
# RAW SOURCE RECORD LOADER
# --------------------------------------------------------------------------------------------------

def load_true_source_raw(
    path,
    ids_df,
    source,
):

    scan = pl.scan_csv(
        path,
        separator="\t",
        has_header=True,
        infer_schema_length=10000,
    )

    cols = scan.collect_schema().names()

    entity_col = (
        "entity_id"
        if "entity_id" in cols
        else " entity_id"
    )

    name_col = resolve_col(
        pl.DataFrame(
            schema={
                c: pl.String
                for c in cols
            }
        ),
        [
            "business_name",
            "name",
            "business",
        ],
        f"{source} business name",
    )

    address_col = None

    for c in [
        "business_address",
        "address",
    ]:
        if c in cols:
            address_col = c
            break


    id_list = (
        ids_df
        .get_column("candidate_entity_id")
        .to_list()
    )


    if address_col is None:

        address_expr = (
            pl.lit("")
            .alias("source_business_address")
        )

    else:

        address_expr = (
            pl.col(address_col)
            .cast(pl.String)
            .fill_null("")
            .alias("source_business_address")
        )


    return (
        scan
        .filter(
            pl.col(entity_col)
            .is_in(id_list)
        )
        .select([
            pl.col(entity_col)
            .alias("candidate_entity_id"),

            pl.col(name_col)
            .cast(pl.String)
            .fill_null("")
            .alias("source_business_name"),

            address_expr,

            pl.lit(source)
            .alias("true_source"),
        ])
        .collect(engine="streaming")
    )


# --------------------------------------------------------------------------------------------------
# LOAD TRUE SOURCE RAW TEXT
# --------------------------------------------------------------------------------------------------

print("\nLoading raw S2 matched records...")

t0 = time.time()

s2_true_raw = load_true_source_raw(
    S2_RAW,
    s2_positive_ids,
    "S2",
)

print(
    f"S2 raw matched rows: "
    f"{s2_true_raw.height:,} "
    f"in {time.time() - t0:.1f}s"
)


print("\nLoading raw S3 matched records...")

t0 = time.time()

s3_true_raw = load_true_source_raw(
    S3_RAW,
    s3_positive_ids,
    "S3",
)

print(
    f"S3 raw matched rows: "
    f"{s3_true_raw.height:,} "
    f"in {time.time() - t0:.1f}s"
)


source_true_raw = pl.concat(
    [
        s2_true_raw,
        s3_true_raw,
    ],
    how="vertical",
).unique([
    "candidate_entity_id",
    "true_source",
])


# --------------------------------------------------------------------------------------------------
# ATTACH RAW TEXT
# --------------------------------------------------------------------------------------------------

diag_raw = (
    diag
    .join(
        s1_raw,
        on="s1_entity_id",
        how="left",
    )
    .join(
        source_true_raw,
        on=[
            "candidate_entity_id",
            "true_source",
        ],
        how="left",
    )
)


# --------------------------------------------------------------------------------------------------
# FUZZY NAME FEATURES
# --------------------------------------------------------------------------------------------------

print("\nComputing name similarity features...")

t0 = time.time()


def safe_fuzz_ratio(x):

    a = x["s1_business_name"]
    b = x["source_business_name"]

    if not a or not b:
        return 0.0

    return float(
        fuzz.ratio(
            anyascii(a).lower(),
            anyascii(b).lower(),
        )
    )


def safe_token_set_ratio(x):

    a = x["s1_business_name"]
    b = x["source_business_name"]

    if not a or not b:
        return 0.0

    return float(
        fuzz.token_set_ratio(
            anyascii(a).lower(),
            anyascii(b).lower(),
        )
    )


def safe_token_sort_ratio(x):

    a = x["s1_business_name"]
    b = x["source_business_name"]

    if not a or not b:
        return 0.0

    return float(
        fuzz.token_sort_ratio(
            anyascii(a).lower(),
            anyascii(b).lower(),
        )
    )


diag_raw = (
    diag_raw
    .with_columns([
        pl.struct([
            "s1_business_name",
            "source_business_name",
        ])
        .map_elements(
            safe_fuzz_ratio,
            return_dtype=pl.Float32,
        )
        .alias("name_fuzz_ratio"),

        pl.struct([
            "s1_business_name",
            "source_business_name",
        ])
        .map_elements(
            safe_token_set_ratio,
            return_dtype=pl.Float32,
        )
        .alias("name_token_set_ratio"),

        pl.struct([
            "s1_business_name",
            "source_business_name",
        ])
        .map_elements(
            safe_token_sort_ratio,
            return_dtype=pl.Float32,
        )
        .alias("name_token_sort_ratio"),
    ])
)


print(
    f"Name features computed in "
    f"{time.time() - t0:.1f}s"
)


# --------------------------------------------------------------------------------------------------
# ADDRESS FEATURES
# --------------------------------------------------------------------------------------------------

print("\nComputing address similarity features...")


def address_token_jaccard(x):

    a = normalize_address_py(
        x["s1_business_address"]
    )

    b = normalize_address_py(
        x["source_business_address"]
    )

    if not a or not b:
        return 0.0

    A = set(a.split())
    B = set(b.split())

    if not A or not B:
        return 0.0

    return float(
        len(A & B) /
        len(A | B)
    )


def address_fuzz_ratio(x):

    a = normalize_address_py(
        x["s1_business_address"]
    )

    b = normalize_address_py(
        x["source_business_address"]
    )

    if not a or not b:
        return 0.0

    return float(
        fuzz.ratio(
            a,
            b,
        )
    )


def address_digit_overlap(x):

    a = normalize_address_py(
        x["s1_business_address"]
    )

    b = normalize_address_py(
        x["source_business_address"]
    )

    nums_a = set(
        re.findall(
            r"\d+",
            a,
        )
    )

    nums_b = set(
        re.findall(
            r"\d+",
            b,
        )
    )

    if not nums_a or not nums_b:
        return 0.0

    return float(
        len(nums_a & nums_b) /
        len(nums_a | nums_b)
    )


diag_raw = (
    diag_raw
    .with_columns([
        pl.struct([
            "s1_business_address",
            "source_business_address",
        ])
        .map_elements(
            address_token_jaccard,
            return_dtype=pl.Float32,
        )
        .alias("address_token_jaccard"),

        pl.struct([
            "s1_business_address",
            "source_business_address",
        ])
        .map_elements(
            address_fuzz_ratio,
            return_dtype=pl.Float32,
        )
        .alias("address_fuzz_ratio"),

        pl.struct([
            "s1_business_address",
            "source_business_address",
        ])
        .map_elements(
            address_digit_overlap,
            return_dtype=pl.Float32,
        )
        .alias("address_digit_overlap"),
    ])
)


print("Address features computed.")


# --------------------------------------------------------------------------------------------------
# ALL-POSITIVE SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("POSITIVE-PAIR SIGNAL SUMMARY")
print("=" * 100)


summary = (
    diag_raw
    .select([
        pl.len()
        .alias("positive_edges"),

        (
            pl.col("name_ascii_exact")
            .cast(pl.Float64)
            .mean()
        )
        .alias("exact_name_rate"),

        (
            pl.col("first_last_exact")
            .cast(pl.Float64)
            .mean()
        )
        .alias("first_last_rate"),

        (
            pl.col("either_current_key")
            .cast(pl.Float64)
            .mean()
        )
        .alias("current_block_rate"),

        (
            (pl.col("name_fuzz_ratio") >= 95)
            .cast(pl.Float64)
            .mean()
        )
        .alias("name_fuzz_ge95"),

        (
            (pl.col("name_fuzz_ratio") >= 90)
            .cast(pl.Float64)
            .mean()
        )
        .alias("name_fuzz_ge90"),

        (
            (pl.col("name_fuzz_ratio") >= 80)
            .cast(pl.Float64)
            .mean()
        )
        .alias("name_fuzz_ge80"),

        (
            (pl.col("name_token_set_ratio") >= 90)
            .cast(pl.Float64)
            .mean()
        )
        .alias("token_set_ge90"),

        (
            (pl.col("address_token_jaccard") >= 0.50)
            .cast(pl.Float64)
            .mean()
        )
        .alias("address_jaccard_ge50"),

        (
            (pl.col("address_token_jaccard") >= 0.25)
            .cast(pl.Float64)
            .mean()
        )
        .alias("address_jaccard_ge25"),

        (
            (pl.col("address_fuzz_ratio") >= 80)
            .cast(pl.Float64)
            .mean()
        )
        .alias("address_fuzz_ge80"),

        (
            (pl.col("address_digit_overlap") > 0)
            .cast(pl.Float64)
            .mean()
        )
        .alias("address_digit_overlap_gt0"),
    ])
)

print(summary)


# --------------------------------------------------------------------------------------------------
# NAME QUANTILES
# --------------------------------------------------------------------------------------------------

print("\nName similarity quantiles:")

name_quantiles = (
    diag_raw
    .select([
        pl.col("name_fuzz_ratio")
        .quantile(0.01)
        .alias("p01"),

        pl.col("name_fuzz_ratio")
        .quantile(0.05)
        .alias("p05"),

        pl.col("name_fuzz_ratio")
        .quantile(0.10)
        .alias("p10"),

        pl.col("name_fuzz_ratio")
        .quantile(0.25)
        .alias("p25"),

        pl.col("name_fuzz_ratio")
        .median()
        .alias("p50"),

        pl.col("name_fuzz_ratio")
        .quantile(0.75)
        .alias("p75"),

        pl.col("name_fuzz_ratio")
        .quantile(0.90)
        .alias("p90"),

        pl.col("name_fuzz_ratio")
        .quantile(0.95)
        .alias("p95"),

        pl.col("name_fuzz_ratio")
        .quantile(0.99)
        .alias("p99"),
    ])
)

print(name_quantiles)


# --------------------------------------------------------------------------------------------------
# RESIDUAL SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("RESIDUAL POSITIVE SIGNALS")
print("=" * 100)


residual_raw = (
    diag_raw
    .filter(~pl.col("either_current_key"))
)


residual_summary = (
    residual_raw
    .select([
        pl.len()
        .alias("residual_edges"),

        (
            (pl.col("name_fuzz_ratio") >= 95)
            .cast(pl.Float64)
            .mean()
        )
        .alias("name_fuzz_ge95"),

        (
            (pl.col("name_fuzz_ratio") >= 90)
            .cast(pl.Float64)
            .mean()
        )
        .alias("name_fuzz_ge90"),

        (
            (pl.col("name_fuzz_ratio") >= 80)
            .cast(pl.Float64)
            .mean()
        )
        .alias("name_fuzz_ge80"),

        (
            (pl.col("name_token_set_ratio") >= 90)
            .cast(pl.Float64)
            .mean()
        )
        .alias("token_set_ge90"),

        (
            (pl.col("address_token_jaccard") >= 0.50)
            .cast(pl.Float64)
            .mean()
        )
        .alias("address_jaccard_ge50"),

        (
            (pl.col("address_token_jaccard") >= 0.25)
            .cast(pl.Float64)
            .mean()
        )
        .alias("address_jaccard_ge25"),

        (
            (pl.col("address_fuzz_ratio") >= 80)
            .cast(pl.Float64)
            .mean()
        )
        .alias("address_fuzz_ge80"),

        (
            (pl.col("address_digit_overlap") > 0)
            .cast(pl.Float64)
            .mean()
        )
        .alias("address_digit_overlap_gt0"),
    ])
)

print(residual_summary)


# --------------------------------------------------------------------------------------------------
# RESIDUAL NAME QUANTILES
# --------------------------------------------------------------------------------------------------

print("\nResidual name similarity quantiles:")

residual_name_quantiles = (
    residual_raw
    .select([
        pl.col("name_fuzz_ratio")
        .quantile(0.01)
        .alias("p01"),

        pl.col("name_fuzz_ratio")
        .quantile(0.05)
        .alias("p05"),

        pl.col("name_fuzz_ratio")
        .quantile(0.10)
        .alias("p10"),

        pl.col("name_fuzz_ratio")
        .quantile(0.25)
        .alias("p25"),

        pl.col("name_fuzz_ratio")
        .median()
        .alias("p50"),

        pl.col("name_fuzz_ratio")
        .quantile(0.75)
        .alias("p75"),

        pl.col("name_fuzz_ratio")
        .quantile(0.90)
        .alias("p90"),

        pl.col("name_fuzz_ratio")
        .quantile(0.95)
        .alias("p95"),

        pl.col("name_fuzz_ratio")
        .quantile(0.99)
        .alias("p99"),
    ])
)

print(residual_name_quantiles)


# --------------------------------------------------------------------------------------------------
# DIFFICULT EXAMPLES
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("DIFFICULT POSITIVE EXAMPLES")
print("=" * 100)


example_cols = [
    "s1_entity_id",
    "candidate_entity_id",
    "true_source",
    "s1_business_name",
    "source_business_name",
    "name_fuzz_ratio",
    "name_token_set_ratio",
    "name_token_sort_ratio",
    "s1_business_address",
    "source_business_address",
    "address_token_jaccard",
    "address_fuzz_ratio",
    "address_digit_overlap",
]


print(
    residual_raw
    .select(example_cols)
    .sort(
        "name_fuzz_ratio",
        descending=True,
    )
    .head(30)
)


# --------------------------------------------------------------------------------------------------
# SAVE
# --------------------------------------------------------------------------------------------------

print("\nSaving diagnostic artifact...")

diag_raw.write_parquet(
    DIAG_OUT,
    compression="zstd",
)

print(
    f"Saved: {DIAG_OUT}"
)


# --------------------------------------------------------------------------------------------------
# FINAL SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("CELL 57 COMPLETE")
print("=" * 100)

print(
    f"Positive edges analyzed : {total:,}"
)

print(
    f"Current-key coverage    : "
    f"{either_count / total:.4%}"
)

print(
    f"Residual positive edges : "
    f"{residual_raw.height:,}"
)

print("=" * 100)

AMLC 2026 — TRUE POSITIVE FAILURE ANALYSIS

Materializing working tables...
S1 working index : 100,000
S2 working index : 5,034,616
S3 working index : 5,285,603

Loading positive edges...
Positive edges: 345,980

Preparing S1 blocking keys...

Preparing true source-side blocking keys...
True source records keyed: 345,980

Joining true positives to their actual blocker keys...

ACTUAL TRUE-PAIR KEY AGREEMENT
Exact ASCII name agreement : 158,079 / 345,980 = 45.6902%
First+last agreement       : 175,552 / 345,980 = 50.7405%
Either current key         : 175,552 / 345,980 = 50.7405%
Country agreement          : 345,980 / 345,980 = 100.0000%

TRUE POSITIVES MISSED BY BOTH CURRENT BLOCKERS
Residual positive edges: 170,428
Residual rate: 49.2595%

Loading raw S1 evaluation records...

Loading raw S2 matched records...
S2 raw matched rows: 167,274 in 3.1s

Loading raw S3 matched records...
S3 raw matched rows: 178,706 in 4.2s

Computing name similarity features...
Name features computed in 3.1s

In [60]:
# ==================================================================================================
# AMLC 2026 — MASTER CHECKPOINT
#
# RUN THIS NOW BEFORE POWER / WIFI CUTS.
#
# After restart:
#   The ONLY cell you need to run is the launcher shown at the bottom of this cell.
#
# It will:
#   - mount Google Drive
#   - install required packages
#   - restore paths
#   - restore S1/S2/S3 normalized indexes
#   - restore blocking keys
#   - restore GT / evaluation artifacts
#   - restore candidate artifacts
#   - restore positive-pair diagnostics
#   - print a RESUME READY summary
# ==================================================================================================

import os
import sys
import json
import time
import subprocess
import importlib.metadata as importlib_metadata


print("=" * 110)
print("AMLC 2026 — MASTER CHECKPOINT")
print("=" * 110)


# --------------------------------------------------------------------------------------------------
# 1. GOOGLE DRIVE
# --------------------------------------------------------------------------------------------------

print("\n[1/8] Mounting Google Drive...")

from google.colab import drive

DRIVE_MOUNT = "/content/drive"

if not os.path.ismount(DRIVE_MOUNT):
    drive.mount(DRIVE_MOUNT)
else:
    print("Drive already mounted.")


# --------------------------------------------------------------------------------------------------
# 2. PROJECT DIRECTORIES
# --------------------------------------------------------------------------------------------------

DRIVE_PROJECT = "/content/drive/MyDrive/AMLC2026"

DATA_DIR = os.path.join(
    DRIVE_PROJECT,
    "dataset",
)

TRAIN_DIR = os.path.join(
    DATA_DIR,
    "train",
)

TEST_DIR = os.path.join(
    DATA_DIR,
    "test",
)

CACHE_DIR = os.path.join(
    DRIVE_PROJECT,
    "cache",
)

BLOCKING_DIR = os.path.join(
    CACHE_DIR,
    "blocking",
)

STATE_DIR = os.path.join(
    CACHE_DIR,
    "state",
    "er_eda",
)

CHECKPOINT_DIR = os.path.join(
    CACHE_DIR,
    "checkpoint_20260925",
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True,
)


print(
    "Project:",
    DRIVE_PROJECT,
)


# --------------------------------------------------------------------------------------------------
# 3. PACKAGE REQUIREMENTS
#
# We write the exact known-good versions we used.
# faiss is left unpinned because its pip wheel versioning can differ across Colab runtimes.
# --------------------------------------------------------------------------------------------------

print("\n[2/8] Recording package requirements...")


REQUIRED_PACKAGES = {
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "polars": "1.35.2",
    "duckdb": "1.3.2",
    "rapidfuzz": "3.14.6",
    "scikit-learn": "1.6.1",
    "lightgbm": "4.6.0",
    "xgboost": "3.4.1",
    "anyascii": "0.3.3",
    "pyarrow": None,
    "faiss-cpu": None,
}


# --------------------------------------------------------------------------------------------------
# 4. CURRENT ENVIRONMENT MANIFEST
# --------------------------------------------------------------------------------------------------

print("\n[3/8] Saving current environment manifest...")

current_versions = {}

for package_name in REQUIRED_PACKAGES:

    try:
        current_versions[package_name] = (
            importlib_metadata.version(package_name)
        )
    except Exception:
        current_versions[package_name] = "NOT_INSTALLED"


manifest = {
    "created": time.strftime(
        "%Y-%m-%d %H:%M:%S"
    ),

    "python": sys.version,

    "packages": current_versions,

    "project": DRIVE_PROJECT,

    "checkpoint_dir": CHECKPOINT_DIR,
}


manifest_path = os.path.join(
    CHECKPOINT_DIR,
    "environment_manifest.json",
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        default=str,
    )


print(
    json.dumps(
        current_versions,
        indent=2,
    )
)


# --------------------------------------------------------------------------------------------------
# 5. HELPER FUNCTIONS FOR SNAPSHOTS
# --------------------------------------------------------------------------------------------------

print("\n[4/8] Preparing checkpoint helpers...")


def eager(x):
    import polars as pl

    if isinstance(x, pl.LazyFrame):
        return x.collect()

    return x


def atomic_write_parquet(
    df,
    path,
):
    """
    Write to temporary Drive file first, then atomically replace.
    """

    df = eager(df)

    temp_path = path + ".tmp"

    if os.path.exists(temp_path):
        os.remove(temp_path)

    df.write_parquet(
        temp_path,
        compression="zstd",
    )

    os.replace(
        temp_path,
        path,
    )

    return path


def snapshot_if_exists(
    variable_name,
    output_name,
):
    """
    Snapshot an in-memory Polars table if available.
    """

    if variable_name not in globals():
        print(
            f"SKIP {variable_name}: not in memory"
        )
        return None

    obj = globals()[variable_name]

    try:
        obj = eager(obj)

        output_path = os.path.join(
            CHECKPOINT_DIR,
            output_name,
        )

        atomic_write_parquet(
            obj,
            output_path,
        )

        print(
            f"Saved {variable_name}: "
            f"{obj.height:,} rows -> {output_path}"
        )

        return output_path

    except Exception as e:

        print(
            f"WARNING: could not snapshot "
            f"{variable_name}: {type(e).__name__}: {e}"
        )

        return None


# --------------------------------------------------------------------------------------------------
# 6. SNAPSHOT CURRENT IN-MEMORY INDEX STATE
# --------------------------------------------------------------------------------------------------

print("\n[5/8] Snapshotting current working indexes...")

snapshot_if_exists(
    "s1_idx",
    "s1_idx_current.parquet",
)

snapshot_if_exists(
    "s2_idx",
    "s2_idx_current.parquet",
)

snapshot_if_exists(
    "s3_idx",
    "s3_idx_current.parquet",
)


# --------------------------------------------------------------------------------------------------
# 7. VERIFY / RECORD ALL IMPORTANT DRIVE ARTIFACTS
# --------------------------------------------------------------------------------------------------

print("\n[6/8] Verifying persisted artifacts...")


IMPORTANT_ARTIFACTS = [

    # EDA / GT
    (
        "gt.parquet",
        os.path.join(
            STATE_DIR,
            "gt.parquet",
        ),
    ),

    (
        "gt_edges.parquet",
        os.path.join(
            STATE_DIR,
            "gt_edges.parquet",
        ),
    ),

    (
        "s1_eval.parquet",
        os.path.join(
            STATE_DIR,
            "s1_eval.parquet",
        ),
    ),

    (
        "eval_gt.parquet",
        os.path.join(
            STATE_DIR,
            "eval_gt.parquet",
        ),
    ),

    (
        "edge_eval.parquet",
        os.path.join(
            STATE_DIR,
            "edge_eval.parquet",
        ),
    ),

    (
        "per_s1_recall.parquet",
        os.path.join(
            STATE_DIR,
            "per_s1_recall.parquet",
        ),
    ),

    # Blocking diagnostics
    (
        "blocker_volume_audit.parquet",
        os.path.join(
            STATE_DIR,
            "blocker_volume_audit.parquet",
        ),
    ),

    (
        "frequency_capped_blocker_oracle.parquet",
        os.path.join(
            STATE_DIR,
            "frequency_capped_blocker_oracle.parquet",
        ),
    ),

    # Candidate artifacts
    (
        "eval_candidates_tier12.parquet",
        os.path.join(
            STATE_DIR,
            "eval_candidates_tier12.parquet",
        ),
    ),

    (
        "eval_candidate_residual.parquet",
        os.path.join(
            STATE_DIR,
            "eval_candidate_residual.parquet",
        ),
    ),

    (
        "eval_candidate_counts_tier12.parquet",
        os.path.join(
            STATE_DIR,
            "eval_candidate_counts_tier12.parquet",
        ),

    ),

    (
        "eval_candidates_tier12_source_capped.parquet",
        os.path.join(
            STATE_DIR,
            "eval_candidates_tier12_source_capped.parquet",
        ),
    ),

    (
        "eval_candidate_residual_source_capped.parquet",
        os.path.join(
            STATE_DIR,
            "eval_candidate_residual_source_capped.parquet",
        ),
    ),

    (
        "eval_candidate_counts_source_capped.parquet",
        os.path.join(
            STATE_DIR,
            "eval_candidate_counts_source_capped.parquet",
        ),
    ),

    # Current diagnostic artifact
    (
        "positive_pair_signal_diagnostics.parquet",
        os.path.join(
            STATE_DIR,
            "positive_pair_signal_diagnostics.parquet",
        ),
    ),
]


artifact_manifest = []


for label, path in IMPORTANT_ARTIFACTS:

    exists = os.path.exists(path)

    size_mb = (
        os.path.getsize(path) / (1024 ** 2)
        if exists
        else None
    )

    artifact_manifest.append({
        "name": label,
        "path": path,
        "exists": exists,
        "size_mb": size_mb,
    })

    if exists:

        print(
            f"[OK]   {label:<55} "
            f"{size_mb:,.2f} MB"
        )

    else:

        print(
            f"[MISS] {label}"
        )


artifact_manifest_path = os.path.join(
    CHECKPOINT_DIR,
    "artifact_manifest.json",
)

with open(
    artifact_manifest_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        artifact_manifest,
        f,
        indent=2,
    )


# --------------------------------------------------------------------------------------------------
# 8. CREATE SELF-CONTAINED RESUME BOOTSTRAP
# --------------------------------------------------------------------------------------------------

print("\n[7/8] Creating ONE-CELL resume bootstrap...")


bootstrap_code = r'''
# ==================================================================================================
# AMLC 2026 — ONE-CELL RESUME BOOTSTRAP
#
# RUN ONLY THIS CELL AFTER THE RUNTIME / POWER / WIFI RESTART.
#
# It:
#   - mounts Drive
#   - installs dependencies
#   - restores project paths
#   - restores checkpointed state
#   - rebuilds blocking-key columns if necessary
#   - restores the positive-pair diagnostic artifact
#
# After this cell finishes, continue from the next analysis/modeling cell.
# ==================================================================================================

import os
import sys
import subprocess
import importlib.metadata as importlib_metadata


# --------------------------------------------------------------------------------------------------
# 1. INSTALL REQUIRED PACKAGES
# --------------------------------------------------------------------------------------------------

REQUIRED = [
    "numpy==2.1.3",
    "pandas==2.2.3",
    "polars==1.35.2",
    "duckdb==1.3.2",
    "rapidfuzz==3.14.6",
    "scikit-learn==1.6.1",
    "lightgbm==4.6.0",
    "xgboost==3.4.1",
    "anyascii==0.3.3",
    "pyarrow",
    "faiss-cpu",
]

print("=" * 110)
print("AMLC 2026 — RESUME BOOTSTRAP")
print("=" * 110)

print("\nInstalling / verifying required packages...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
    ] + REQUIRED,
    check=True,
)


# --------------------------------------------------------------------------------------------------
# 2. IMPORTS
# --------------------------------------------------------------------------------------------------

import os
import re
import json
import time

import numpy as np
import pandas as pd
import polars as pl
import duckdb

from rapidfuzz import fuzz
from anyascii import anyascii


# --------------------------------------------------------------------------------------------------
# 3. GOOGLE DRIVE
# --------------------------------------------------------------------------------------------------

print("\nMounting Google Drive...")

from google.colab import drive

DRIVE_MOUNT = "/content/drive"

if not os.path.ismount(DRIVE_MOUNT):
    drive.mount(DRIVE_MOUNT)
else:
    print("Drive already mounted.")


# --------------------------------------------------------------------------------------------------
# 4. PROJECT PATHS
# --------------------------------------------------------------------------------------------------

DRIVE_PROJECT = "/content/drive/MyDrive/AMLC2026"

DATA_DIR = os.path.join(
    DRIVE_PROJECT,
    "dataset",
)

TRAIN_DIR = os.path.join(
    DATA_DIR,
    "train",
)

TEST_DIR = os.path.join(
    DATA_DIR,
    "test",
)

CACHE_DIR = os.path.join(
    DRIVE_PROJECT,
    "cache",
)

BLOCKING_DIR = os.path.join(
    CACHE_DIR,
    "blocking",
)

STATE_DIR = os.path.join(
    CACHE_DIR,
    "state",
    "er_eda",
)

CHECKPOINT_DIR = os.path.join(
    CACHE_DIR,
    "checkpoint_20260925",
)


# --------------------------------------------------------------------------------------------------
# 5. HELPERS
# --------------------------------------------------------------------------------------------------

def eager(x):

    if isinstance(x, pl.LazyFrame):
        return x.collect()

    return x


LEGAL_SUFFIXES = [
    "private limited",
    "private",
    "pvt ltd",
    "pvt",
    "limited",
    "ltd",
    "corporation",
    "corp",
    "company",
    "co",
    "incorporated",
    "inc",
    "llc",
    "llp",
    "sarl",
    "sasu",
    "sas",
    "plc",
    "gmbh",
]


def normalize_name_python(s):

    if s is None:
        return ""

    s = str(s).lower()

    s = s.replace(
        "&",
        " and ",
    )

    s = "".join(
        ch if (
            ch.isalnum()
            or ch.isspace()
        )
        else " "
        for ch in s
    )

    s = re.sub(
        r"\s+",
        " ",
        s,
    ).strip()

    changed = True

    while changed and s:

        changed = False

        for suffix in LEGAL_SUFFIXES:

            if s == suffix:

                s = ""
                changed = True
                break

            if s.endswith(
                " " + suffix
            ):

                s = s[
                    :-(len(suffix) + 1)
                ].rstrip()

                changed = True
                break

    return s


def add_block_keys(df):

    df = eager(df)

    if "name_ascii" in df.columns and "first_last" in df.columns:
        return df

    ascii_values = [
        anyascii(x) if x else ""
        for x in df["name_norm"].to_list()
    ]

    df = df.with_columns(
        pl.Series(
            "name_ascii",
            ascii_values,
            dtype=pl.String,
        )
    )

    df = (
        df
        .with_columns([
            pl.col("name_ascii")
            .str.extract(
                r"^(\S+)",
                1,
            )
            .fill_null("")
            .alias("first_token"),

            pl.col("name_ascii")
            .str.extract(
                r"(\S+)$",
                1,
            )
            .fill_null("")
            .alias("last_token"),
        ])
        .with_columns(
            (
                pl.col("first_token")
                + pl.lit("\x1f")
                + pl.col("last_token")
            )
            .alias("first_last")
        )
    )

    return df


# --------------------------------------------------------------------------------------------------
# 6. LOAD S1 EVAL
# --------------------------------------------------------------------------------------------------

print("\nLoading S1 evaluation data...")

s1_eval_path = os.path.join(
    STATE_DIR,
    "s1_eval.parquet",
)

s1_eval = pl.read_parquet(
    s1_eval_path
)


# --------------------------------------------------------------------------------------------------
# 7. RESTORE S1 INDEX
# --------------------------------------------------------------------------------------------------

s1_checkpoint = os.path.join(
    CHECKPOINT_DIR,
    "s1_idx_current.parquet",
)

if os.path.exists(s1_checkpoint):

    s1_idx = pl.read_parquet(
        s1_checkpoint
    )

else:

    # Fallback reconstruction.
    name_col = None

    for c in [
        "business_name",
        "name",
        "business",
    ]:

        if c in s1_eval.columns:
            name_col = c
            break

    if name_col is None:
        raise RuntimeError(
            "Cannot identify S1 business-name column."
        )

    s1_idx = (
        s1_eval
        .select([
            pl.col("entity_id"),
            pl.col("country"),
            pl.col(name_col)
            .cast(pl.String)
            .fill_null("")
            .alias("_raw_name"),
        ])
        .with_columns(
            pl.col("_raw_name")
            .map_elements(
                normalize_name_python,
                return_dtype=pl.String,
            )
            .alias("name_norm")
        )
        .drop("_raw_name")
    )

    s1_idx = add_block_keys(
        s1_idx
    )

    s1_idx = s1_idx.rename({
        "entity_id": "s1_entity_id"
    })


# --------------------------------------------------------------------------------------------------
# 8. RESTORE S2 INDEX
# --------------------------------------------------------------------------------------------------

s2_checkpoint = os.path.join(
    CHECKPOINT_DIR,
    "s2_idx_current.parquet",
)

if os.path.exists(s2_checkpoint):

    s2_idx = pl.read_parquet(
        s2_checkpoint
    )

else:

    s2_norm = os.path.join(
        BLOCKING_DIR,
        "train_s2_name_norm.parquet",
    )

    if not os.path.exists(s2_norm):

        raise FileNotFoundError(
            "Missing S2 normalized index:\n"
            + s2_norm
        )

    s2_idx = pl.read_parquet(
        s2_norm
    )

s2_idx = add_block_keys(
    s2_idx
)


# --------------------------------------------------------------------------------------------------
# 9. RESTORE S3 INDEX
# --------------------------------------------------------------------------------------------------

s3_checkpoint = os.path.join(
    CHECKPOINT_DIR,
    "s3_idx_current.parquet",
)

if os.path.exists(s3_checkpoint):

    s3_idx = pl.read_parquet(
        s3_checkpoint
    )

else:

    s3_norm = os.path.join(
        BLOCKING_DIR,
        "train_s3_name_norm.parquet",
    )

    if not os.path.exists(s3_norm):

        raise FileNotFoundError(
            "Missing S3 normalized index:\n"
            + s3_norm
        )

    s3_idx = pl.read_parquet(
        s3_norm
    )

s3_idx = add_block_keys(
    s3_idx
)


# --------------------------------------------------------------------------------------------------
# 10. RESTORE CORE EVALUATION ARTIFACTS
# --------------------------------------------------------------------------------------------------

gt = pl.read_parquet(
    os.path.join(
        STATE_DIR,
        "gt.parquet",
    )
)

gt_edges = pl.read_parquet(
    os.path.join(
        STATE_DIR,
        "gt_edges.parquet",
    )
)

eval_gt = pl.read_parquet(
    os.path.join(
        STATE_DIR,
        "eval_gt.parquet",
    )
)

edge_eval = pl.read_parquet(
    os.path.join(
        STATE_DIR,
        "edge_eval.parquet",
    )
)

per_s1_recall = pl.read_parquet(
    os.path.join(
        STATE_DIR,
        "per_s1_recall.parquet",
    )
)


# --------------------------------------------------------------------------------------------------
# 11. RESTORE CANDIDATE ARTIFACTS
# --------------------------------------------------------------------------------------------------

candidate_source_capped_path = os.path.join(
    STATE_DIR,
    "eval_candidates_tier12_source_capped.parquet",
)

if os.path.exists(
    candidate_source_capped_path
):

    candidate_pairs = pl.read_parquet(
        candidate_source_capped_path
    )

else:

    candidate_pairs = pl.read_parquet(
        os.path.join(
            STATE_DIR,
            "eval_candidates_tier12.parquet",
        )
    )


candidate_counts_path = os.path.join(
    STATE_DIR,
    "eval_candidate_counts_source_capped.parquet",
)

if os.path.exists(candidate_counts_path):

    candidate_counts = pl.read_parquet(
        candidate_counts_path
    )

else:

    candidate_counts = pl.read_parquet(
        os.path.join(
            STATE_DIR,
            "eval_candidate_counts_tier12.parquet",
        )
    )


candidate_residual_path = os.path.join(
    STATE_DIR,
    "eval_candidate_residual_source_capped.parquet",
)

if os.path.exists(candidate_residual_path):

    eval_candidate_residual = pl.read_parquet(
        candidate_residual_path
    )

else:

    eval_candidate_residual = pl.read_parquet(
        os.path.join(
            STATE_DIR,
            "eval_candidate_residual.parquet",
        )
    )


# --------------------------------------------------------------------------------------------------
# 12. RESTORE POSITIVE-PAIR DIAGNOSTICS
# --------------------------------------------------------------------------------------------------

diagnostics_path = os.path.join(
    STATE_DIR,
    "positive_pair_signal_diagnostics.parquet",
)

if os.path.exists(diagnostics_path):

    positive_pair_diagnostics = pl.read_parquet(
        diagnostics_path
    )

else:

    positive_pair_diagnostics = None


# --------------------------------------------------------------------------------------------------
# 13. RESTORE MANIFEST
# --------------------------------------------------------------------------------------------------

manifest_path = os.path.join(
    CHECKPOINT_DIR,
    "artifact_manifest.json",
)

if os.path.exists(manifest_path):

    with open(
        manifest_path,
        "r",
        encoding="utf-8",
    ) as f:

        checkpoint_manifest = json.load(f)

else:

    checkpoint_manifest = {}


# --------------------------------------------------------------------------------------------------
# 14. SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("AMLC 2026 — RESUME READY")
print("=" * 110)

print(
    f"S1 eval rows             : {s1_eval.height:,}"
)

print(
    f"S1 index rows            : {s1_idx.height:,}"
)

print(
    f"S2 index rows            : {s2_idx.height:,}"
)

print(
    f"S3 index rows            : {s3_idx.height:,}"
)

print(
    f"Positive eval edges      : {edge_eval.height:,}"
)

print(
    f"Candidate pairs loaded   : {candidate_pairs.height:,}"
)

print(
    f"Candidate S1 rows       : {candidate_counts.height:,}"
)

print(
    f"Candidate residuals     : {eval_candidate_residual.height:,}"
)

if positive_pair_diagnostics is not None:

    print(
        f"Positive diagnostics    : "
        f"{positive_pair_diagnostics.height:,}"
    )

else:

    print(
        "Positive diagnostics    : NOT FOUND"
    )


print("\nAvailable resume variables:")

print(
    """
s1_eval
s1_idx
s2_idx
s3_idx
gt
gt_edges
eval_gt
edge_eval
per_s1_recall
candidate_pairs
candidate_counts
eval_candidate_residual
positive_pair_diagnostics
"""
)

print(
    "\nCheckpoint is restored."
)

print(
    "Next stage: rescue-blocker design / residual retrieval."
)

print("=" * 110)
'''


bootstrap_path = os.path.join(
    CHECKPOINT_DIR,
    "AMLC2026_RESUME_BOOTSTRAP.py",
)


with open(
    bootstrap_path,
    "w",
    encoding="utf-8",
) as f:

    f.write(
        bootstrap_code
    )


# --------------------------------------------------------------------------------------------------
# 9. CREATE HUMAN-READABLE CHECKPOINT README
# --------------------------------------------------------------------------------------------------

print("\n[8/8] Writing checkpoint README...")


resume_readme = f"""
# AMLC 2026 — CHECKPOINT 2026-09-25

## Current stage

True-positive failure analysis completed.

Latest validated facts:

- Evaluation S1 rows: 100,000
- Positive evaluation edges: 345,980
- Current exact/first-last blocker coverage: 50.7405%
- Residual positive edges after current blockers: 170,428
- Positive-pair diagnostic artifact saved to:
  {os.path.join(STATE_DIR, "positive_pair_signal_diagnostics.parquet")}

## Important conclusion

The original combined S2+S3 frequency-cap assumption was wrong.
Source-specific frequency caps increased candidate volume but did NOT
solve the fundamental retrieval problem.

Current blockers cover only about half of true positive edges.

The next task is targeted rescue retrieval / blocking based on the
positive-pair diagnostics.

DO NOT start final pairwise ML until candidate recall has been improved.

## Resume

After restarting Colab, the only bootstrap needed is:

    from google.colab import drive
    drive.mount('/content/drive')
    exec(open(
        '/content/drive/MyDrive/AMLC2026/cache/checkpoint_20260925/AMLC2026_RESUME_BOOTSTRAP.py',
        'r',
        encoding='utf-8'
    ).read())

The bootstrap installs dependencies, restores Drive, paths, normalized
indexes, evaluation artifacts, candidate artifacts and the positive-pair
diagnostic table.

## Main diagnostic artifact

    {os.path.join(STATE_DIR, "positive_pair_signal_diagnostics.parquet")}

## Source-capped candidate artifact

    {os.path.join(STATE_DIR, "eval_candidates_tier12_source_capped.parquet")}

## Environment

See:

    environment_manifest.json

and:

    artifact_manifest.json
"""


readme_path = os.path.join(
    CHECKPOINT_DIR,
    "README_RESUME.md",
)

with open(
    readme_path,
    "w",
    encoding="utf-8",
) as f:

    f.write(
        resume_readme.strip()
        + "\n"
    )


# --------------------------------------------------------------------------------------------------
# 10. FINAL CHECKPOINT SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("CHECKPOINT COMPLETE")
print("=" * 110)

print(
    "\nCheckpoint directory:"
)

print(
    CHECKPOINT_DIR
)

print(
    "\nResume bootstrap:"
)

print(
    bootstrap_path
)

print(
    "\nEnvironment manifest:"
)

print(
    manifest_path
)

print(
    "\nArtifact manifest:"
)

print(
    artifact_manifest_path
)

print(
    "\nResume README:"
)

print(
    readme_path
)

print(
    "\n"
    "AFTER POWER RETURNS, THE ONLY CELL YOU NEED TO RUN IS:"
)

print(
    """
from google.colab import drive
drive.mount('/content/drive')
exec(open(
    '/content/drive/MyDrive/AMLC2026/cache/checkpoint_20260925/AMLC2026_RESUME_BOOTSTRAP.py',
    'r',
    encoding='utf-8'
).read())
"""
)

print("=" * 110)

AMLC 2026 — MASTER CHECKPOINT

[1/8] Mounting Google Drive...
Drive already mounted.
Project: /content/drive/MyDrive/AMLC2026

[2/8] Recording package requirements...

[3/8] Saving current environment manifest...
{
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "polars": "1.35.2",
  "duckdb": "1.3.2",
  "rapidfuzz": "3.14.6",
  "scikit-learn": "1.6.1",
  "lightgbm": "4.6.0",
  "xgboost": "3.4.1",
  "anyascii": "0.3.3",
  "pyarrow": "23.0.1",
  "faiss-cpu": "1.15.1"
}

[4/8] Preparing checkpoint helpers...

[5/8] Snapshotting current working indexes...
Saved s1_idx: 100,000 rows -> /content/drive/MyDrive/AMLC2026/cache/checkpoint_20260925/s1_idx_current.parquet
Saved s2_idx: 5,034,616 rows -> /content/drive/MyDrive/AMLC2026/cache/checkpoint_20260925/s2_idx_current.parquet
Saved s3_idx: 5,285,603 rows -> /content/drive/MyDrive/AMLC2026/cache/checkpoint_20260925/s3_idx_current.parquet

[6/8] Verifying persisted artifacts...
[OK]   gt.parquet                                              55.01 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
exec(open(
    '/content/drive/MyDrive/AMLC2026/cache/checkpoint_20260925/AMLC2026_RESUME_BOOTSTRAP.py',
    'r',
    encoding='utf-8'
).read())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
AMLC 2026 — RESUME BOOTSTRAP

Installing / verifying required packages...

Mounting Google Drive...
Drive already mounted.

Loading S1 evaluation data...


In [ ]:
# ==================================================================================================
# AMLC 2026 — CELL 58 FIX / CONTINUATION
#
# The original Cell 58 already successfully loaded:
#   s1_addr
#   s1_numeric
#   s2_addr
#   s3_addr
#   candidate_pairs
#
# The crash happened because S1 did not carry country into the address join.
#
# FIX:
#   - attach country to S1 address keys
#   - join EXACT ADDRESS on (country, address_norm)
#   - join NUMERIC ADDRESS on (country, numeric_token)
#
# DO NOT rerun the original Cell 58.
# ==================================================================================================

import os
import time
import gc
import polars as pl

print("=" * 100)
print("AMLC 2026 — CELL 58 FIX / CONTINUATION")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# CONFIG
# --------------------------------------------------------------------------------------------------

DRIVE_PROJECT = "/content/drive/MyDrive/AMLC2026"

STATE_DIR = os.path.join(
    DRIVE_PROJECT,
    "cache",
    "state",
    "er_eda",
)

ADDRESS_EXACT_CAP = 500
ADDRESS_NUMERIC_CAP = 500

ADDRESS_CANDIDATE_OUT = os.path.join(
    STATE_DIR,
    "eval_candidates_address_rescue.parquet",
)

ADDRESS_COUNTS_OUT = os.path.join(
    STATE_DIR,
    "eval_candidate_counts_address_rescue.parquet",
)

ADDRESS_RESIDUAL_OUT = os.path.join(
    STATE_DIR,
    "eval_candidate_residual_address_rescue.parquet",
)


# --------------------------------------------------------------------------------------------------
# VERIFY STATE FROM THE CRASHED CELL
# --------------------------------------------------------------------------------------------------

required_vars = [
    "s1_addr",
    "s2_addr",
    "s3_addr",
    "candidate_pairs",
]

missing = [
    x for x in required_vars
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "The original Cell 58 state is missing: "
        + ", ".join(missing)
        + "\n"
        "In that case, rerun the original Cell 58 after the RAM-cleanup cell."
    )


s1_addr = (
    s1_addr.collect()
    if isinstance(s1_addr, pl.LazyFrame)
    else s1_addr
)

s2_addr = (
    s2_addr.collect()
    if isinstance(s2_addr, pl.LazyFrame)
    else s2_addr
)

s3_addr = (
    s3_addr.collect()
    if isinstance(s3_addr, pl.LazyFrame)
    else s3_addr
)

candidate_pairs = (
    candidate_pairs.collect()
    if isinstance(candidate_pairs, pl.LazyFrame)
    else candidate_pairs
)


print(
    f"Existing name candidates: "
    f"{candidate_pairs.height:,}"
)

print(
    f"S1 address rows: {s1_addr.height:,}"
)

print(
    f"S2 address rows: {s2_addr.height:,}"
)

print(
    f"S3 address rows: {s3_addr.height:,}"
)


# --------------------------------------------------------------------------------------------------
# ATTACH S1 COUNTRY
# --------------------------------------------------------------------------------------------------

print("\nAttaching country to S1 address keys...")

if "s1_eval" in globals():

    s1_eval_local = (
        s1_eval.collect()
        if isinstance(s1_eval, pl.LazyFrame)
        else s1_eval
    )

elif "s1_base" in globals():

    s1_eval_local = (
        s1_base.collect()
        if isinstance(s1_base, pl.LazyFrame)
        else s1_base
    )

else:

    s1_eval_local = pl.read_parquet(
        os.path.join(
            STATE_DIR,
            "s1_eval.parquet",
        )
    )


s1_country = (
    s1_eval_local
    .select([
        pl.col("entity_id")
        .alias("s1_entity_id"),

        pl.col("country")
        .cast(pl.String)
        .fill_null("")
        .alias("country"),
    ])
    .unique(
        subset=["s1_entity_id"]
    )
)


s1_addr_fixed = (
    s1_addr
    .drop(
        "country",
        strict=False,
    )
    .join(
        s1_country,
        on="s1_entity_id",
        how="left",
    )
)


print(
    f"S1 address rows with country: "
    f"{s1_addr_fixed.height:,}"
)


# --------------------------------------------------------------------------------------------------
# REBUILD S1 NUMERIC KEYS WITH COUNTRY
# --------------------------------------------------------------------------------------------------

s1_numeric_fixed = (
    s1_addr_fixed
    .select([
        "s1_entity_id",
        "country",
        "numeric_tokens",
    ])
    .explode(
        "numeric_tokens"
    )
    .rename({
        "numeric_tokens": "numeric_token",
    })
    .filter(
        pl.col("numeric_token") != ""
    )
    .unique([
        "s1_entity_id",
        "country",
        "numeric_token",
    ])
)


print(
    f"S1 numeric-key rows: "
    f"{s1_numeric_fixed.height:,}"
)


# --------------------------------------------------------------------------------------------------
# EXACT-ADDRESS CANDIDATES
#
# IMPORTANT:
# Country is now part of the join key.
# --------------------------------------------------------------------------------------------------

print("\nGenerating exact-address candidates...")

t0 = time.time()


s2_exact_addr = (
    s2_addr
    .filter(
        pl.col("usable_exact_address")
    )
    .select([
        "entity_id",
        "country",
        "address_norm",
    ])
)


s3_exact_addr = (
    s3_addr
    .filter(
        pl.col("usable_exact_address")
    )
    .select([
        "entity_id",
        "country",
        "address_norm",
    ])
)


c_s2_address = (
    s1_addr_fixed
    .filter(
        pl.col("address_norm") != ""
    )
    .select([
        "s1_entity_id",
        "country",
        "address_norm",
    ])
    .join(
        s2_exact_addr,
        on=[
            "country",
            "address_norm",
        ],
        how="inner",
    )
    .select([
        "s1_entity_id",

        pl.col("entity_id")
        .alias("candidate_entity_id"),

        pl.lit("S2")
        .alias("source"),

        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_exact"),

        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_first_last"),

        pl.lit(1)
        .cast(pl.UInt8)
        .alias("block_address_exact"),

        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_address_numeric"),
    ])
)


c_s3_address = (
    s1_addr_fixed
    .filter(
        pl.col("address_norm") != ""
    )
    .select([
        "s1_entity_id",
        "country",
        "address_norm",
    ])
    .join(
        s3_exact_addr,
        on=[
            "country",
            "address_norm",
        ],
        how="inner",
    )
    .select([
        "s1_entity_id",

        pl.col("entity_id")
        .alias("candidate_entity_id"),

        pl.lit("S3")
        .alias("source"),

        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_exact"),

        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_first_last"),

        pl.lit(1)
        .cast(pl.UInt8)
        .alias("block_address_exact"),

        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_address_numeric"),
    ])
)


print(
    f"Exact-address candidate generation: "
    f"{time.time() - t0:.1f}s"
)

print(
    f"S2 exact-address candidates: "
    f"{c_s2_address.height:,}"
)

print(
    f"S3 exact-address candidates: "
    f"{c_s3_address.height:,}"
)


# --------------------------------------------------------------------------------------------------
# FREE SOME EXACT-ADDRESS TEMPORARIES
# --------------------------------------------------------------------------------------------------

del s2_exact_addr
del s3_exact_addr

gc.collect()


# --------------------------------------------------------------------------------------------------
# NUMERIC SOURCE KEYS
# --------------------------------------------------------------------------------------------------

print("\nBuilding source numeric-address keys...")


s2_numeric = (
    s2_addr
    .select([
        "entity_id",
        "country",
        "numeric_tokens",
    ])
    .explode(
        "numeric_tokens"
    )
    .rename({
        "numeric_tokens": "numeric_token",
    })
    .filter(
        pl.col("numeric_token") != ""
    )
    .unique([
        "entity_id",
        "country",
        "numeric_token",
    ])
)


s3_numeric = (
    s3_addr
    .select([
        "entity_id",
        "country",
        "numeric_tokens",
    ])
    .explode(
        "numeric_tokens"
    )
    .rename({
        "numeric_tokens": "numeric_token",
    })
    .filter(
        pl.col("numeric_token") != ""
    )
    .unique([
        "entity_id",
        "country",
        "numeric_token",
    ])
)


print(
    f"S2 numeric rows before cap: "
    f"{s2_numeric.height:,}"
)

print(
    f"S3 numeric rows before cap: "
    f"{s3_numeric.height:,}"
)


# --------------------------------------------------------------------------------------------------
# SOURCE-SPECIFIC NUMERIC FREQUENCIES
# --------------------------------------------------------------------------------------------------

print("\nComputing source-specific numeric frequencies...")


s2_numeric_freq = (
    s2_numeric
    .group_by([
        "country",
        "numeric_token",
    ])
    .len()
    .rename({
        "len": "numeric_freq",
    })
)


s3_numeric_freq = (
    s3_numeric
    .group_by([
        "country",
        "numeric_token",
    ])
    .len()
    .rename({
        "len": "numeric_freq",
    })
)


s2_numeric = (
    s2_numeric
    .join(
        s2_numeric_freq,
        on=[
            "country",
            "numeric_token",
        ],
        how="left",
    )
    .filter(
        (pl.col("numeric_freq") > 0)
        &
        (pl.col("numeric_freq") <= ADDRESS_NUMERIC_CAP)
    )
    .drop(
        "numeric_freq"
    )
)


s3_numeric = (
    s3_numeric
    .join(
        s3_numeric_freq,
        on=[
            "country",
            "numeric_token",
        ],
        how="left",
    )
    .filter(
        (pl.col("numeric_freq") > 0)
        &
        (pl.col("numeric_freq") <= ADDRESS_NUMERIC_CAP)
    )
    .drop(
        "numeric_freq"
    )
)


del s2_numeric_freq
del s3_numeric_freq

gc.collect()


print(
    f"S2 usable numeric rows: "
    f"{s2_numeric.height:,}"
)

print(
    f"S3 usable numeric rows: "
    f"{s3_numeric.height:,}"
)


# --------------------------------------------------------------------------------------------------
# NUMERIC ADDRESS CANDIDATES
#
# Again: country is part of the join key.
# --------------------------------------------------------------------------------------------------

print("\nGenerating numeric-address candidates...")

t0 = time.time()


c_s2_numeric = (
    s1_numeric_fixed
    .join(
        s2_numeric,
        on=[
            "country",
            "numeric_token",
        ],
        how="inner",
    )
    .select([
        "s1_entity_id",

        pl.col("entity_id")
        .alias("candidate_entity_id"),

        pl.lit("S2")
        .alias("source"),

        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_exact"),

        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_first_last"),

        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_address_exact"),

        pl.lit(1)
        .cast(pl.UInt8)
        .alias("block_address_numeric"),
    ])
)


c_s3_numeric = (
    s1_numeric_fixed
    .join(
        s3_numeric,
        on=[
            "country",
            "numeric_token",
        ],
        how="inner",
    )
    .select([
        "s1_entity_id",

        pl.col("entity_id")
        .alias("candidate_entity_id"),

        pl.lit("S3")
        .alias("source"),

        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_exact"),

        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_first_last"),

        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_address_exact"),

        pl.lit(1)
        .cast(pl.UInt8)
        .alias("block_address_numeric"),
    ])
)


print(
    f"Numeric candidate generation: "
    f"{time.time() - t0:.1f}s"
)

print(
    f"S2 numeric-address candidates: "
    f"{c_s2_numeric.height:,}"
)

print(
    f"S3 numeric-address candidates: "
    f"{c_s3_numeric.height:,}"
)


# --------------------------------------------------------------------------------------------------
# COMBINE ADDRESS CANDIDATES
# --------------------------------------------------------------------------------------------------

address_candidates = pl.concat(
    [
        c_s2_address,
        c_s3_address,
        c_s2_numeric,
        c_s3_numeric,
    ],
    how="vertical",
)


address_candidates = (
    address_candidates
    .group_by([
        "s1_entity_id",
        "candidate_entity_id",
        "source",
    ])
    .agg([
        pl.max("block_exact")
        .alias("block_exact"),

        pl.max("block_first_last")
        .alias("block_first_last"),

        pl.max("block_address_exact")
        .alias("block_address_exact"),

        pl.max("block_address_numeric")
        .alias("block_address_numeric"),
    ])
)


# Free numeric source tables now that candidates are generated.
del s2_numeric
del s3_numeric
del c_s2_address
del c_s3_address
del c_s2_numeric
del c_s3_numeric

gc.collect()


print(
    f"\nUnique address rescue pairs: "
    f"{address_candidates.height:,}"
)


# --------------------------------------------------------------------------------------------------
# UNION NAME + ADDRESS CANDIDATES
# --------------------------------------------------------------------------------------------------

print("\nUnioning name + address candidate sets...")


name_candidates_for_union = (
    candidate_pairs
    .select([
        "s1_entity_id",
        "candidate_entity_id",
        "source",
        "block_exact",
        "block_first_last",
        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_address_exact"),
        pl.lit(0)
        .cast(pl.UInt8)
        .alias("block_address_numeric"),
    ])
)


combined_candidates = pl.concat(
    [
        name_candidates_for_union,
        address_candidates,
    ],
    how="vertical",
)


combined_candidates = (
    combined_candidates
    .group_by([
        "s1_entity_id",
        "candidate_entity_id",
        "source",
    ])
    .agg([
        pl.max("block_exact")
        .alias("block_exact"),

        pl.max("block_first_last")
        .alias("block_first_last"),

        pl.max("block_address_exact")
        .alias("block_address_exact"),

        pl.max("block_address_numeric")
        .alias("block_address_numeric"),
    ])
    .sort([
        "s1_entity_id",
        "source",
        "candidate_entity_id",
    ])
)


print(
    f"Name candidate pairs     : "
    f"{candidate_pairs.height:,}"
)

print(
    f"Address rescue pairs     : "
    f"{address_candidates.height:,}"
)

print(
    f"Combined candidate pairs : "
    f"{combined_candidates.height:,}"
)


# --------------------------------------------------------------------------------------------------
# SAVE EARLY
# --------------------------------------------------------------------------------------------------

print("\nSaving candidate artifact immediately...")

combined_candidates.write_parquet(
    ADDRESS_CANDIDATE_OUT,
    compression="zstd",
)

print(
    f"Saved: {ADDRESS_CANDIDATE_OUT}"
)


# --------------------------------------------------------------------------------------------------
# CANDIDATE COUNTS
# --------------------------------------------------------------------------------------------------

candidate_counts_rescue = (
    s1_addr_fixed
    .select([
        "s1_entity_id",
    ])
    .join(
        combined_candidates
        .group_by("s1_entity_id")
        .len()
        .rename({
            "len": "candidate_count",
        }),
        on="s1_entity_id",
        how="left",
    )
    .with_columns(
        pl.col("candidate_count")
        .fill_null(0)
        .cast(pl.UInt32)
    )
)


# --------------------------------------------------------------------------------------------------
# VOLUME
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("ADDRESS-RESCUED CANDIDATE VOLUME")
print("=" * 100)


volume = (
    candidate_counts_rescue
    .select([
        pl.col("candidate_count")
        .mean()
        .alias("mean"),

        pl.col("candidate_count")
        .median()
        .alias("median"),

        pl.col("candidate_count")
        .quantile(0.90)
        .alias("p90"),

        pl.col("candidate_count")
        .quantile(0.95)
        .alias("p95"),

        pl.col("candidate_count")
        .quantile(0.99)
        .alias("p99"),

        pl.col("candidate_count")
        .max()
        .alias("max"),

        (
            pl.col("candidate_count") == 0
        )
        .mean()
        .alias("zero_rate"),
    ])
)

print(volume)


# --------------------------------------------------------------------------------------------------
# LOAD GROUND-TRUTH POSITIVE EDGES
# --------------------------------------------------------------------------------------------------

if "edge_eval" in globals():

    gt = eager(
        edge_eval
    )

else:

    gt = pl.read_parquet(
        os.path.join(
            STATE_DIR,
            "edge_eval.parquet",
        )
    )


gt = (
    gt
    .select([
        "source1_entity_id",
        "matched_entity_id",
        "matched_source",
    ])
    .rename({
        "source1_entity_id": "s1_entity_id",
        "matched_entity_id": "candidate_entity_id",
        "matched_source": "true_source",
    })
    .unique()
)


# --------------------------------------------------------------------------------------------------
# RECALL
# --------------------------------------------------------------------------------------------------

hits = (
    gt
    .join(
        combined_candidates.select([
            "s1_entity_id",
            "candidate_entity_id",
            "source",
        ]),
        left_on=[
            "s1_entity_id",
            "candidate_entity_id",
            "true_source",
        ],
        right_on=[
            "s1_entity_id",
            "candidate_entity_id",
            "source",
        ],
        how="inner",
    )
    .unique([
        "s1_entity_id",
        "candidate_entity_id",
        "true_source",
    ])
)


total_edges = gt.height
hit_edges = hits.height

edge_recall = (
    hit_edges / total_edges
    if total_edges
    else 0.0
)


print("\n" + "=" * 100)
print("ADDRESS RESCUE — ACTUAL CANDIDATE RECALL")
print("=" * 100)

print(
    f"Positive edges found : "
    f"{hit_edges:,} / {total_edges:,}"
)

print(
    f"Overall edge recall  : "
    f"{edge_recall:.4%}"
)


# --------------------------------------------------------------------------------------------------
# SOURCE RECALL
# --------------------------------------------------------------------------------------------------

source_recall = (
    gt
    .group_by("true_source")
    .len()
    .rename({
        "len": "positive_edges",
    })
    .join(
        hits
        .group_by("true_source")
        .len()
        .rename({
            "len": "hit_edges",
        }),
        on="true_source",
        how="left",
    )
    .with_columns([
        pl.col("hit_edges")
        .fill_null(0),

        (
            pl.col("hit_edges")
            .fill_null(0)
            /
            pl.col("positive_edges")
        )
        .alias("edge_recall"),
    ])
    .sort("true_source")
)

print("\nSource-wise recall:")
print(source_recall)


# --------------------------------------------------------------------------------------------------
# FULL-SET RECOVERY
# --------------------------------------------------------------------------------------------------

gt_counts = (
    gt
    .group_by("s1_entity_id")
    .len()
    .rename({
        "len": "true_match_count",
    })
)

hit_counts = (
    hits
    .group_by("s1_entity_id")
    .len()
    .rename({
        "len": "candidate_hit_count",
    })
)

full_set = (
    gt_counts
    .join(
        hit_counts,
        on="s1_entity_id",
        how="left",
    )
    .with_columns(
        pl.col("candidate_hit_count")
        .fill_null(0)
    )
    .with_columns(
        (
            pl.col("true_match_count")
            ==
            pl.col("candidate_hit_count")
        )
        .alias("full_set_recovered")
    )
)

positive_s1 = full_set.height

fully_recovered = (
    full_set
    .filter(
        pl.col("full_set_recovered")
    )
    .height
)

full_recovery = (
    fully_recovered / positive_s1
    if positive_s1
    else 0.0
)


print("\n" + "=" * 100)
print("FULL-SET RECOVERY")
print("=" * 100)

print(
    f"Positive S1 entities : "
    f"{positive_s1:,}"
)

print(
    f"Fully recovered      : "
    f"{fully_recovered:,}"
)

print(
    f"Full-set recovery    : "
    f"{full_recovery:.4%}"
)


# --------------------------------------------------------------------------------------------------
# RESIDUAL
# --------------------------------------------------------------------------------------------------

residual = (
    gt
    .join(
        combined_candidates.select([
            "s1_entity_id",
            "candidate_entity_id",
            "source",
        ]),
        left_on=[
            "s1_entity_id",
            "candidate_entity_id",
            "true_source",
        ],
        right_on=[
            "s1_entity_id",
            "candidate_entity_id",
            "source",
        ],
        how="anti",
    )
)


print("\n" + "=" * 100)
print("RESIDUAL AFTER ADDRESS RESCUE")
print("=" * 100)

print(
    f"Residual positives : "
    f"{residual.height:,}"
)

print(
    f"Residual rate      : "
    f"{(
        residual.height / total_edges
        if total_edges
        else 0.0
    ):.4%}"
)


# --------------------------------------------------------------------------------------------------
# SAVE REMAINING ARTIFACTS
# --------------------------------------------------------------------------------------------------

candidate_counts_rescue.write_parquet(
    ADDRESS_COUNTS_OUT,
    compression="zstd",
)

residual.write_parquet(
    ADDRESS_RESIDUAL_OUT,
    compression="zstd",
)


print(
    f"\nCounts saved   : {ADDRESS_COUNTS_OUT}"
)

print(
    f"Residual saved : {ADDRESS_RESIDUAL_OUT}"
)


# --------------------------------------------------------------------------------------------------
# FINAL
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("CELL 58 FIX COMPLETE")
print("=" * 100)

print(
    f"Candidate pairs         : "
    f"{combined_candidates.height:,}"
)

print(
    f"Mean candidates / S1    : "
    f"{candidate_counts_rescue['candidate_count'].mean():.3f}"
)

print(
    f"Median candidates / S1  : "
    f"{candidate_counts_rescue['candidate_count'].median():.3f}"
)

print(
    f"P90 candidates / S1     : "
    f"{candidate_counts_rescue['candidate_count'].quantile(0.90):.3f}"
)

print(
    f"P95 candidates / S1     : "
    f"{candidate_counts_rescue['candidate_count'].quantile(0.95):.3f}"
)

print(
    f"P99 candidates / S1     : "
    f"{candidate_counts_rescue['candidate_count'].quantile(0.99):.3f}"
)

print(
    f"Max candidates / S1     : "
    f"{candidate_counts_rescue['candidate_count'].max():,}"
)

print(
    f"Zero-candidate S1s      : "
    f"{candidate_counts_rescue.filter(pl.col('candidate_count') == 0).height:,}"
)

print(
    f"Positive edge recall    : "
    f"{edge_recall:.4%}"
)

print(
    f"Full-set recovery       : "
    f"{full_recovery:.4%}"
)

print(
    f"Residual positives      : "
    f"{residual.height:,}"
)

print("=" * 100)

AMLC 2026 — CELL 58 FIX / CONTINUATION
Existing name candidates: 9,494,493
S1 address rows: 100,000
S2 address rows: 5,034,616
S3 address rows: 5,285,603

Attaching country to S1 address keys...
S1 address rows with country: 100,000
S1 numeric-key rows: 151,644

Generating exact-address candidates...
Exact-address candidate generation: 2.1s
S2 exact-address candidates: 25,683
S3 exact-address candidates: 9,269

Building source numeric-address keys...
S2 numeric rows before cap: 7,297,951
S3 numeric rows before cap: 7,645,295

Computing source-specific numeric frequencies...
S2 usable numeric rows: 1,898,010
S3 usable numeric rows: 1,947,109

Generating numeric-address candidates...
Numeric candidate generation: 0.8s
S2 numeric-address candidates: 6,577,339
S3 numeric-address candidates: 6,266,839

Unique address rescue pairs: 12,861,961

Unioning name + address candidate sets...
